### Customer Table Generator

In [5]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import string

# Initialize Faker with multiple locales for diversity
fake = Faker(["en_US", "en_GB", "en_CA", "en_AU"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)


def generate_messy_customer_data(num_rows=1000):
    """
    Generate a messy, realistic customer dataset with realistic but challenging column names
    and various data quality issues for testing data mapping and cleaning.
    """

    data = []

    # Track some IDs for creating duplicates
    used_ids = []
    duplicate_names = []

    # Choose one consistent ID format for the entire dataset
    id_format_choice = random.choice(["CUST", "CUSTOMER", "NUMBER"])

    for i in range(num_rows):
        record = {}

        # Customer ID with CONSISTENT format
        if i % 50 == 0 and used_ids:  # 2% duplicates
            cust_id = random.choice(used_ids)
        else:
            if id_format_choice == "CUST":
                cust_id = f"CUST_{str(i + 1000).zfill(5)}"
            elif id_format_choice == "CUSTOMER":
                cust_id = f"CUSTOMER-{str(i + 1000).zfill(5)}"
            else:  # NUMBER
                cust_id = str(10000 + i)

            used_ids.append(cust_id)

        record["cust_id"] = cust_id if i % 100 != 0 else None  # 1% null IDs

        # Customer Name with variations
        if i % 30 == 0:  # Some nulls
            name = None
        elif i % 40 == 0:  # Some duplicates
            if duplicate_names:
                name = random.choice(duplicate_names)
            else:
                name = fake.name()
                duplicate_names.append(name)
        elif i % 25 == 0:  # Business names mixed in (business logic violation for B2C)
            name = fake.company()
        elif i % 35 == 0:  # Names with special characters
            name = fake.name() + " Jr."
        elif i % 45 == 0:  # Names with numbers (data quality issue)
            name = fake.name() + "123"
        elif i % 55 == 0:  # Unicode characters
            name = fake.name() + " 中文"
        else:
            name = fake.name()

        record["full_name"] = name

        # Customer Type with inconsistencies
        if i % 50 == 0:
            cust_type = None
        elif i % 60 == 0:
            # Wrong/inconsistent values
            cust_type = random.choice(
                ["Individual", "Business", "b2c", "b2b", "B-2-C", "B-2-B"]
            )
        elif i % 70 == 0:
            cust_type = random.choice(["1", "2", "3"])  # Coded values
        else:
            cust_type = random.choice(["B2C", "B2B", "Guest"])

        record["customer_category"] = cust_type

        # Gender with variations
        if i % 20 == 0:
            gender = None
        elif i % 25 == 0:
            # Various formats
            gender = random.choice(
                ["M", "F", "O", "N/A", "male", "female", "1", "2", "Unknown"]
            )
        else:
            gender = random.choice(["Male", "Female", "Other", "Prefer not to say"])

        record["sex"] = gender

        # Date of Birth with various formats and issues
        if i % 15 == 0:
            dob = None
        elif i % 20 == 0:
            # String format variations
            dob_date = fake.date_of_birth(minimum_age=18, maximum_age=80)
            formats = ["%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y", "%Y%m%d", "%b %d, %Y"]
            dob = dob_date.strftime(random.choice(formats))
        elif i % 30 == 0:
            # Invalid/extreme dates
            dob = random.choice(
                ["0000-00-00", "1900-01-01", "9999-99-99", "99-99-9999", "31-31-2020"]
            )
        elif i % 40 == 0:
            # Future dates (business logic violation)
            dob = fake.date_between(start_date="today", end_date="+10y")
        elif i % 80 == 0:
            # Too old (business logic violation - 150+ years old)
            dob = fake.date_between(start_date="-150y", end_date="-120y")
        else:
            dob = fake.date_of_birth(minimum_age=18, maximum_age=80)

        record["birth_date"] = dob

        # Registration Date with variations
        if i % 25 == 0:
            reg_date = None
        elif i % 35 == 0:
            # String timestamps
            reg_date = fake.date_time_between(
                start_date="-5y", end_date="now"
            ).strftime("%Y-%m-%d %H:%M:%S")
        elif i % 45 == 0:
            # Unix timestamp
            reg_date = int(
                fake.date_time_between(start_date="-5y", end_date="now").timestamp()
            )
        elif i % 55 == 0:
            # ISO format
            reg_date = fake.date_time_between(
                start_date="-5y", end_date="now"
            ).isoformat()
        elif i % 65 == 0:
            # Business logic violation: registration before birth
            if record["birth_date"] and not isinstance(record["birth_date"], str):
                reg_date = fake.date_between(start_date="-100y", end_date="-50y")
            else:
                reg_date = fake.date_time_between(start_date="-5y", end_date="now")
        else:
            reg_date = fake.date_time_between(start_date="-5y", end_date="now")

        record["account_created"] = reg_date

        # Customer Status with variations
        if i % 30 == 0:
            status = None
        elif i % 40 == 0:
            # Inconsistent values
            status = random.choice(
                ["active", "ACTIVE", "1", "A", "inactive", "INACTIVE", "0", "I"]
            )
        elif i % 50 == 0:
            # Typos
            status = random.choice(["Actve", "Inactiv", "Bloked"])
        else:
            status = random.choice(["Active", "Inactive", "Blocked"])

        record["status"] = status

        # Acquisition Channel with messy data
        if i % 20 == 0:
            channel = None
        elif i % 30 == 0:
            # Variations and typos
            channel = random.choice(
                [
                    "google",
                    "Google",
                    "GOOGLE",
                    "Google Ads",
                    "GoogleAds",
                    "google_ads",
                    "FB",
                    "facebook",
                    "Facebook Ads",
                    "Meta",
                    "organic",
                    "Organic",
                    "SEO",
                    "Direct",
                ]
            )
        elif i % 40 == 0:
            # Unexpected values
            channel = random.choice(["Unknown", "N/A", "?", "", " "])
        else:
            channel = random.choice(
                [
                    "Google Ads",
                    "Facebook",
                    "Organic Search",
                    "Email Marketing",
                    "Referral",
                    "Direct",
                    "Instagram",
                    "Twitter",
                    "LinkedIn",
                ]
            )

        record["acquisition_source"] = channel

        # Customer Segment with variations
        if i % 25 == 0:
            segment = None
        elif i % 35 == 0:
            # Coded values
            segment = random.choice(["HV", "OB", "NC", "1", "2", "3"])
        elif i % 45 == 0:
            # Different naming conventions
            segment = random.choice(
                ["VIP", "Regular", "Bronze", "Silver", "Gold", "Platinum"]
            )
        else:
            segment = random.choice(
                ["High Value", "Occasional Buyer", "New Customer", "Churned", "At Risk"]
            )

        record["customer_tier"] = segment

        # Record creation timestamp with variations
        if i % 30 == 0:
            created = None
        elif i % 40 == 0:
            created = datetime.now() - timedelta(days=random.randint(0, 365))
        else:
            created = fake.date_time_between(start_date="-1y", end_date="now")

        record["created_timestamp"] = created

        # Add additional realistic columns

        # Email
        if random.random() > 0.1:  # 90% have this field
            if i % 20 == 0:
                email = None
            elif i % 30 == 0:
                email = random.choice(["N/A", "NA", "not provided", "", "NULL"])
            elif i % 40 == 0:
                email = fake.email().upper()  # Case issues
            elif i % 100 == 0:
                # Business logic violation: invalid email format
                email = random.choice(
                    ["notanemail", "@gmail.com", "user@", "user@@gmail.com"]
                )
            else:
                email = fake.email()
            record["email_address"] = email

        # Phone
        if random.random() > 0.15:  # 85% have this field
            if i % 25 == 0:
                phone = None
            elif i % 35 == 0:
                phone = random.choice(["N/A", "0000000000", "999-999-9999", "UNKNOWN"])
            elif i % 45 == 0:
                # Different formats
                phone = (
                    fake.phone_number()
                    .replace("-", "")
                    .replace("(", "")
                    .replace(")", "")
                    .replace(" ", "")
                )
            elif i % 90 == 0:
                # Business logic violation: invalid phone
                phone = random.choice(["123", "111111111111111111", "PHONE"])
            else:
                phone = fake.phone_number()
            record["phone_number"] = phone

        # Country
        if random.random() > 0.25:  # 75% have country
            if i % 30 == 0:
                country = None
            elif i % 40 == 0:
                # Mix of formats
                country = random.choice(
                    [fake.country_code(), fake.country(), "USA", "US", "United States"]
                )
            else:
                country = fake.country_code()
            record["country"] = country

        # Age (redundant with DOB - business logic issue)
        if random.random() > 0.3:  # 70% have this
            if i % 25 == 0:
                age = None
            elif i % 35 == 0:
                # Extreme values
                age = random.choice([-5, 0, 999, -999, 200])
            elif i % 45 == 0:
                # String instead of number
                age = random.choice(["N/A", "Unknown", "NULL"])
            elif i % 60 == 0:
                # Business logic violation: age doesn't match DOB
                age = random.randint(18, 80)  # Will often not match the DOB
            else:
                # Calculate correct age (sometimes)
                if record["birth_date"] and not isinstance(record["birth_date"], str):
                    try:
                        age = (datetime.now() - record["birth_date"]).days // 365
                    except:
                        age = random.randint(18, 80)
                else:
                    age = random.randint(18, 80)
            record["age"] = age

        # Lifetime value
        if random.random() > 0.3:  # 70% have this
            if i % 20 == 0:
                ltv = None
            elif i % 30 == 0:
                ltv = random.choice(["N/A", "0", "-1", "NULL"])
            elif i % 40 == 0:
                # Extreme values
                ltv = random.choice([-99999999, 99999999, 0.0000001, -1000])
            elif i % 50 == 0:
                ltv = str(round(random.uniform(0, 50000), 2))  # String instead of float
            else:
                ltv = round(random.uniform(0, 50000), 2)
            record["lifetime_value"] = ltv

        # Last purchase date
        if random.random() > 0.2:  # 80% have this
            if i % 25 == 0:
                last_purchase = None
            elif i % 35 == 0:
                last_purchase = "1900-01-01"  # Default/invalid date
            elif i % 70 == 0:
                # Business logic violation: last purchase before registration
                if record["account_created"] and not isinstance(
                    record["account_created"], str
                ):
                    last_purchase = fake.date_between(start_date="-10y", end_date="-6y")
                else:
                    last_purchase = fake.date_between(
                        start_date="-1y", end_date="today"
                    )
            elif i % 80 == 0:
                # Future purchase (business logic violation)
                last_purchase = fake.date_between(start_date="+1y", end_date="+2y")
            else:
                last_purchase = fake.date_between(start_date="-1y", end_date="today")
            record["last_purchase_date"] = last_purchase

        # Purchase count
        if random.random() > 0.4:  # 60% have this
            if i % 30 == 0:
                count = None
            elif i % 40 == 0:
                # Extreme values
                count = random.choice([-10, -1, 999999, 0.5])
            elif i % 50 == 0:
                # Business logic violation: high value customer with 0 purchases
                if record.get("customer_tier") in [
                    "High Value",
                    "VIP",
                    "Gold",
                    "Platinum",
                ]:
                    count = 0
                else:
                    count = random.randint(0, 100)
            else:
                count = random.randint(0, 100)
            record["total_purchases"] = count

        # Loyalty points
        if random.random() > 0.5:  # 50% have this
            if i % 35 == 0:
                points = None
            elif i % 45 == 0:
                # Extreme/invalid values
                points = random.choice([-9999, -1, 99999999, "UNLIMITED"])
            elif i % 55 == 0:
                # Business logic violation: inactive customer with high points
                if record.get("status") in [
                    "Inactive",
                    "inactive",
                    "INACTIVE",
                    "0",
                    "I",
                ]:
                    points = random.randint(10000, 50000)
                else:
                    points = random.randint(0, 5000)
            else:
                points = random.randint(0, 5000)
            record["loyalty_points"] = points

        data.append(record)

    # Create DataFrame
    df = pd.DataFrame(data)

    # Add EXACT duplicate rows (every attribute is same)
    num_exact_duplicates = int(num_rows * 0.02)  # 2% exact duplicates
    for _ in range(num_exact_duplicates):
        if len(df) > 0:
            # Pick a random row to duplicate
            row_to_duplicate = df.sample(1)
            df = pd.concat([df, row_to_duplicate], ignore_index=True)

    # Add some completely empty rows
    for _ in range(int(num_rows * 0.005)):  # 0.5% empty rows
        empty_row = pd.Series([None] * len(df.columns), index=df.columns)
        df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)

    # Add some rows with all string 'NULL' or 'N/A' values (common in real data)
    for _ in range(int(num_rows * 0.01)):  # 1% NULL string rows
        null_values = ["NULL", "N/A", "null", "NA", "", " "]
        null_row = pd.Series(
            [random.choice(null_values) for _ in range(len(df.columns))],
            index=df.columns,
        )
        df = pd.concat([df, pd.DataFrame([null_row])], ignore_index=True)

    # Shuffle the dataframe to mix duplicates throughout
    df = df.sample(frac=1).reset_index(drop=True)

    return df


def add_more_messiness(df):
    """
    Add additional data quality issues to make the dataset more challenging
    """
    # Add trailing/leading spaces to some string columns
    string_cols = df.select_dtypes(include=["object"]).columns
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05  # 5% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    # Add case inconsistencies
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03  # 3% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    # Add special characters to some values
    for col in string_cols[:3]:  # Only first 3 string columns
        mask = np.random.random(len(df)) < 0.02  # 2% of values
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    # Add encoding issues (common in real data)
    for col in string_cols[:2]:  # First 2 string columns
        mask = np.random.random(len(df)) < 0.01  # 1% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).replace("a", "Ã¡").replace("e", "Ã©")
                if pd.notna(x) and random.random() > 0.5
                else x
            )
        )

    return df


# Generate the dataset
if __name__ == "__main__":
    # Set the number of rows you want
    NUM_ROWS = 1000  # Change this to your desired number

    print(f"Generating {NUM_ROWS} rows of messy customer data...")
    df = generate_messy_customer_data(NUM_ROWS)

    # Add more messiness
    df = add_more_messiness(df)

    # Display basic info
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumn names (realistic but challenging for mapping):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")

    print(f"\nFirst 10 rows:")
    print(df.head(10))

    # Show data quality issues summary
    print("\n" + "=" * 50)
    print("DATA QUALITY ISSUES SUMMARY:")
    print("=" * 50)
    print(f"Total null values: {df.isnull().sum().sum()}")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    print(f"Total rows: {len(df)}")

    # Show business logic violations examples
    print("\n" + "=" * 50)
    print("BUSINESS LOGIC VIOLATIONS EXAMPLES:")
    print("=" * 50)

    # Check for future dates of birth
    future_dobs = df[pd.to_datetime(df["birth_date"], errors="coerce") > datetime.now()]
    if len(future_dobs) > 0:
        print(f"✗ Future birth dates: {len(future_dobs)} rows")

    # Check for age mismatches
    age_col_exists = "age" in df.columns
    if age_col_exists:
        negative_ages = df[pd.to_numeric(df["age"], errors="coerce") < 0]
        extreme_ages = df[pd.to_numeric(df["age"], errors="coerce") > 150]
        if len(negative_ages) > 0:
            print(f"✗ Negative ages: {len(negative_ages)} rows")
        if len(extreme_ages) > 0:
            print(f"✗ Extreme ages (>150): {len(extreme_ages)} rows")

    # Check for negative lifetime values
    if "lifetime_value" in df.columns:
        negative_ltv = df[pd.to_numeric(df["lifetime_value"], errors="coerce") < 0]
        if len(negative_ltv) > 0:
            print(f"✗ Negative lifetime values: {len(negative_ltv)} rows")

    # Check for high value customers with 0 purchases
    if "customer_tier" in df.columns and "total_purchases" in df.columns:
        high_value_zero_purchases = df[
            (df["customer_tier"].isin(["High Value", "VIP", "Gold", "Platinum"]))
            & (pd.to_numeric(df["total_purchases"], errors="coerce") == 0)
        ]
        if len(high_value_zero_purchases) > 0:
            print(
                f"✗ High value customers with 0 purchases: {len(high_value_zero_purchases)} rows"
            )

    # Show extreme values
    print("\n" + "=" * 50)
    print("EXTREME VALUES EXAMPLES:")
    print("=" * 50)

    # Check for extreme dates
    extreme_dates = df[
        df["birth_date"].isin(["9999-99-99", "99-99-9999", "0000-00-00", "31-31-2020"])
    ]
    if len(extreme_dates) > 0:
        print(f"✗ Invalid/extreme date formats: {len(extreme_dates)} rows")
        print(f"  Examples: {extreme_dates['birth_date'].unique()[:5].tolist()}")

    # Check for extreme numeric values
    if "lifetime_value" in df.columns:
        ltv_numeric = pd.to_numeric(df["lifetime_value"], errors="coerce")
        extreme_ltv = df[(ltv_numeric < -10000) | (ltv_numeric > 90000000)]
        if len(extreme_ltv) > 0:
            print(f"✗ Extreme lifetime values: {len(extreme_ltv)} rows")
            print(f"  Examples: {extreme_ltv['lifetime_value'].unique()[:5].tolist()}")

    # Show main columns analysis
    print("\n" + "=" * 50)
    print("MAIN COLUMNS ANALYSIS:")
    print("=" * 50)

    main_columns = [
        "cust_id",
        "full_name",
        "customer_category",
        "sex",
        "birth_date",
        "account_created",
        "status",
        "acquisition_source",
        "customer_tier",
        "created_timestamp",
    ]

    for col in main_columns:
        if col in df.columns:
            null_count = df[col].isnull().sum()
            unique_count = df[col].nunique()
            print(f"\n{col}:")
            print(f"  - Null values: {null_count} ({null_count/len(df)*100:.1f}%)")
            print(f"  - Unique values: {unique_count}")
            if df[col].dtype == "object" and unique_count > 0:
                sample_values = df[col].dropna().unique()[:3].tolist()
                print(f"  - Sample values: {sample_values}")

    # Save to CSV
    output_file = "messy_customer_data.xlsx"
    df.to_excel(output_file, index=False)
    print(f"\n✅ Dataset saved to '{output_file}'")

    # Optional: Save to Excel with multiple formats
    # df.to_excel('messy_customer_data.xlsx', index=False)

    # Optional: Save to JSON for different format challenges
    # df.to_json('messy_customer_data.json', orient='records', date_format='iso')

Generating 1000 rows of messy customer data...


/tmp/ipykernel_8161/1434308968.py:415: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)
/tmp/ipykernel_8161/1434308968.py:415: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)
/tmp/ipykernel_8161/1434308968.py:415: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or 


Dataset shape: (1035, 18)

Column names (realistic but challenging for mapping):
  1. cust_id
  2. full_name
  3. customer_category
  4. sex
  5. birth_date
  6. account_created
  7. status
  8. acquisition_source
  9. customer_tier
  10. created_timestamp
  11. email_address
  12. phone_number
  13. last_purchase_date
  14. total_purchases
  15. loyalty_points
  16. age
  17. lifetime_value
  18. country

First 10 rows:
  cust_id           full_name customer_category                sex  \
0   10369     Cynthia Gregory               B2C               Male   
1   10078     Ms Karen Turner            Guest*             Female   
2   10299       Maureen Lewis               B2C              Other   
3   10890         James Lopez               B2C               Male   
4   10821  Charlotte Anderson               B2C              Other   
5   10931         John Harris             guest             Female   
6   10862     Timothy Gardner               B2C               Male   
7   10542     

### Address Table Generator

In [10]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import string

# Initialize Faker with multiple locales for diversity
fake = Faker(["en_US", "en_GB", "en_CA", "en_AU", "fr_FR", "de_DE", "es_ES"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)


def generate_messy_address_data(num_rows=1000, customer_id_format="CUST"):
    """
    Generate a messy, realistic address dataset with realistic but challenging column names
    and various data quality issues for testing data mapping and cleaning.

    Args:
        num_rows: Number of address records to generate
        customer_id_format: Format of customer IDs ('CUST', 'CUSTOMER', or 'NUMBER')
    """

    data = []

    # Track some IDs for creating duplicates
    used_address_ids = []
    used_customer_ids = []

    # Generate a pool of customer IDs (assuming ~3 addresses per customer on average)
    num_customers = max(num_rows // 3, 100)
    for i in range(num_customers):
        if customer_id_format == "CUST":
            cust_id = f"CUST_{str(i + 1000).zfill(5)}"
        elif customer_id_format == "CUSTOMER":
            cust_id = f"CUSTOMER-{str(i + 1000).zfill(5)}"
        else:  # NUMBER
            cust_id = str(10000 + i)
        used_customer_ids.append(cust_id)

    # Choose one consistent address ID format for the entire dataset
    addr_id_format_choice = random.choice(["ADDR", "ADDRESS", "NUMBER"])

    # Track default addresses per customer for business logic violations
    customer_default_count = {}

    for i in range(num_rows):
        record = {}

        # Address ID with CONSISTENT format
        if i % 50 == 0 and used_address_ids:  # 2% duplicates
            addr_id = random.choice(used_address_ids)
        else:
            if addr_id_format_choice == "ADDR":
                addr_id = f"ADDR_{str(i + 5000).zfill(6)}"
            elif addr_id_format_choice == "ADDRESS":
                addr_id = f"ADDRESS-{str(i + 5000).zfill(6)}"
            else:  # NUMBER
                addr_id = str(50000 + i)

            used_address_ids.append(addr_id)

        record["addr_id"] = addr_id if i % 100 != 0 else None  # 1% null IDs

        # Customer ID - references customer table
        if i % 80 == 0:  # Some nulls (violates NOT NULL constraint)
            cust_id = None
        elif i % 60 == 0:  # Non-existent customer IDs (FK violation)
            if customer_id_format == "CUST":
                cust_id = f"CUST_{str(99999).zfill(5)}"
            elif customer_id_format == "CUSTOMER":
                cust_id = f"CUSTOMER-{str(99999).zfill(5)}"
            else:
                cust_id = "99999"
        elif i % 40 == 0:  # Invalid format
            cust_id = random.choice(["INVALID", "NULL", "N/A", ""])
        else:
            cust_id = random.choice(used_customer_ids)

        record["customer_ref"] = cust_id

        # Address Type with inconsistencies
        if i % 30 == 0:
            addr_type = None
        elif i % 25 == 0:
            # Wrong/inconsistent values
            addr_type = random.choice(
                ["billing", "BILLING", "Bil", "B", "shipping", "SHIPPING", "Ship", "S"]
            )
        elif i % 35 == 0:
            # Invalid values
            addr_type = random.choice(
                ["Home", "Work", "Office", "1", "2", "Primary", "Secondary"]
            )
        elif i % 45 == 0:
            # Typos
            addr_type = random.choice(["Biling", "Shiping", "Booth"])
        else:
            addr_type = random.choice(["Billing", "Shipping", "Both"])

        record["address_category"] = addr_type

        # Generate realistic address components with issues

        # Street Address (line 1)
        if i % 20 == 0:
            street = None
        elif i % 30 == 0:
            # Extreme/invalid values
            street = random.choice(["N/A", "NULL", "Unknown", "123", "???", ""])
        elif i % 40 == 0:
            # Special characters and encoding issues
            street = fake.street_address() + random.choice(["#", "@", "!", "™", "©"])
        elif i % 50 == 0:
            # Very long address
            street = (
                fake.street_address()
                + " "
                + fake.secondary_address()
                + " Building "
                + fake.building_number()
            )
        else:
            street = fake.street_address()

        record["street_line1"] = street

        # Street Address (line 2) - optional
        if random.random() > 0.6:  # 40% have line 2
            if i % 35 == 0:
                street2 = random.choice(["N/A", "NA", ""])
            else:
                street2 = fake.secondary_address()
            record["street_line2"] = street2

        # City - NOT NULL in schema
        if i % 60 == 0:  # Violates NOT NULL (will be challenging)
            city = None
        elif i % 25 == 0:
            # Invalid values
            city = random.choice(["N/A", "NULL", "Unknown", "123", ""])
        elif i % 35 == 0:
            # All caps or lowercase
            city = fake.city().upper() if random.random() > 0.5 else fake.city().lower()
        elif i % 45 == 0:
            # With special characters
            city = fake.city() + random.choice(["!", "?", "#"])
        elif i % 55 == 0:
            # Numbers in city name (invalid)
            city = fake.city() + "123"
        elif i % 70 == 0:
            # Business logic violation: city doesn't match state/country
            city = random.choice(
                ["Paris", "London", "Tokyo", "Sydney"]
            )  # Will mismatch with US states
        else:
            city = fake.city()

        record["city_name"] = city

        # State/Province
        if i % 20 == 0:
            state = None
        elif i % 30 == 0:
            # Mix of formats (full name vs abbreviation)
            state = fake.state() if random.random() > 0.5 else fake.state_abbr()
        elif i % 40 == 0:
            # Invalid values
            state = random.choice(["XX", "N/A", "Unknown", "99", ""])
        elif i % 50 == 0:
            # Wrong country states (business logic violation)
            state = random.choice(
                ["Ontario", "Quebec", "Bavaria", "Tokyo", "New South Wales"]
            )
        elif i % 60 == 0:
            # Typos
            state = random.choice(["Californai", "Texus", "Florda", "New Yrok"])
        else:
            state = fake.state()

        record["state_region"] = state

        # Postal Code
        if i % 25 == 0:
            postal = None
        elif i % 30 == 0:
            # Invalid formats
            postal = random.choice(["00000", "99999", "XXXXX", "N/A", "NULL", ""])
        elif i % 40 == 0:
            # Wrong format for country (e.g., UK postcode for US address)
            postal = random.choice(["SW1A 1AA", "M5H 2N2", "75001", "100-0001"])
        elif i % 50 == 0:
            # Extreme values
            postal = random.choice(["00000-0000", "99999-9999", "123", "1234567890"])
        elif i % 60 == 0:
            # Special characters
            postal = fake.postcode() + random.choice(["!", "#", "@"])
        else:
            postal = fake.postcode()

        record["zip_postal"] = postal

        # Country - NOT NULL in schema
        if i % 70 == 0:  # Violates NOT NULL
            country = None
        elif i % 30 == 0:
            # Mix of formats
            country = random.choice(
                [
                    "United States",
                    "USA",
                    "US",
                    "U.S.A.",
                    "United States of America",
                    "UK",
                    "United Kingdom",
                    "GB",
                    "Great Britain",
                    "Canada",
                    "CA",
                    "CAN",
                ]
            )
        elif i % 40 == 0:
            # Invalid values
            country = random.choice(["N/A", "NULL", "Unknown", "1", ""])
        elif i % 50 == 0:
            # Country code instead of name
            country = fake.country_code()
        elif i % 60 == 0:
            # Typos
            country = random.choice(["Untied States", "Canadia", "Australa", "Germny"])
        elif i % 80 == 0:
            # Business logic violation: country doesn't match state/city
            if record.get("state_region") in [
                "California",
                "Texas",
                "Florida",
                "New York",
            ]:
                country = random.choice(["Canada", "Mexico", "United Kingdom"])
            else:
                country = "United States"
        else:
            country = fake.country()

        record["country_name"] = country

        # Is Default flag
        if i % 25 == 0:
            is_default = None
        elif i % 30 == 0:
            # Various boolean representations
            is_default = random.choice(["Y", "N", "Yes", "No", "1", "0", "T", "F"])
        elif i % 40 == 0:
            # Invalid values
            is_default = random.choice(["Maybe", "Unknown", "NULL", ""])
        else:
            # Business logic violation: multiple defaults per customer
            if cust_id and cust_id not in ["INVALID", "NULL", "N/A", ""]:
                if cust_id not in customer_default_count:
                    customer_default_count[cust_id] = 0

                if i % 15 == 0 and customer_default_count[cust_id] > 0:
                    # Create violation: another default for same customer
                    is_default = True
                    customer_default_count[cust_id] += 1
                elif random.random() > 0.8:
                    is_default = True
                    customer_default_count[cust_id] += 1
                else:
                    is_default = False
            else:
                is_default = random.choice([True, False])

        record["default_flag"] = is_default

        # Created timestamp
        if i % 30 == 0:
            created = None
        elif i % 40 == 0:
            # String format
            created = fake.date_time_between(start_date="-2y", end_date="now").strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        elif i % 50 == 0:
            # Unix timestamp
            created = int(
                fake.date_time_between(start_date="-2y", end_date="now").timestamp()
            )
        elif i % 60 == 0:
            # Future date (business logic violation)
            created = fake.date_time_between(start_date="+1y", end_date="+2y")
        elif i % 70 == 0:
            # Very old date (business logic violation)
            created = fake.date_time_between(start_date="-50y", end_date="-30y")
        else:
            created = fake.date_time_between(start_date="-2y", end_date="now")

        record["record_created"] = created

        # Additional realistic columns that might exist

        # Latitude/Longitude (with issues)
        if random.random() > 0.3:  # 70% have coordinates
            if i % 30 == 0:
                lat = None
                lon = None
            elif i % 40 == 0:
                # Invalid coordinates
                lat = random.choice(["N/A", "NULL", "999", "-999"])
                lon = random.choice(["N/A", "NULL", "999", "-999"])
            elif i % 50 == 0:
                # Extreme values (impossible coordinates)
                lat = random.choice([91, -91, 180, -180, 999])
                lon = random.choice([181, -181, 360, -360, 999])
            elif i % 60 == 0:
                # Coordinates don't match address (business logic violation)
                lat = fake.latitude()
                lon = fake.longitude()
            else:
                lat = float(fake.latitude())
                lon = float(fake.longitude())

            record["latitude"] = lat
            record["longitude"] = lon

        # Verified flag
        if random.random() > 0.4:  # 60% have this
            if i % 35 == 0:
                verified = None
            elif i % 45 == 0:
                verified = random.choice(["Y", "N", "Pending", "Unknown"])
            else:
                verified = random.choice([True, False])
            record["address_verified"] = verified

        # Last updated
        if random.random() > 0.5:  # 50% have this
            if i % 40 == 0:
                last_updated = None
            elif i % 50 == 0:
                # Business logic violation: updated before created
                if record.get("record_created"):
                    if isinstance(record["record_created"], int):
                        # Handle Unix timestamp
                        created_datetime = datetime.fromtimestamp(
                            record["record_created"]
                        )
                        last_updated = created_datetime - timedelta(
                            days=random.randint(1, 365)
                        )
                    elif isinstance(record["record_created"], datetime):
                        # Handle datetime object
                        last_updated = record["record_created"] - timedelta(
                            days=random.randint(1, 365)
                        )
                    else:
                        # Handle string or other formats
                        last_updated = fake.date_time_between(
                            start_date="-1y", end_date="now"
                        )
                else:
                    last_updated = fake.date_time_between(
                        start_date="-1y", end_date="now"
                    )
            else:
                last_updated = fake.date_time_between(start_date="-1y", end_date="now")
            record["last_modified"] = last_updated

        data.append(record)

    # Create DataFrame
    df = pd.DataFrame(data)

    # Add EXACT duplicate rows (every attribute is same)
    num_exact_duplicates = int(num_rows * 0.02)  # 2% exact duplicates
    for _ in range(num_exact_duplicates):
        if len(df) > 0:
            # Pick a random row to duplicate
            row_to_duplicate = df.sample(1)
            df = pd.concat([df, row_to_duplicate], ignore_index=True)

    # Add some completely empty rows
    for _ in range(int(num_rows * 0.005)):  # 0.5% empty rows
        empty_row = pd.Series([None] * len(df.columns), index=df.columns)
        df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)

    # Add some rows with all string 'NULL' or 'N/A' values
    for _ in range(int(num_rows * 0.01)):  # 1% NULL string rows
        null_values = ["NULL", "N/A", "null", "NA", "", " "]
        null_row = pd.Series(
            [random.choice(null_values) for _ in range(len(df.columns))],
            index=df.columns,
        )
        df = pd.concat([df, pd.DataFrame([null_row])], ignore_index=True)

    # Shuffle the dataframe to mix duplicates throughout
    df = df.sample(frac=1).reset_index(drop=True)

    return df


def add_more_messiness(df):
    """
    Add additional data quality issues to make the dataset more challenging
    """
    # Add trailing/leading spaces to some string columns
    string_cols = df.select_dtypes(include=["object"]).columns
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05  # 5% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    # Add case inconsistencies
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03  # 3% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    # Add special characters to some values
    for col in string_cols[:3]:  # Only first 3 string columns
        mask = np.random.random(len(df)) < 0.02  # 2% of values
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


# Generate the dataset
if __name__ == "__main__":
    # Set the number of rows you want
    NUM_ROWS = 1500  # Change this to your desired number

    # Customer ID format should match your customer dataset
    # Options: 'CUST', 'CUSTOMER', or 'NUMBER'
    CUSTOMER_ID_FORMAT = "CUST"  # Change to match your customer dataset

    print(f"Generating {NUM_ROWS} rows of messy address data...")
    df = generate_messy_address_data(NUM_ROWS, CUSTOMER_ID_FORMAT)

    # Add more messiness
    df = add_more_messiness(df)

    # Display basic info
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumn names (realistic but challenging for mapping):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")

    print(f"\nFirst 10 rows:")
    print(df.head(10))

    # Show data quality issues summary
    print("\n" + "=" * 50)
    print("DATA QUALITY ISSUES SUMMARY:")
    print("=" * 50)
    print(f"Total null values: {df.isnull().sum().sum()}")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    print(f"Total rows: {len(df)}")

    # Check NOT NULL constraint violations
    print("\n" + "=" * 50)
    print("NOT NULL CONSTRAINT VIOLATIONS:")
    print("=" * 50)

    # customer_ref should not be null
    null_customers = df[df["customer_ref"].isnull()].shape[0]
    if null_customers > 0:
        print(f"✗ NULL customer_ref (violates NOT NULL): {null_customers} rows")

    # city_name should not be null
    null_cities = df[df["city_name"].isnull()].shape[0]
    if null_cities > 0:
        print(f"✗ NULL city_name (violates NOT NULL): {null_cities} rows")

    # country_name should not be null
    null_countries = df[df["country_name"].isnull()].shape[0]
    if null_countries > 0:
        print(f"✗ NULL country_name (violates NOT NULL): {null_countries} rows")

    # Show business logic violations
    print("\n" + "=" * 50)
    print("BUSINESS LOGIC VIOLATIONS EXAMPLES:")
    print("=" * 50)

    # Check for multiple default addresses per customer
    default_addresses = df[df["default_flag"].isin([True, "Y", "Yes", "1", "T"])]
    customer_defaults = default_addresses.groupby("customer_ref").size()
    multiple_defaults = customer_defaults[customer_defaults > 1]
    if len(multiple_defaults) > 0:
        print(
            f"✗ Customers with multiple default addresses: {len(multiple_defaults)} customers"
        )
        print(f"  Examples: {list(multiple_defaults.head(3).index)}")

    # Check for invalid foreign keys
    invalid_customers = df[df["customer_ref"].isin(["INVALID", "NULL", "N/A", ""])]
    if len(invalid_customers) > 0:
        print(f"✗ Invalid customer references: {len(invalid_customers)} rows")

    # Check for geographic mismatches
    us_states = ["California", "Texas", "Florida", "New York", "Illinois"]
    non_us_with_us_states = df[
        (df["state_region"].isin(us_states))
        & (~df["country_name"].isin(["United States", "USA", "US", "U.S.A."]))
    ]
    if len(non_us_with_us_states) > 0:
        print(f"✗ US states with non-US countries: {len(non_us_with_us_states)} rows")

    # Check for future created dates
    if "record_created" in df.columns:
        future_dates = df[
            pd.to_datetime(df["record_created"], errors="coerce") > datetime.now()
        ]
        if len(future_dates) > 0:
            print(f"✗ Future creation dates: {len(future_dates)} rows")

    # Show extreme values
    print("\n" + "=" * 50)
    print("EXTREME VALUES EXAMPLES:")
    print("=" * 50)

    # Check for invalid coordinates
    if "latitude" in df.columns and "longitude" in df.columns:
        lat_numeric = pd.to_numeric(df["latitude"], errors="coerce")
        lon_numeric = pd.to_numeric(df["longitude"], errors="coerce")
        invalid_coords = df[
            (lat_numeric > 90)
            | (lat_numeric < -90)
            | (lon_numeric > 180)
            | (lon_numeric < -180)
        ]
        if len(invalid_coords) > 0:
            print(f"✗ Invalid coordinates: {len(invalid_coords)} rows")
            if len(invalid_coords) > 0:
                print(
                    f"  Example lat/lon: {list(zip(invalid_coords['latitude'].head(3), invalid_coords['longitude'].head(3)))}"
                )

    # Check for invalid postal codes
    invalid_postcodes = df[
        df["zip_postal"].isin(["00000", "99999", "XXXXX", "123", "1234567890"])
    ]
    if len(invalid_postcodes) > 0:
        print(f"✗ Invalid postal codes: {len(invalid_postcodes)} rows")
        print(f"  Examples: {invalid_postcodes['zip_postal'].unique()[:5].tolist()}")

    # Show main columns analysis
    print("\n" + "=" * 50)
    print("MAIN COLUMNS ANALYSIS:")
    print("=" * 50)

    main_columns = [
        "addr_id",
        "customer_ref",
        "address_category",
        "street_line1",
        "city_name",
        "state_region",
        "zip_postal",
        "country_name",
        "default_flag",
        "record_created",
    ]

    for col in main_columns:
        if col in df.columns:
            null_count = df[col].isnull().sum()
            unique_count = df[col].nunique()
            print(f"\n{col}:")
            print(f"  - Null values: {null_count} ({null_count/len(df)*100:.1f}%)")
            print(f"  - Unique values: {unique_count}")
            if df[col].dtype == "object" and unique_count > 0:
                sample_values = df[col].dropna().unique()[:3].tolist()
                print(f"  - Sample values: {sample_values}")

    # Save to CSV
    output_file = "messy_address_data.xlsx"
    df.to_excel(output_file, index=False)
    print(f"\n✅ Dataset saved to '{output_file}'")

    # Optional: Save customer-address mapping for testing joins
    if "customer_ref" in df.columns:
        customer_address_summary = (
            df.groupby("customer_ref")
            .agg(
                {
                    "addr_id": "count",
                    "default_flag": lambda x: sum(x.isin([True, "Y", "Yes", "1", "T"])),
                }
            )
            .rename(
                columns={"addr_id": "address_count", "default_flag": "default_count"}
            )
        )
        customer_address_summary.to_csv("customer_address_summary.csv")
        print(f"✅ Customer-address summary saved to 'customer_address_summary.csv'")

Generating 1500 rows of messy address data...


/tmp/ipykernel_8161/885469759.py:388: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)
/tmp/ipykernel_8161/885469759.py:388: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)
/tmp/ipykernel_8161/885469759.py:388: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all


Dataset shape: (1552, 15)

Column names (realistic but challenging for mapping):
  1. addr_id
  2. customer_ref
  3. address_category
  4. street_line1
  5. city_name
  6. state_region
  7. zip_postal
  8. country_name
  9. default_flag
  10. record_created
  11. latitude
  12. longitude
  13. street_line2
  14. address_verified
  15. last_modified

First 10 rows:
  addr_id customer_ref address_category  \
0   50555   CUST_01343             Both   
1   50938   CUST_01357         Shipping   
2   50060   CUST_99999             None   
3   50693   CUST_01494         Shipping   
4   50407   CUST_01435         Shipping   
5   50592   CUST_01325          Billing   
6   50642   CUST_01328          BILLING   
7   51232   CUST_01003             Both   
8   51055   CUST_01420             Both   
9   50636   CUST_01408         Shipping   

                                     street_line1        city_name  \
0                                     5 iain loop        West Sean   
1                 

### Products Table Generator

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import string

# Initialize Faker
fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)


def generate_messy_product_data(num_rows=1000):
    """
    Generate a messy, realistic product dataset with realistic but challenging column names
    and various data quality issues for testing data mapping and cleaning.

    Args:
        num_rows: Number of product records to generate
    """

    data = []

    # Track some IDs for creating duplicates
    used_product_ids = []
    used_skus = []
    duplicate_names = []

    # Choose one consistent ID format for the entire dataset
    prod_id_format_choice = random.choice(["PROD", "PRODUCT", "NUMBER"])
    cat_id_format_choice = random.choice(["CAT", "CATEGORY", "NUMBER"])
    supp_id_format_choice = random.choice(["SUPP", "SUPPLIER", "NUMBER"])

    # Product name components for realistic generation
    brands = [
        "Nike",
        "Adidas",
        "Apple",
        "Samsung",
        "Sony",
        "Dell",
        "HP",
        "Lenovo",
        "Microsoft",
        "Google",
        "Amazon",
        "LG",
        "Panasonic",
        "Canon",
        "Nikon",
        "Bose",
        "JBL",
        "Reebok",
        "Puma",
        "Under Armour",
        "New Balance",
        "ASICS",
        "Brooks",
        "Saucony",
        "Mizuno",
    ]

    product_types = [
        "Laptop",
        "Smartphone",
        "Tablet",
        "Headphones",
        "Smartwatch",
        "Camera",
        "Running Shoes",
        "Basketball Shoes",
        "Training Shoes",
        "Casual Shoes",
        "T-Shirt",
        "Jacket",
        "Backpack",
        "Monitor",
        "Keyboard",
        "Mouse",
        "Speaker",
        "Charger",
        "Cable",
        "Case",
        "Screen Protector",
    ]

    models = [
        "Pro",
        "Air",
        "Ultra",
        "Max",
        "Plus",
        "Elite",
        "Sport",
        "Classic",
        "Premium",
        "Essential",
        "Basic",
        "Advanced",
        "Professional",
        "Home",
        "Office",
    ]

    colors = [
        "Black",
        "White",
        "Blue",
        "Red",
        "Gray",
        "Silver",
        "Gold",
        "Green",
        "Navy",
        "Pink",
        "Purple",
        "Orange",
        "Yellow",
        "Brown",
        "Beige",
        "Turquoise",
        "Coral",
        "Mint",
    ]

    # Sizes for different product categories
    clothing_sizes = ["XS", "S", "M", "L", "XL", "XXL", "XXXL", "2XL", "3XL"]
    shoe_sizes_us = [
        "6",
        "6.5",
        "7",
        "7.5",
        "8",
        "8.5",
        "9",
        "9.5",
        "10",
        "10.5",
        "11",
        "11.5",
        "12",
        "13",
    ]
    shoe_sizes_eu = ["38", "39", "40", "41", "42", "43", "44", "45", "46"]
    electronic_sizes = [
        '13"',
        '14"',
        '15.6"',
        '17"',
        '21"',
        '24"',
        '27"',
        '32"',
        "Compact",
        "Standard",
        "Large",
    ]
    generic_sizes = ["Small", "Medium", "Large", "Extra Large", "One Size"]

    # Generate categories
    categories = []
    for i in range(20):  # Create 20 categories
        if cat_id_format_choice == "CAT":
            cat_id = f"CAT_{str(i + 100).zfill(3)}"
        elif cat_id_format_choice == "CATEGORY":
            cat_id = f"CATEGORY-{str(i + 100).zfill(3)}"
        else:
            cat_id = str(100 + i)
        categories.append(cat_id)

    # Generate suppliers
    suppliers = []
    for i in range(30):  # Create 30 suppliers
        if supp_id_format_choice == "SUPP":
            supp_id = f"SUPP_{str(i + 1).zfill(3)}"
        elif supp_id_format_choice == "SUPPLIER":
            supp_id = f"SUPPLIER-{str(i + 1).zfill(3)}"
        else:
            supp_id = str(1000 + i)
        suppliers.append(supp_id)

    for i in range(num_rows):
        record = {}

        # Product ID with CONSISTENT format
        if i % 50 == 0 and used_product_ids:  # 2% duplicates
            prod_id = random.choice(used_product_ids)
        else:
            if prod_id_format_choice == "PROD":
                prod_id = f"PROD_{str(i + 1).zfill(4)}"
            elif prod_id_format_choice == "PRODUCT":
                prod_id = f"PRODUCT-{str(i + 1).zfill(4)}"
            else:  # NUMBER
                prod_id = str(10000 + i)

            used_product_ids.append(prod_id)

        record["prod_id"] = prod_id if i % 100 != 0 else None  # 1% null IDs

        # Product Name - NOT NULL in schema
        if i % 80 == 0:  # Violates NOT NULL
            name = None
        elif i % 40 == 0:  # Some duplicates
            if duplicate_names:
                name = random.choice(duplicate_names)
            else:
                brand = random.choice(brands)
                product = random.choice(product_types)
                model = random.choice(models)
                name = f"{brand} {product} {model}"
                duplicate_names.append(name)
        elif i % 30 == 0:  # Invalid values
            name = random.choice(["N/A", "NULL", "Product", "Test", ""])
        elif i % 35 == 0:  # Names with special characters
            brand = random.choice(brands)
            product = random.choice(product_types)
            name = f"{brand} {product}™ #{i}"
        elif i % 45 == 0:  # Names with encoding issues
            brand = random.choice(brands)
            product = random.choice(product_types)
            name = f"{brand} {product} Édition Spéciale"
        elif i % 55 == 0:  # Very long names
            brand = random.choice(brands)
            product = random.choice(product_types)
            model = random.choice(models)
            color = random.choice(colors)
            name = f"{brand} {product} {model} {color} Edition with Extra Features and Extended Warranty Limited Time Offer"
        else:
            brand = random.choice(brands)
            product = random.choice(product_types)
            model = random.choice(models) if random.random() > 0.3 else ""
            name = f"{brand} {product} {model}".strip()

        record["product_description"] = name

        # Determine product category for related attributes
        product_category = None
        if (
            name
            and not isinstance(name, str)
            or name not in ["N/A", "NULL", "Product", "Test", ""]
        ):
            if any(shoe in str(name).lower() for shoe in ["shoe", "sneaker", "boot"]):
                product_category = "footwear"
            elif any(
                cloth in str(name).lower()
                for cloth in ["shirt", "jacket", "pant", "dress", "coat"]
            ):
                product_category = "clothing"
            elif any(
                elec in str(name).lower()
                for elec in ["laptop", "monitor", "tablet", "phone", "watch"]
            ):
                product_category = "electronics"
            else:
                product_category = "general"

        # SKU - should be UNIQUE
        if i % 25 == 0:
            sku = None
        elif i % 35 == 0 and used_skus:  # Duplicate SKUs (violates UNIQUE)
            sku = random.choice(used_skus)
        elif i % 45 == 0:  # Invalid SKU formats
            sku = random.choice(["N/A", "NULL", "SKU", "", "000000"])
        elif i % 55 == 0:  # SKUs with special characters
            sku = f"SKU-{i}!@#"
        else:
            # Generate realistic SKU
            brand_code = name[:2].upper() if name else "XX"
            category_code = random.choice(["EL", "CL", "SP", "AC", "HM"])
            sku = f"{brand_code}-{category_code}{str(i).zfill(4)}-{random.choice(colors)[:3].upper()}"
            if i % 20 == 0:  # Some case variations
                sku = sku.lower()
            used_skus.append(sku)

        record["stock_code"] = sku

        # Category ID
        if i % 30 == 0:
            category = None
        elif i % 40 == 0:  # Non-existent categories (FK violation)
            if cat_id_format_choice == "CAT":
                category = f"CAT_{str(9999).zfill(3)}"
            elif cat_id_format_choice == "CATEGORY":
                category = f"CATEGORY-{str(9999).zfill(3)}"
            else:
                category = "9999"
        elif i % 50 == 0:  # Invalid format
            category = random.choice(["INVALID", "N/A", ""])
        else:
            category = random.choice(categories)

        record["category_ref"] = category

        # Brand
        if i % 25 == 0:
            brand = None
        elif i % 35 == 0:  # Brand inconsistencies
            brand = random.choice(["nike", "NIKE", "Nike Inc.", "Nike®"])
        elif i % 45 == 0:  # Typos
            brand = random.choice(["Addidas", "Appl", "Samung", "Miscrosoft"])
        elif i % 55 == 0:  # Invalid values
            brand = random.choice(["N/A", "Unknown", "Generic", ""])
        else:
            brand = random.choice(brands)

        record["manufacturer"] = brand

        # Supplier ID
        if i % 20 == 0:
            supplier = None
        elif i % 30 == 0:  # Non-existent suppliers (FK violation)
            if supp_id_format_choice == "SUPP":
                supplier = f"SUPP_{str(999).zfill(3)}"
            elif supp_id_format_choice == "SUPPLIER":
                supplier = f"SUPPLIER-{str(999).zfill(3)}"
            else:
                supplier = "9999"
        elif i % 40 == 0:  # Invalid format
            supplier = random.choice(["INVALID", "N/A", "NULL"])
        else:
            supplier = random.choice(suppliers)

        record["vendor_id"] = supplier

        # Cost Price
        if i % 20 == 0:
            cost = None
        elif i % 30 == 0:  # String values instead of decimal
            cost = random.choice(["N/A", "NULL", "Free", ""])
        elif i % 40 == 0:  # Negative values (business logic violation)
            cost = round(random.uniform(-100, -1), 2)
        elif i % 50 == 0:  # Extreme values
            cost = random.choice([0, 999999.99, 0.001, -9999])
        elif i % 60 == 0:  # Cost as integer instead of decimal
            cost = random.randint(10, 500)
        else:
            cost = round(random.uniform(10, 500), 2)

        record["unit_cost"] = cost

        # Selling Price
        if i % 25 == 0:
            price = None
        elif i % 35 == 0:  # String values
            price = random.choice(["N/A", "Contact for price", "TBD"])
        elif i % 45 == 0:  # Business logic violation: selling price less than cost
            if isinstance(cost, (int, float)) and cost > 0:
                price = round(cost * 0.5, 2)  # 50% of cost
            else:
                price = round(random.uniform(1, 10), 2)
        elif i % 55 == 0:  # Extreme values
            price = random.choice([0, 999999.99, 0.01, -100])
        elif i % 65 == 0:  # Price with more than 2 decimal places
            price = round(random.uniform(10, 1000), 5)
        else:
            if isinstance(cost, (int, float)) and cost > 0:
                price = round(cost * random.uniform(1.2, 2.5), 2)
            else:
                price = round(random.uniform(20, 1000), 2)

        record["retail_price"] = price

        # COLOR - New column with related inconsistencies
        if i % 20 == 0:
            color = None
        elif i % 30 == 0:  # Invalid values
            color = random.choice(["N/A", "Unknown", "Various", "", "null"])
        elif i % 40 == 0:  # Case inconsistencies
            color_choice = random.choice(colors)
            color = random.choice(
                [color_choice.upper(), color_choice.lower(), color_choice.title()]
            )
        elif i % 50 == 0:  # Multiple colors
            color = f"{random.choice(colors)}/{random.choice(colors)}"
        elif i % 60 == 0:  # Color codes instead of names
            color = random.choice(["#000000", "#FFFFFF", "#FF0000", "RGB(255,0,0)"])
        elif i % 70 == 0:  # Typos and variations
            color = random.choice(
                ["Balck", "Whtie", "Grey", "Blu", "Red-ish", "Dark Blue"]
            )
        elif i % 80 == 0:  # Special characters
            color = f"{random.choice(colors)}™"
        elif i % 90 == 0:  # Very long color descriptions
            base_color = random.choice(colors)
            color = f"{base_color} with {random.choice(colors)} accents and {random.choice(colors)} trim"
        else:
            # Related to product category
            if product_category == "electronics":
                color = random.choice(
                    ["Black", "Silver", "White", "Space Gray", "Gold"]
                )
            elif product_category == "footwear":
                color = random.choice(
                    ["Black", "White", "Blue", "Red", "Gray", "Black/White", "Navy/Red"]
                )
            else:
                color = random.choice(colors)

        record["color"] = color

        # SIZE - New column with category-appropriate values
        if i % 25 == 0:
            size = None
        elif i % 35 == 0:  # Invalid values
            size = random.choice(["N/A", "One Size Fits All", "Standard", "", "NA"])
        elif i % 45 == 0:  # Wrong format for category
            if product_category == "footwear":
                size = random.choice(clothing_sizes)  # Wrong: clothing size for shoes
            elif product_category == "clothing":
                size = random.choice(shoe_sizes_us)  # Wrong: shoe size for clothing
            elif product_category == "electronics":
                size = random.choice(
                    clothing_sizes
                )  # Wrong: clothing size for electronics
            else:
                size = str(random.randint(1, 100))
        elif i % 55 == 0:  # Mixed formats
            if product_category == "footwear":
                size = f"US {random.choice(shoe_sizes_us)} / EU {random.choice(shoe_sizes_eu)}"
            else:
                size = f"{random.choice(generic_sizes)} ({random.choice(['S', 'M', 'L', 'XL'])})"
        elif i % 65 == 0:  # Case variations
            size_choice = random.choice(clothing_sizes + generic_sizes)
            size = random.choice(
                [size_choice.upper(), size_choice.lower(), size_choice.title()]
            )
        elif i % 75 == 0:  # Numeric for clothing
            size = str(
                random.choice([36, 38, 40, 42, 44, 46, 48])
            )  # European clothing sizes
        elif i % 85 == 0:  # Special characters or typos
            size = random.choice(
                ["Smal", "Mediun", "Larg", "X-Large", "XX-Large", "Med."]
            )
        else:
            # Appropriate size for category
            if product_category == "footwear":
                size = random.choice(shoe_sizes_us + shoe_sizes_eu)
            elif product_category == "clothing":
                size = random.choice(clothing_sizes)
            elif product_category == "electronics":
                size = random.choice(electronic_sizes)
            else:
                size = random.choice(generic_sizes)

        record["size"] = size

        # Weight (DECIMAL(8,3) format) - Enhanced version
        if i % 20 == 0:
            weight = None
        elif i % 30 == 0:  # String values
            weight = random.choice(
                ["N/A", "Unknown", "Varies", "TBD", "Light", "Heavy"]
            )
        elif i % 40 == 0:  # Negative values (invalid)
            weight = round(random.uniform(-10, -0.001), 3)
        elif i % 50 == 0:  # Extreme values
            weight = random.choice(
                [0, 99999.999, 0.0001, 100000]
            )  # Last one exceeds DECIMAL(8,3)
        elif i % 60 == 0:  # Wrong units mixed in
            weight = f"{round(random.uniform(0.1, 10), 3)} kg"
        elif i % 70 == 0:  # Too many decimal places
            weight = round(random.uniform(0.1, 50), 6)
        elif i % 80 == 0:  # Weight doesn't match product category
            if product_category == "electronics" and "laptop" in str(name).lower():
                weight = round(random.uniform(10, 50), 3)  # Too heavy for a laptop
            elif product_category == "footwear":
                weight = round(random.uniform(5, 20), 3)  # Too heavy for shoes
            else:
                weight = round(random.uniform(0.001, 0.01), 3)  # Too light
        else:
            # Realistic weights based on product category
            if product_category == "electronics":
                if "laptop" in str(name).lower():
                    weight = round(random.uniform(1.2, 3.5), 3)
                elif "phone" in str(name).lower():
                    weight = round(random.uniform(0.1, 0.3), 3)
                else:
                    weight = round(random.uniform(0.5, 5), 3)
            elif product_category == "footwear":
                weight = round(random.uniform(0.2, 1.5), 3)
            elif product_category == "clothing":
                weight = round(random.uniform(0.1, 2), 3)
            else:
                weight = round(random.uniform(0.1, 10), 3)

        record["weight"] = weight

        # Dimensions (VARCHAR(50) - 'LxWxH') - Enhanced version
        if i % 25 == 0:
            dimensions = None
        elif i % 35 == 0:  # Invalid format
            dimensions = random.choice(
                ["Large", "Small", "N/A", "Compact", "Standard Size"]
            )
        elif i % 45 == 0:  # Wrong separators
            l = random.randint(5, 100)
            w = random.randint(5, 100)
            h = random.randint(5, 100)
            dimensions = random.choice(
                [f"{l}-{w}-{h}", f"{l}/{w}/{h}", f"{l},{w},{h}", f"{l} x {w} x {h}"]
            )
        elif i % 55 == 0:  # Missing dimensions
            dimensions = random.choice(
                [
                    f"{random.randint(10, 50)}x{random.randint(10, 50)}",
                    f"x{random.randint(10, 50)}x{random.randint(10, 50)}",
                ]
            )
        elif i % 65 == 0:  # With units mixed in
            l = random.randint(5, 100)
            w = random.randint(5, 100)
            h = random.randint(5, 100)
            dimensions = f"{l}cm x {w}cm x {h}cm"
        elif i % 75 == 0:  # Extreme or invalid values
            dimensions = random.choice(
                ["0x0x0", "9999x9999x9999", "-10x-10x-10", "1x1x1"]
            )
        elif i % 85 == 0:  # Too long for VARCHAR(50)
            dimensions = f"{random.randint(100, 999)}x{random.randint(100, 999)}x{random.randint(100, 999)} centimeters (LxWxH) approx"
        else:
            # Realistic dimensions based on category
            if product_category == "electronics":
                if "laptop" in str(name).lower():
                    l = random.randint(30, 40)
                    w = random.randint(20, 30)
                    h = random.randint(1, 3)
                elif "phone" in str(name).lower():
                    l = random.randint(12, 18)
                    w = random.randint(6, 9)
                    h = round(random.uniform(0.5, 1.5), 1)
                else:
                    l = random.randint(10, 50)
                    w = random.randint(10, 50)
                    h = random.randint(5, 30)
            elif product_category == "footwear":
                # Shoe box dimensions
                l = random.randint(25, 35)
                w = random.randint(15, 20)
                h = random.randint(10, 15)
            else:
                l = random.randint(5, 100)
                w = random.randint(5, 100)
                h = random.randint(5, 100)

            dimensions = f"{l}x{w}x{h}"

        record["dimensions"] = dimensions

        # Launch Date
        if i % 30 == 0:
            launch = None
        elif i % 40 == 0:  # String format variations
            launch_date = fake.date_between(start_date="-5y", end_date="today")
            formats = ["%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y", "%Y%m%d", "%b %d, %Y"]
            launch = launch_date.strftime(random.choice(formats))
        elif i % 50 == 0:  # Invalid dates
            launch = random.choice(
                ["0000-00-00", "9999-99-99", "99-99-9999", "Invalid"]
            )
        elif i % 60 == 0:  # Future launch dates (could be valid for pre-orders)
            launch = fake.date_between(start_date="today", end_date="+1y")
        elif i % 70 == 0:  # Very old dates (business logic issue)
            launch = fake.date_between(start_date="-50y", end_date="-30y")
        else:
            launch = fake.date_between(start_date="-5y", end_date="today")

        record["release_date"] = launch

        # Product Status
        if i % 25 == 0:
            status = None
        elif i % 35 == 0:  # Inconsistent values
            status = random.choice(
                ["active", "ACTIVE", "1", "A", "discontinued", "DISCONTINUED"]
            )
        elif i % 45 == 0:  # Invalid values
            status = random.choice(["Available", "Unavailable", "In Stock", "Sold Out"])
        elif i % 55 == 0:  # Typos
            status = random.choice(["Activ", "Discontined", "Out of Stok"])
        else:
            status = random.choice(["Active", "Discontinued", "Out of Stock"])

        record["availability_status"] = status

        # Is Digital
        if i % 30 == 0:
            digital = None
        elif i % 40 == 0:  # Various boolean representations
            digital = random.choice(["Y", "N", "Yes", "No", "1", "0", "true", "false"])
        elif i % 50 == 0:  # Invalid values
            digital = random.choice(["Maybe", "Unknown", "Physical"])
        else:
            digital = random.choice([True, False])

        record["digital_product"] = digital

        # Average Rating
        if i % 20 == 0:
            rating = None
        elif i % 30 == 0:  # String values
            rating = random.choice(["N/A", "No ratings", ""])
        elif i % 40 == 0:  # Values outside valid range (should be 0-5)
            rating = random.choice([-1, 6, 10, 999])
        elif i % 50 == 0:  # Invalid precision
            rating = round(random.uniform(0, 5), 5)  # Too many decimal places
        elif i % 60 == 0:  # Business logic violation: rating with 0 reviews
            rating = round(random.uniform(3, 5), 2)
            # Will set reviews to 0 later
        else:
            rating = round(random.uniform(1, 5), 2)

        record["avg_customer_rating"] = rating

        # Total Reviews
        if i % 25 == 0:
            reviews = None
        elif i % 35 == 0:  # String values
            reviews = random.choice(["N/A", "None", ""])
        elif i % 45 == 0:  # Negative values (business logic violation)
            reviews = random.randint(-100, -1)
        elif i % 55 == 0:  # Extreme values
            reviews = random.choice([999999, 0.5, -999])
        elif i % 60 == 0:  # Business logic violation: 0 reviews with rating
            reviews = 0
            if record.get("avg_customer_rating") is None:
                record["avg_customer_rating"] = 4.5  # Has rating but no reviews
        else:
            if isinstance(rating, (int, float)) and rating > 0:
                # More reviews for better ratings
                if rating >= 4:
                    reviews = random.randint(10, 1000)
                else:
                    reviews = random.randint(1, 100)
            else:
                reviews = 0

        record["review_count"] = reviews

        # Created timestamp
        if i % 30 == 0:
            created = None
        elif i % 40 == 0:  # String format
            created = fake.date_time_between(start_date="-1y", end_date="now").strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        elif i % 50 == 0:  # Unix timestamp
            created = int(
                fake.date_time_between(start_date="-1y", end_date="now").timestamp()
            )
        elif i % 60 == 0:  # Future date (business logic violation)
            created = fake.date_time_between(start_date="+1y", end_date="+2y")
        elif i % 70 == 0:  # Business logic violation: created before launch
            if record.get("release_date") and not isinstance(
                record["release_date"], str
            ):
                created = record["release_date"] - timedelta(
                    days=random.randint(30, 365)
                )
            else:
                created = fake.date_time_between(start_date="-2y", end_date="-1y")
        else:
            created = fake.date_time_between(start_date="-1y", end_date="now")

        record["record_created_at"] = created

        # Stock quantity
        if random.random() > 0.5:
            if i % 30 == 0:
                stock = None
            elif i % 40 == 0:  # String values
                stock = random.choice(["In Stock", "Out of Stock", "Limited"])
            elif i % 50 == 0:  # Negative values
                stock = random.randint(-100, -1)
            elif (
                i % 60 == 0
            ):  # Business logic violation: Out of Stock status with positive quantity
                if record.get("availability_status") in [
                    "Out of Stock",
                    "out of stock",
                ]:
                    stock = random.randint(10, 100)
                else:
                    stock = 0
            elif i % 70 == 0:  # Extreme values
                stock = random.choice([999999, 0.5, -9999])
            else:
                stock = random.randint(0, 1000)
            record["inventory_qty"] = stock

        # Discount percentage
        if random.random() > 0.6:
            if i % 35 == 0:
                discount = None
            elif i % 45 == 0:  # Invalid values
                discount = random.choice(["Sale", "Clearance", "N/A"])
            elif i % 55 == 0:  # Values outside 0-100 range
                discount = random.choice([-50, 150, 999])
            elif i % 65 == 0:  # Business logic violation: discount makes price negative
                discount = 110
            else:
                discount = random.randint(0, 50)
            record["discount_pct"] = discount

        data.append(record)

    # Create DataFrame
    df = pd.DataFrame(data)

    # Add EXACT duplicate rows (every attribute is same)
    num_exact_duplicates = int(num_rows * 0.02)  # 2% exact duplicates
    for _ in range(num_exact_duplicates):
        if len(df) > 0:
            # Pick a random row to duplicate
            row_to_duplicate = df.sample(1)
            df = pd.concat([df, row_to_duplicate], ignore_index=True)

    # Add some completely empty rows
    for _ in range(int(num_rows * 0.005)):  # 0.5% empty rows
        empty_row = pd.Series([None] * len(df.columns), index=df.columns)
        df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)

    # Add some rows with all string 'NULL' or 'N/A' values
    for _ in range(int(num_rows * 0.01)):  # 1% NULL string rows
        null_values = ["NULL", "N/A", "null", "NA", "", " "]
        null_row = pd.Series(
            [random.choice(null_values) for _ in range(len(df.columns))],
            index=df.columns,
        )
        df = pd.concat([df, pd.DataFrame([null_row])], ignore_index=True)

    # Shuffle the dataframe to mix duplicates throughout
    df = df.sample(frac=1).reset_index(drop=True)

    return df


def add_more_messiness(df):
    """
    Add additional data quality issues to make the dataset more challenging
    """
    # Add trailing/leading spaces to some string columns
    string_cols = df.select_dtypes(include=["object"]).columns
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05  # 5% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    # Add case inconsistencies
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03  # 3% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    # Add special characters to some values
    for col in string_cols[:3]:  # Only first 3 string columns
        mask = np.random.random(len(df)) < 0.02  # 2% of values
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


# Generate the dataset
if __name__ == "__main__":
    # Set the number of rows you want
    NUM_ROWS = 1000  # Change this to your desired number

    print(f"Generating {NUM_ROWS} rows of messy product data...")
    df = generate_messy_product_data(NUM_ROWS)

    # Add more messiness
    df = add_more_messiness(df)

    # Display basic info
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumn names (realistic but challenging for mapping):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")

    print(f"\nFirst 10 rows:")
    print(df.head(10))

    # Show data quality issues summary
    print("\n" + "=" * 50)
    print("DATA QUALITY ISSUES SUMMARY:")
    print("=" * 50)
    print(f"Total null values: {df.isnull().sum().sum()}")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    print(f"Total rows: {len(df)}")

    # Check for new columns analysis
    print("\n" + "=" * 50)
    print("NEW COLUMNS ANALYSIS:")
    print("=" * 50)

    # Analyze color column
    if "color" in df.columns:
        print("\nColor column:")
        print(f"  - Null values: {df['color'].isnull().sum()}")
        print(f"  - Unique values: {df['color'].nunique()}")
        print(f"  - Sample values: {df['color'].dropna().unique()[:10].tolist()}")
        # Check for inconsistencies
        color_issues = df[df["color"].isin(["N/A", "Unknown", "Various", "null", ""])]
        print(f"  - Invalid values: {len(color_issues)} rows")

    # Analyze size column
    if "size" in df.columns:
        print("\nSize column:")
        print(f"  - Null values: {df['size'].isnull().sum()}")
        print(f"  - Unique values: {df['size'].nunique()}")
        print(f"  - Sample values: {df['size'].dropna().unique()[:10].tolist()}")
        size_issues = df[df["size"].isin(["N/A", "NA", "", "Standard"])]
        print(f"  - Invalid values: {len(size_issues)} rows")

    # Analyze weight column
    if "weight" in df.columns:
        print("\nWeight column (DECIMAL(8,3)):")
        print(f"  - Null values: {df['weight'].isnull().sum()}")
        print(f"  - Unique values: {df['weight'].nunique()}")
        # Check for string values
        weight_strings = df[
            df["weight"].apply(lambda x: isinstance(x, str) and x not in ["", None])
        ]
        print(f"  - String values instead of decimal: {len(weight_strings)} rows")
        if len(weight_strings) > 0:
            print(f"    Examples: {weight_strings['weight'].unique()[:5].tolist()}")
        # Check for negative weights
        weight_numeric = pd.to_numeric(df["weight"], errors="coerce")
        negative_weights = df[weight_numeric < 0]
        print(f"  - Negative weights: {len(negative_weights)} rows")
        # Check for extreme values
        extreme_weights = df[(weight_numeric > 10000) | (weight_numeric == 0)]
        print(f"  - Extreme values (0 or >10000): {len(extreme_weights)} rows")

    # Analyze dimensions column
    if "dimensions" in df.columns:
        print("\nDimensions column (VARCHAR(50) - LxWxH):")
        print(f"  - Null values: {df['dimensions'].isnull().sum()}")
        print(f"  - Unique values: {df['dimensions'].nunique()}")
        # Check for invalid formats
        valid_format = df["dimensions"].str.match(r"^\d+x\d+x\d+$", na=False)
        invalid_format = df[~valid_format & df["dimensions"].notna()]
        print(f"  - Invalid format (not NxNxN): {len(invalid_format)} rows")
        if len(invalid_format) > 0:
            print(f"    Examples: {invalid_format['dimensions'].unique()[:5].tolist()}")
        # Check for length violations
        too_long = df[df["dimensions"].str.len() > 50]
        print(f"  - Exceeds VARCHAR(50): {len(too_long)} rows")

    # Check UNIQUE constraint violations
    print("\n" + "=" * 50)
    print("UNIQUE CONSTRAINT VIOLATIONS:")
    print("=" * 50)

    # Check for duplicate SKUs (should be unique)
    if "stock_code" in df.columns:
        sku_duplicates = df[
            df.duplicated(subset=["stock_code"], keep=False) & df["stock_code"].notna()
        ]
        if len(sku_duplicates) > 0:
            print(f"✗ Duplicate SKUs (violates UNIQUE): {len(sku_duplicates)} rows")
            duplicate_skus = sku_duplicates["stock_code"].value_counts().head(3)
            print(f"  Most duplicated SKUs: {duplicate_skus.to_dict()}")

    # Check NOT NULL constraint violations
    print("\n" + "=" * 50)
    print("NOT NULL CONSTRAINT VIOLATIONS:")
    print("=" * 50)

    # product_description should not be null
    if "product_description" in df.columns:
        null_names = df[df["product_description"].isnull()].shape[0]
        if null_names > 0:
            print(f"✗ NULL product_description (violates NOT NULL): {null_names} rows")

    # Show business logic violations
    print("\n" + "=" * 50)
    print("BUSINESS LOGIC VIOLATIONS EXAMPLES:")
    print("=" * 50)

    # Check for selling price less than cost price
    if "unit_cost" in df.columns and "retail_price" in df.columns:
        cost_numeric = pd.to_numeric(df["unit_cost"], errors="coerce")
        price_numeric = pd.to_numeric(df["retail_price"], errors="coerce")
        price_below_cost = df[(cost_numeric > 0) & (price_numeric < cost_numeric)]
        if len(price_below_cost) > 0:
            print(f"✗ Selling price below cost: {len(price_below_cost)} products")
            examples = price_below_cost[["unit_cost", "retail_price"]].head(3)
            print(f"  Examples (cost -> price): {examples.values.tolist()}")

    # Check for mismatched color/size with product category
    if "product_description" in df.columns and "size" in df.columns:
        # Find shoes with clothing sizes
        shoe_products = df[
            df["product_description"].str.contains("Shoe|shoe", case=False, na=False)
        ]
        shoe_with_clothing_size = shoe_products[
            shoe_products["size"].isin(["XS", "S", "M", "L", "XL", "XXL"])
        ]
        if len(shoe_with_clothing_size) > 0:
            print(
                f"✗ Shoes with clothing sizes: {len(shoe_with_clothing_size)} products"
            )

        # Find clothing with shoe sizes
        clothing_products = df[
            df["product_description"].str.contains(
                "Shirt|Jacket|shirt|jacket", case=False, na=False
            )
        ]
        clothing_with_shoe_size = clothing_products[
            clothing_products["size"].str.match(r"^\d+\.?\d*$", na=False)
        ]
        if len(clothing_with_shoe_size) > 0:
            print(
                f"✗ Clothing with numeric (shoe) sizes: {len(clothing_with_shoe_size)} products"
            )

    # Check for ratings with no reviews
    if "avg_customer_rating" in df.columns and "review_count" in df.columns:
        rating_numeric = pd.to_numeric(df["avg_customer_rating"], errors="coerce")
        reviews_numeric = pd.to_numeric(df["review_count"], errors="coerce")
        rating_no_reviews = df[(rating_numeric > 0) & (reviews_numeric == 0)]
        if len(rating_no_reviews) > 0:
            print(
                f"✗ Products with ratings but no reviews: {len(rating_no_reviews)} products"
            )

    # Check for out of stock with positive inventory
    if "availability_status" in df.columns and "inventory_qty" in df.columns:
        qty_numeric = pd.to_numeric(df["inventory_qty"], errors="coerce")
        out_with_stock = df[
            (
                df["availability_status"].isin(
                    ["Out of Stock", "out of stock", "Out of Stok"]
                )
            )
            & (qty_numeric > 0)
        ]
        if len(out_with_stock) > 0:
            print(
                f"✗ Out of Stock status with positive inventory: {len(out_with_stock)} products"
            )

    # Save to CSV
    output_file = "messy_product_data.xlsx"
    df.to_excel(output_file, index=False)
    print(f"\n✅ Dataset saved to '{output_file}'")

    # Save product-category-supplier summary for testing joins
    summary_data = []
    for cat in df["category_ref"].dropna().unique()[:10]:
        cat_products = df[df["category_ref"] == cat]
        summary_data.append(
            {
                "category": cat,
                "product_count": len(cat_products),
                "avg_price": cat_products["retail_price"]
                .apply(pd.to_numeric, errors="coerce")
                .mean(),
                "total_stock": (
                    cat_products["inventory_qty"]
                    .apply(pd.to_numeric, errors="coerce")
                    .sum()
                    if "inventory_qty" in df.columns
                    else 0
                ),
                "popular_colors": (
                    cat_products["color"].mode().values[0]
                    if "color" in df.columns and len(cat_products["color"].mode()) > 0
                    else "N/A"
                ),
                "popular_sizes": (
                    cat_products["size"].mode().values[0]
                    if "size" in df.columns and len(cat_products["size"].mode()) > 0
                    else "N/A"
                ),
            }
        )

    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        summary_df.to_csv("product_category_summary_enhanced.csv", index=False)
        print(
            f"✅ Product-category summary saved to 'product_category_summary_enhanced.csv'"
        )

Generating 1000 rows of messy product data...

Dataset shape: (1035, 20)

Column names (realistic but challenging for mapping):
  1. prod_id
  2. product_description
  3. stock_code
  4. category_ref
  5. manufacturer
  6. vendor_id
  7. unit_cost
  8. retail_price
  9. color
  10. size
  11. weight
  12. dimensions
  13. release_date
  14. availability_status
  15. digital_product
  16. avg_customer_rating
  17. review_count
  18. record_created_at
  19. inventory_qty
  20. discount_pct

First 10 rows:
  prod_id          product_description     stock_code category_ref  \
0   10369                    JBL Cable  JB-EL0369-GRE      CAT_115   
1   10078  Google Basketball Shoes Pro  GO-CL0078-BRO      CAT_119   
2   10299      Adidas Screen Protector  AD-EL0299-WHI      CAT_105   
3   10890  Google Smartphone Essential  GO-SP0890-GRE      CAT_107   
4   10821    JBL Keyboard Professional  JB-EL0821-MIN      CAT_101   
5   10931  Adidas Training Shoes Elite  AD-SP0931-GRA      CAT_103   
6

### Categories Table Generator

In [2]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import string

# Initialize Faker
fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)


def generate_messy_category_data(num_rows=200):
    """
    Generate a messy, realistic category dataset with realistic but challenging column names
    and various data quality issues for testing data mapping and cleaning.

    Args:
        num_rows: Number of category records to generate
    """

    data = []

    # Track IDs for creating relationships and duplicates
    used_category_ids = []
    root_categories = []  # Top-level categories
    mid_categories = []  # Second-level categories
    duplicate_names = []

    # Choose one consistent ID format for the entire dataset
    cat_id_format_choice = random.choice(["CAT", "CATEGORY", "NUMBER"])

    # Realistic category hierarchy
    category_hierarchy = {
        "Electronics": {
            "subcategories": [
                "Computers",
                "Mobile Devices",
                "Audio",
                "Cameras",
                "Gaming",
                "Accessories",
            ],
            "sub_subcategories": {
                "Computers": [
                    "Laptops",
                    "Desktops",
                    "Tablets",
                    "Monitors",
                    "Computer Accessories",
                ],
                "Mobile Devices": [
                    "Smartphones",
                    "Smart Watches",
                    "Phone Cases",
                    "Chargers",
                    "Screen Protectors",
                ],
                "Audio": [
                    "Headphones",
                    "Speakers",
                    "Soundbars",
                    "Earbuds",
                    "Microphones",
                ],
                "Cameras": [
                    "DSLR",
                    "Mirrorless",
                    "Action Cameras",
                    "Camera Lenses",
                    "Camera Accessories",
                ],
                "Gaming": [
                    "Consoles",
                    "PC Gaming",
                    "Gaming Accessories",
                    "Gaming Chairs",
                    "VR Equipment",
                ],
                "Accessories": [
                    "Cables",
                    "Adapters",
                    "Power Banks",
                    "Memory Cards",
                    "USB Drives",
                ],
            },
        },
        "Clothing": {
            "subcategories": [
                "Men's Clothing",
                "Women's Clothing",
                "Kids' Clothing",
                "Shoes",
                "Accessories",
            ],
            "sub_subcategories": {
                "Men's Clothing": ["T-Shirts", "Shirts", "Pants", "Jackets", "Suits"],
                "Women's Clothing": ["Dresses", "Tops", "Pants", "Skirts", "Jackets"],
                "Kids' Clothing": [
                    "Boys Clothing",
                    "Girls Clothing",
                    "Baby Clothing",
                    "School Uniforms",
                ],
                "Shoes": [
                    "Running Shoes",
                    "Casual Shoes",
                    "Formal Shoes",
                    "Boots",
                    "Sandals",
                ],
                "Accessories": ["Belts", "Wallets", "Watches", "Sunglasses", "Hats"],
            },
        },
        "Sports & Outdoors": {
            "subcategories": [
                "Exercise & Fitness",
                "Outdoor Recreation",
                "Team Sports",
                "Water Sports",
                "Winter Sports",
            ],
            "sub_subcategories": {
                "Exercise & Fitness": [
                    "Gym Equipment",
                    "Yoga",
                    "Running Gear",
                    "Fitness Trackers",
                    "Supplements",
                ],
                "Outdoor Recreation": [
                    "Camping",
                    "Hiking",
                    "Cycling",
                    "Climbing",
                    "Fishing",
                ],
                "Team Sports": [
                    "Basketball",
                    "Football",
                    "Soccer",
                    "Baseball",
                    "Volleyball",
                ],
                "Water Sports": [
                    "Swimming",
                    "Surfing",
                    "Diving",
                    "Kayaking",
                    "Water Polo",
                ],
                "Winter Sports": [
                    "Skiing",
                    "Snowboarding",
                    "Ice Skating",
                    "Hockey",
                    "Sledding",
                ],
            },
        },
        "Home & Garden": {
            "subcategories": ["Furniture", "Kitchen", "Bedroom", "Bathroom", "Garden"],
            "sub_subcategories": {
                "Furniture": [
                    "Living Room",
                    "Dining Room",
                    "Office Furniture",
                    "Outdoor Furniture",
                    "Storage",
                ],
                "Kitchen": [
                    "Cookware",
                    "Appliances",
                    "Utensils",
                    "Storage Containers",
                    "Dinnerware",
                ],
                "Bedroom": [
                    "Mattresses",
                    "Bedding",
                    "Pillows",
                    "Nightstands",
                    "Dressers",
                ],
                "Bathroom": [
                    "Towels",
                    "Shower Accessories",
                    "Bath Mats",
                    "Storage",
                    "Toiletries",
                ],
                "Garden": [
                    "Plants",
                    "Garden Tools",
                    "Outdoor Decor",
                    "Grills",
                    "Patio Sets",
                ],
            },
        },
        "Books & Media": {
            "subcategories": ["Books", "E-Books", "Audiobooks", "Movies", "Music"],
            "sub_subcategories": {
                "Books": [
                    "Fiction",
                    "Non-Fiction",
                    "Textbooks",
                    "Children Books",
                    "Comics",
                ],
                "E-Books": [
                    "Digital Fiction",
                    "Digital Non-Fiction",
                    "Digital Magazines",
                    "Digital Comics",
                ],
                "Audiobooks": [
                    "Fiction Audio",
                    "Non-Fiction Audio",
                    "Podcasts",
                    "Language Learning",
                ],
                "Movies": [
                    "DVDs",
                    "Blu-ray",
                    "Digital Movies",
                    "4K Movies",
                    "TV Series",
                ],
                "Music": [
                    "CDs",
                    "Vinyl",
                    "Digital Music",
                    "Music Instruments",
                    "Sheet Music",
                ],
            },
        },
    }

    # First, create root categories
    root_count = min(len(category_hierarchy), num_rows // 10)
    for i, (root_name, _) in enumerate(list(category_hierarchy.items())[:root_count]):
        record = {}

        # Category ID
        if cat_id_format_choice == "CAT":
            cat_id = f"CAT_{str(i).zfill(3)}"
        elif cat_id_format_choice == "CATEGORY":
            cat_id = f"CATEGORY-{str(i).zfill(3)}"
        else:
            cat_id = str(100 + i)

        used_category_ids.append(cat_id)
        root_categories.append(cat_id)

        record["cat_id"] = cat_id
        record["cat_name"] = root_name
        record["parent_cat_id"] = None  # Root categories have no parent
        record["active_flag"] = True
        record["date_created"] = fake.date_time_between(
            start_date="-3y", end_date="-2y"
        )

        data.append(record)

    # Create remaining categories with various issues
    current_id = len(data)
    while len(data) < num_rows:
        record = {}
        i = current_id
        current_id += 1

        # Category ID with CONSISTENT format
        if (
            i % 50 == 0 and used_category_ids and len(used_category_ids) > 5
        ):  # 2% duplicates
            cat_id = random.choice(used_category_ids)
        else:
            if cat_id_format_choice == "CAT":
                cat_id = f"CAT_{str(i).zfill(3)}"
            elif cat_id_format_choice == "CATEGORY":
                cat_id = f"CATEGORY-{str(i).zfill(3)}"
            else:
                cat_id = str(100 + i)

            used_category_ids.append(cat_id)

        record["cat_id"] = cat_id if i % 100 != 0 else None  # 1% null IDs

        # Category Name - NOT NULL in schema
        if i % 80 == 0:  # Violates NOT NULL
            name = None
        elif i % 40 == 0 and duplicate_names:  # Some duplicates
            name = random.choice(duplicate_names)
        elif i % 30 == 0:  # Invalid values
            name = random.choice(
                ["N/A", "NULL", "Category", "Test", "", "Uncategorized"]
            )
        elif i % 35 == 0:  # Names with special characters
            base_name = random.choice(["Electronics", "Clothing", "Sports"])
            name = f"{base_name}™ #{i}"
        elif i % 45 == 0:  # Names with encoding issues
            name = random.choice(
                ["Électronique", "Deportes y Recreación", "Möbel & Garten"]
            )
        elif i % 55 == 0:  # Very long names
            name = "Super Ultra Mega Premium Deluxe Category for Special Items and Products with Extended Features"
        elif i % 25 == 0:  # Case variations
            name = random.choice(["electronics", "ELECTRONICS", "ElEcTrOnIcS"])
        elif i % 65 == 0:  # Typos
            name = random.choice(
                ["Electonics", "Colthing", "Spoorts", "Furiture", "Graden"]
            )
        else:
            # Generate realistic subcategory names
            if random.random() < 0.3 and root_categories:  # 30% are subcategories
                parent_root = random.choice(list(category_hierarchy.keys()))
                if (
                    parent_root in category_hierarchy
                    and category_hierarchy[parent_root]["subcategories"]
                ):
                    name = random.choice(
                        category_hierarchy[parent_root]["subcategories"]
                    )
                    if name not in duplicate_names and random.random() < 0.1:
                        duplicate_names.append(name)
            elif random.random() < 0.5:  # Sub-subcategories
                parent_root = random.choice(list(category_hierarchy.keys()))
                if parent_root in category_hierarchy:
                    subcat = random.choice(
                        list(
                            category_hierarchy[parent_root]["sub_subcategories"].keys()
                        )
                    )
                    if subcat in category_hierarchy[parent_root]["sub_subcategories"]:
                        name = random.choice(
                            category_hierarchy[parent_root]["sub_subcategories"][subcat]
                        )
            else:
                # Random category names
                name = random.choice(
                    [
                        "Sale Items",
                        "New Arrivals",
                        "Clearance",
                        "Featured Products",
                        "Best Sellers",
                        "Limited Edition",
                        "Seasonal",
                        "Gift Ideas",
                        "Premium Collection",
                        "Budget Friendly",
                        "Eco Friendly",
                        "Luxury Items",
                    ]
                )

        record["cat_name"] = name

        # Parent Category ID (hierarchical relationship)
        if i % 20 == 0:  # Some nulls (root categories)
            parent = None
            if cat_id and cat_id not in root_categories:
                root_categories.append(cat_id)
        elif i % 30 == 0:  # Invalid parent references (orphaned categories)
            if cat_id_format_choice == "CAT":
                parent = f"CAT_{str(9999).zfill(3)}"
            elif cat_id_format_choice == "CATEGORY":
                parent = f"CATEGORY-{str(9999).zfill(3)}"
            else:
                parent = "9999"
        elif i % 40 == 0:  # Self-reference (business logic violation)
            parent = cat_id
        elif i % 50 == 0:  # Circular reference (A->B->A)
            if len(used_category_ids) > 1:
                # Create potential circular reference
                parent = (
                    random.choice(used_category_ids[-5:])
                    if len(used_category_ids) > 5
                    else None
                )
            else:
                parent = None
        elif i % 60 == 0:  # Invalid format
            parent = random.choice(["INVALID", "N/A", "NULL", ""])
        elif i % 70 == 0:  # Very deep nesting (business logic issue)
            # Reference the most recent category to create deep chains
            if used_category_ids and len(used_category_ids) > 1:
                parent = used_category_ids[-2]
            else:
                parent = None
        else:
            # Normal parent reference
            if random.random() < 0.2:  # 20% are root categories
                parent = None
                if cat_id and cat_id not in root_categories:
                    root_categories.append(cat_id)
            elif root_categories:
                if random.random() < 0.4:  # 40% are direct children of root
                    parent = random.choice(root_categories)
                    if cat_id and cat_id not in mid_categories:
                        mid_categories.append(cat_id)
                elif mid_categories:  # Rest are third level
                    parent = random.choice(mid_categories)
                else:
                    parent = random.choice(root_categories)
            else:
                parent = None

        record["parent_cat_id"] = parent

        # Is Active flag
        if i % 25 == 0:
            active = None
        elif i % 35 == 0:  # Various boolean representations
            active = random.choice(
                ["Y", "N", "Yes", "No", "1", "0", "true", "false", "T", "F"]
            )
        elif i % 45 == 0:  # Invalid values
            active = random.choice(["Maybe", "Unknown", "Pending", ""])
        elif i % 55 == 0:  # Business logic violation: inactive parent with active child
            if parent and parent in used_category_ids:
                # Make this active but we'll mark some parents as inactive
                active = True
            else:
                active = False
        else:
            # Some categories are inactive
            active = random.choice([True, True, True, False])  # 75% active

        record["active_flag"] = active

        # Created timestamp
        if i % 30 == 0:
            created = None
        elif i % 40 == 0:  # String format
            created = fake.date_time_between(start_date="-2y", end_date="now").strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        elif i % 50 == 0:  # Unix timestamp
            created = int(
                fake.date_time_between(start_date="-2y", end_date="now").timestamp()
            )
        elif i % 60 == 0:  # Future date (business logic violation)
            created = fake.date_time_between(start_date="+1y", end_date="+2y")
        elif i % 70 == 0:  # Very old date
            created = fake.date_time_between(start_date="-50y", end_date="-30y")
        elif i % 80 == 0:  # Business logic violation: child created before parent
            # Create a date older than typical parent dates
            created = fake.date_time_between(start_date="-5y", end_date="-3y")
        else:
            created = fake.date_time_between(start_date="-2y", end_date="now")

        record["date_created"] = created

        # Additional realistic columns that might exist

        # Category description
        if random.random() > 0.3:  # 70% have description
            if i % 35 == 0:
                description = None
            elif i % 45 == 0:
                description = random.choice(["N/A", "No description", ""])
            elif i % 55 == 0:  # Very long description
                description = fake.text(max_nb_chars=500)
            else:
                description = fake.sentence(nb_words=10)
            record["category_desc"] = description

        # Display order/sequence
        if random.random() > 0.4:  # 60% have display order
            if i % 40 == 0:
                order = None
            elif i % 50 == 0:  # String instead of number
                order = random.choice(["First", "Last", "N/A"])
            elif i % 60 == 0:  # Negative values
                order = random.randint(-100, -1)
            elif i % 70 == 0:  # Extreme values
                order = random.choice([999999, 0.5, -9999])
            elif i % 80 == 0:  # Duplicate orders (business logic issue)
                order = 1  # Many categories with same order
            else:
                order = random.randint(1, 100)
            record["display_sequence"] = order

        # Category level (depth in hierarchy)
        if random.random() > 0.5:  # 50% have level
            if i % 45 == 0:
                level = None
            elif i % 55 == 0:  # String values
                level = random.choice(["Root", "Child", "Leaf"])
            elif i % 65 == 0:  # Business logic violation: wrong level
                if parent is None:
                    level = random.choice([2, 3, 4])  # Root with non-zero level
                else:
                    level = 0  # Child with zero level
            elif i % 75 == 0:  # Extreme values
                level = random.choice([-1, 99, 0.5])
            else:
                if parent is None:
                    level = 0
                elif parent in root_categories:
                    level = 1
                elif parent in mid_categories:
                    level = 2
                else:
                    level = random.randint(1, 3)
            record["hierarchy_level"] = level

        # Category path (breadcrumb)
        if random.random() > 0.6:  # 40% have path
            if i % 50 == 0:
                path = None
            elif i % 60 == 0:  # Invalid format
                path = random.choice(["N/A", "path/to/category", ""])
            elif i % 70 == 0:  # Wrong delimiter
                path = random.choice(
                    ["Electronics|Computers|Laptops", "Electronics\\Computers\\Laptops"]
                )
            else:
                if parent is None:
                    path = name if name else "Root"
                else:
                    parent_name = random.choice(["Electronics", "Clothing", "Sports"])
                    path = f"{parent_name} > {name}" if name else parent_name
            record["breadcrumb_path"] = path

        # Product count in category
        if random.random() > 0.5:  # 50% have product count
            if i % 40 == 0:
                count = None
            elif i % 50 == 0:  # String values
                count = random.choice(["Many", "Few", "None"])
            elif i % 60 == 0:  # Negative values (business logic violation)
                count = random.randint(-100, -1)
            elif (
                i % 70 == 0
            ):  # Business logic violation: inactive category with products
                if record.get("active_flag") in [False, "N", "No", "0", "false"]:
                    count = random.randint(10, 100)
                else:
                    count = 0
            elif i % 80 == 0:  # Extreme values
                count = random.choice([999999, 0.5, -9999])
            else:
                count = random.randint(0, 1000)
            record["product_count"] = count

        # SEO URL slug
        if random.random() > 0.6:  # 40% have URL slug
            if i % 45 == 0:
                slug = None
            elif i % 55 == 0:  # Invalid slug format
                slug = random.choice(["category name", "N/A", "slug with spaces"])
            elif i % 65 == 0:  # Special characters in slug
                slug = "category-slug-#1!"
            elif i % 75 == 0:  # Duplicate slugs (should be unique)
                slug = "electronics"
            else:
                if name:
                    slug = name.lower().replace(" ", "-").replace("&", "and")[:50]
                else:
                    slug = f"category-{i}"
            record["url_slug"] = slug

        data.append(record)

    # Create DataFrame
    df = pd.DataFrame(data)

    # Add EXACT duplicate rows (every attribute is same)
    num_exact_duplicates = int(num_rows * 0.02)  # 2% exact duplicates
    for _ in range(num_exact_duplicates):
        if len(df) > 0:
            # Pick a random row to duplicate
            row_to_duplicate = df.sample(1)
            df = pd.concat([df, row_to_duplicate], ignore_index=True)

    # Add some completely empty rows
    for _ in range(int(num_rows * 0.005)):  # 0.5% empty rows
        empty_row = pd.Series([None] * len(df.columns), index=df.columns)
        df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)

    # Add some rows with all string 'NULL' or 'N/A' values
    for _ in range(int(num_rows * 0.01)):  # 1% NULL string rows
        null_values = ["NULL", "N/A", "null", "NA", "", " "]
        null_row = pd.Series(
            [random.choice(null_values) for _ in range(len(df.columns))],
            index=df.columns,
        )
        df = pd.concat([df, pd.DataFrame([null_row])], ignore_index=True)

    # Shuffle the dataframe to mix duplicates throughout
    df = df.sample(frac=1).reset_index(drop=True)

    return df


def add_more_messiness(df):
    """
    Add additional data quality issues to make the dataset more challenging
    """
    # Add trailing/leading spaces to some string columns
    string_cols = df.select_dtypes(include=["object"]).columns
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05  # 5% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    # Add case inconsistencies
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03  # 3% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    # Add special characters to some values
    for col in string_cols[:3]:  # Only first 3 string columns
        mask = np.random.random(len(df)) < 0.02  # 2% of values
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


def detect_circular_references(df):
    """
    Detect circular references in the category hierarchy
    """
    circular_refs = []

    for _, row in df.iterrows():
        cat_id = row["cat_id"]
        parent_id = row["parent_cat_id"]

        if pd.isna(cat_id) or pd.isna(parent_id):
            continue

        # Check for self-reference
        if cat_id == parent_id:
            circular_refs.append((cat_id, "self-reference"))
            continue

        # Check for circular chains (limited depth for performance)
        visited = set()
        current = parent_id
        visited.add(cat_id)

        for _ in range(10):  # Check up to 10 levels
            if current in visited:
                circular_refs.append((cat_id, f"circular with {current}"))
                break

            visited.add(current)
            # Find parent of current
            parent_row = df[df["cat_id"] == current]
            if parent_row.empty:
                break

            current = parent_row.iloc[0]["parent_cat_id"]
            if pd.isna(current):
                break

    return circular_refs


# Generate the dataset
if __name__ == "__main__":
    # Set the number of rows you want
    NUM_ROWS = 200  # Change this to your desired number

    print(f"Generating {NUM_ROWS} rows of messy category data...")
    df = generate_messy_category_data(NUM_ROWS)

    # Add more messiness
    df = add_more_messiness(df)

    # Display basic info
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumn names (realistic but challenging for mapping):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")

    print(f"\nFirst 10 rows:")
    print(df.head(10))

    # Show data quality issues summary
    print("\n" + "=" * 50)
    print("DATA QUALITY ISSUES SUMMARY:")
    print("=" * 50)
    print(f"Total null values: {df.isnull().sum().sum()}")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    print(f"Total rows: {len(df)}")

    # Check NOT NULL constraint violations
    print("\n" + "=" * 50)
    print("NOT NULL CONSTRAINT VIOLATIONS:")
    print("=" * 50)

    if "cat_name" in df.columns:
        null_names = df[df["cat_name"].isnull()].shape[0]
        if null_names > 0:
            print(f"✗ NULL cat_name (violates NOT NULL): {null_names} rows")

    # Show business logic violations
    print("\n" + "=" * 50)
    print("BUSINESS LOGIC VIOLATIONS EXAMPLES:")
    print("=" * 50)

    # Check for self-references
    if "cat_id" in df.columns and "parent_cat_id" in df.columns:
        self_refs = df[df["cat_id"] == df["parent_cat_id"]]
        if len(self_refs) > 0:
            print(f"✗ Self-referencing categories: {len(self_refs)} categories")
            print(f"  Examples: {self_refs['cat_id'].head(3).tolist()}")

    # Check for circular references
    circular_refs = detect_circular_references(df)
    if circular_refs:
        print(f"✗ Circular references detected: {len(circular_refs)} categories")
        print(f"  Examples: {circular_refs[:3]}")

    # Check for orphaned categories (invalid parent references)
    if "parent_cat_id" in df.columns:
        all_cat_ids = set(df["cat_id"].dropna().astype(str))
        parent_refs = set(df["parent_cat_id"].dropna().astype(str))
        orphaned = parent_refs - all_cat_ids - {"INVALID", "N/A", "NULL", ""}
        if orphaned:
            print(f"✗ Orphaned categories (invalid parent): {len(orphaned)} references")
            print(f"  Examples: {list(orphaned)[:3]}")

    # Check for inactive parents with active children
    if "active_flag" in df.columns and "parent_cat_id" in df.columns:
        inactive_parents = df[df["active_flag"].isin([False, "N", "No", "0", "false"])][
            "cat_id"
        ].dropna()
        active_children_of_inactive = df[
            (df["parent_cat_id"].isin(inactive_parents))
            & (df["active_flag"].isin([True, "Y", "Yes", "1", "true"]))
        ]
        if len(active_children_of_inactive) > 0:
            print(
                f"✗ Active categories with inactive parents: {len(active_children_of_inactive)} categories"
            )

    # Check for inactive categories with products
    if "active_flag" in df.columns and "product_count" in df.columns:
        product_count_numeric = pd.to_numeric(df["product_count"], errors="coerce")
        inactive_with_products = df[
            (df["active_flag"].isin([False, "N", "No", "0", "false"]))
            & (product_count_numeric > 0)
        ]
        if len(inactive_with_products) > 0:
            print(
                f"✗ Inactive categories with products: {len(inactive_with_products)} categories"
            )

    # Check for wrong hierarchy levels
    if "hierarchy_level" in df.columns and "parent_cat_id" in df.columns:
        level_numeric = pd.to_numeric(df["hierarchy_level"], errors="coerce")
        root_wrong_level = df[(df["parent_cat_id"].isna()) & (level_numeric != 0)]
        if len(root_wrong_level) > 0:
            print(
                f"✗ Root categories with non-zero level: {len(root_wrong_level)} categories"
            )

        child_zero_level = df[(df["parent_cat_id"].notna()) & (level_numeric == 0)]
        if len(child_zero_level) > 0:
            print(
                f"✗ Child categories with zero level: {len(child_zero_level)} categories"
            )

    # Show extreme values
    print("\n" + "=" * 50)
    print("EXTREME VALUES EXAMPLES:")
    print("=" * 50)

    # Check for extreme product counts
    if "product_count" in df.columns:
        count_numeric = pd.to_numeric(df["product_count"], errors="coerce")
        negative_counts = df[count_numeric < 0]
        if len(negative_counts) > 0:
            print(f"✗ Negative product counts: {len(negative_counts)} categories")

        extreme_counts = df[count_numeric > 100000]
        if len(extreme_counts) > 0:
            print(
                f"✗ Extreme product counts (>100000): {len(extreme_counts)} categories"
            )

    # Check for extreme display orders
    if "display_sequence" in df.columns:
        order_numeric = pd.to_numeric(df["display_sequence"], errors="coerce")
        negative_orders = df[order_numeric < 0]
        if len(negative_orders) > 0:
            print(f"✗ Negative display orders: {len(negative_orders)} categories")

    # Check for duplicate URL slugs
    if "url_slug" in df.columns:
        slug_duplicates = df[
            df.duplicated(subset=["url_slug"], keep=False) & df["url_slug"].notna()
        ]
        if len(slug_duplicates) > 0:
            print(f"✗ Duplicate URL slugs: {len(slug_duplicates)} categories")
            duplicate_slugs = slug_duplicates["url_slug"].value_counts().head(3)
            print(f"  Most duplicated slugs: {duplicate_slugs.to_dict()}")

    # Show hierarchy analysis
    print("\n" + "=" * 50)
    print("HIERARCHY ANALYSIS:")
    print("=" * 50)

    if "parent_cat_id" in df.columns:
        root_categories = df[df["parent_cat_id"].isna()]
        print(f"Root categories: {len(root_categories)}")

        max_depth = 0
        for _, row in root_categories.iterrows():
            if pd.notna(row["cat_id"]):
                # Find all children
                children = df[df["parent_cat_id"] == row["cat_id"]]
                if len(children) > 0:
                    print(f"  {row['cat_name']}: {len(children)} direct children")

    # Show main columns analysis
    print("\n" + "=" * 50)
    print("MAIN COLUMNS ANALYSIS:")
    print("=" * 50)

    main_columns = [
        "cat_id",
        "cat_name",
        "parent_cat_id",
        "active_flag",
        "date_created",
    ]

    for col in main_columns:
        if col in df.columns:
            null_count = df[col].isnull().sum()
            unique_count = df[col].nunique()
            print(f"\n{col}:")
            print(f"  - Null values: {null_count} ({null_count/len(df)*100:.1f}%)")
            print(f"  - Unique values: {unique_count}")
            if df[col].dtype == "object" and unique_count > 0 and unique_count < 10:
                sample_values = df[col].dropna().unique()[:5].tolist()
                print(f"  - Sample values: {sample_values}")

    # Save to CSV
    output_file = "messy_category_data.xlsx"
    df.to_excel(output_file, index=False)
    print(f"\n✅ Dataset saved to '{output_file}'")

    # Save category hierarchy summary
    hierarchy_summary = []
    root_cats = df[df["parent_cat_id"].isna()]
    for _, root in root_cats.iterrows():
        if pd.notna(root["cat_id"]):
            children = df[df["parent_cat_id"] == root["cat_id"]]
            grandchildren = df[df["parent_cat_id"].isin(children["cat_id"].dropna())]
            hierarchy_summary.append(
                {
                    "root_category": root["cat_name"],
                    "root_id": root["cat_id"],
                    "direct_children": len(children),
                    "grandchildren": len(grandchildren),
                    "total_descendants": len(children) + len(grandchildren),
                }
            )

    if hierarchy_summary:
        summary_df = pd.DataFrame(hierarchy_summary)
        summary_df.to_csv("category_hierarchy_summary.csv", index=False)
        print(
            f"✅ Category hierarchy summary saved to 'category_hierarchy_summary.csv'"
        )

Generating 200 rows of messy category data...

Dataset shape: (207, 11)

Column names (realistic but challenging for mapping):
  1. cat_id
  2. cat_name
  3. parent_cat_id
  4. active_flag
  5. date_created
  6. category_desc
  7. product_count
  8. breadcrumb_path
  9. display_sequence
  10. hierarchy_level
  11. url_slug

First 10 rows:
  cat_id            cat_name parent_cat_id active_flag  \
0    295             Spoorts           222        True   
1    259  Premium Collection           103        True   
2    171       Boys Clothing          None        True   
3    164        Best Sellers           111        True   
4    270            Bathroom           258        True   
5   221!          Sale Items          None        True   
6    140    Office Furniture          None        True   
7    280                NULL          None     Pending   
8    230            Furiture           151        True   
9    115   Music Instruments           109        true   

                 dat

### Wishlist Table Generator

In [3]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import string

# Initialize Faker
fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)


def generate_messy_wishlist_data(
    num_rows=2000, customer_id_format="CUST", product_id_format="PROD"
):
    """
    Generate a messy, realistic wishlist dataset with realistic but challenging column names
    and various data quality issues for testing data mapping and cleaning.

    Args:
        num_rows: Number of wishlist records to generate
        customer_id_format: Format of customer IDs ('CUST', 'CUSTOMER', or 'NUMBER')
        product_id_format: Format of product IDs ('PROD', 'PRODUCT', or 'NUMBER')
    """

    data = []

    # Track IDs for creating duplicates and relationships
    used_wishlist_ids = []

    # Generate pools of customer and product IDs
    num_customers = max(num_rows // 10, 50)  # Assuming ~10 wishlist items per customer
    num_products = max(num_rows // 5, 100)  # Pool of products

    customer_ids = []
    for i in range(num_customers):
        if customer_id_format == "CUST":
            cust_id = f"CUST_{str(i + 1000).zfill(5)}"
        elif customer_id_format == "CUSTOMER":
            cust_id = f"CUSTOMER-{str(i + 1000).zfill(5)}"
        else:  # NUMBER
            cust_id = str(10000 + i)
        customer_ids.append(cust_id)

    product_ids = []
    for i in range(num_products):
        if product_id_format == "PROD":
            prod_id = f"PROD_{str(i + 1).zfill(4)}"
        elif product_id_format == "PRODUCT":
            prod_id = f"PRODUCT-{str(i + 1).zfill(4)}"
        else:  # NUMBER
            prod_id = str(10000 + i)
        product_ids.append(prod_id)

    # Choose one consistent wishlist ID format
    wish_id_format_choice = random.choice(["WISH", "WISHLIST", "NUMBER"])

    # Track customer-product combinations for duplicate detection
    customer_product_pairs = {}

    for i in range(num_rows):
        record = {}

        # Wishlist ID with CONSISTENT format
        if i % 50 == 0 and used_wishlist_ids:  # 2% duplicates
            wish_id = random.choice(used_wishlist_ids)
        else:
            if wish_id_format_choice == "WISH":
                wish_id = f"WISH_{str(i + 1).zfill(6)}"
            elif wish_id_format_choice == "WISHLIST":
                wish_id = f"WISHLIST-{str(i + 1).zfill(6)}"
            else:  # NUMBER
                wish_id = str(100000 + i)

            used_wishlist_ids.append(wish_id)

        record["wish_id"] = wish_id if i % 100 != 0 else None  # 1% null IDs

        # Customer ID - NOT NULL in schema
        if i % 80 == 0:  # Violates NOT NULL
            cust_id = None
        elif i % 60 == 0:  # Non-existent customer IDs (FK violation)
            if customer_id_format == "CUST":
                cust_id = f"CUST_{str(99999).zfill(5)}"
            elif customer_id_format == "CUSTOMER":
                cust_id = f"CUSTOMER-{str(99999).zfill(5)}"
            else:
                cust_id = "99999"
        elif i % 40 == 0:  # Invalid format
            cust_id = random.choice(["INVALID", "NULL", "N/A", "", "UNKNOWN"])
        else:
            cust_id = random.choice(customer_ids)

        record["user_id"] = cust_id

        # Product ID - NOT NULL in schema
        if i % 75 == 0:  # Violates NOT NULL
            prod_id = None
        elif i % 55 == 0:  # Non-existent product IDs (FK violation)
            if product_id_format == "PROD":
                prod_id = f"PROD_{str(9999).zfill(4)}"
            elif product_id_format == "PRODUCT":
                prod_id = f"PRODUCT-{str(9999).zfill(4)}"
            else:
                prod_id = "99999"
        elif i % 45 == 0:  # Invalid format
            prod_id = random.choice(["INVALID", "N/A", "NULL", "", "DISCONTINUED"])
        else:
            # Check for duplicate customer-product pairs (business logic issue)
            if i % 30 == 0 and cust_id in customer_product_pairs:
                # Intentionally create duplicate wishlist item for same customer
                if customer_product_pairs[cust_id]:
                    prod_id = random.choice(customer_product_pairs[cust_id])
                else:
                    prod_id = random.choice(product_ids)
            else:
                prod_id = random.choice(product_ids)

        # Track customer-product pairs
        if cust_id and prod_id:
            if cust_id not in customer_product_pairs:
                customer_product_pairs[cust_id] = []
            customer_product_pairs[cust_id].append(prod_id)

        record["item_id"] = prod_id

        # Added Date - should have DEFAULT CURRENT_TIMESTAMP
        if i % 25 == 0:
            added = None
        elif i % 35 == 0:  # String format variations
            added_date = fake.date_time_between(start_date="-2y", end_date="now")
            formats = ["%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%d-%m-%Y", "%Y%m%d"]
            added = added_date.strftime(random.choice(formats))
        elif i % 45 == 0:  # Unix timestamp
            added = int(
                fake.date_time_between(start_date="-2y", end_date="now").timestamp()
            )
        elif i % 55 == 0:  # Future date (business logic violation)
            added = fake.date_time_between(start_date="+1m", end_date="+1y")
        elif i % 65 == 0:  # Very old date (unusual for wishlist)
            added = fake.date_time_between(start_date="-10y", end_date="-5y")
        else:
            added = fake.date_time_between(start_date="-2y", end_date="now")

        record["date_added"] = added

        # Priority Level
        if i % 20 == 0:
            priority = None
        elif i % 30 == 0:  # Inconsistent values
            priority = random.choice(
                [
                    "high",
                    "HIGH",
                    "H",
                    "1",
                    "medium",
                    "MEDIUM",
                    "M",
                    "2",
                    "low",
                    "LOW",
                    "L",
                    "3",
                ]
            )
        elif i % 40 == 0:  # Invalid values
            priority = random.choice(
                ["Urgent", "Important", "Normal", "Critical", "ASAP", ""]
            )
        elif i % 50 == 0:  # Numeric values
            priority = random.choice([1, 2, 3, 4, 5])
        elif i % 60 == 0:  # Typos
            priority = random.choice(["Hihg", "Mediun", "Loww"])
        else:
            priority = random.choice(["High", "Medium", "Low"])

        record["priority"] = priority

        # Purchased Date (should be after added_date)
        purchased = None
        removed = None

        if random.random() < 0.15:  # 15% are purchased
            if i % 40 == 0:  # String format
                if isinstance(added, datetime):
                    purchased = (
                        added + timedelta(days=random.randint(1, 90))
                    ).strftime("%Y-%m-%d")
                else:
                    purchased = fake.date_time_between(
                        start_date="-1y", end_date="now"
                    ).strftime("%Y-%m-%d")
            elif i % 50 == 0:  # Business logic violation: purchased before added
                if isinstance(added, datetime):
                    purchased = added - timedelta(days=random.randint(1, 30))
                else:
                    purchased = fake.date_time_between(start_date="-3y", end_date="-2y")
            elif i % 60 == 0:  # Future purchase date
                purchased = fake.date_time_between(start_date="+1m", end_date="+6m")
            elif i % 70 == 0:  # Invalid date format
                purchased = random.choice(["0000-00-00", "N/A", "PURCHASED", ""])
            else:
                if isinstance(added, datetime):
                    purchased = fake.date_time_between(start_date=added, end_date="now")
                else:
                    purchased = fake.date_time_between(start_date="-1y", end_date="now")

        record["purchase_date"] = purchased

        # Removed Date (should be after added_date)
        if (
            random.random() < 0.10 and not purchased
        ):  # 10% are removed (but not if purchased)
            if i % 45 == 0:  # String format
                if isinstance(added, datetime):
                    removed = (added + timedelta(days=random.randint(1, 60))).strftime(
                        "%Y-%m-%d"
                    )
                else:
                    removed = fake.date_time_between(
                        start_date="-1y", end_date="now"
                    ).strftime("%Y-%m-%d")
            elif i % 55 == 0:  # Business logic violation: removed before added
                if isinstance(added, datetime):
                    removed = added - timedelta(days=random.randint(1, 30))
                else:
                    removed = fake.date_time_between(start_date="-3y", end_date="-2y")
            elif i % 65 == 0:  # Future removal date
                removed = fake.date_time_between(start_date="+1m", end_date="+6m")
            else:
                if isinstance(added, datetime):
                    removed = fake.date_time_between(start_date=added, end_date="now")
                else:
                    removed = fake.date_time_between(start_date="-1y", end_date="now")
        elif i % 80 == 0:  # Business logic violation: both purchased and removed
            removed = fake.date_time_between(start_date="-6m", end_date="now")
            purchased = fake.date_time_between(start_date="-1y", end_date="-6m")
            record["purchase_date"] = purchased

        record["removal_date"] = removed

        # Additional realistic columns that might exist

        # Notification sent flag
        if random.random() > 0.4:  # 60% have this
            if i % 35 == 0:
                notified = None
            elif i % 45 == 0:  # Various boolean representations
                notified = random.choice(
                    ["Y", "N", "Yes", "No", "1", "0", "true", "false"]
                )
            elif i % 55 == 0:  # Invalid values
                notified = random.choice(["Pending", "Sent", "Failed"])
            else:
                notified = random.choice([True, False])
            record["notification_sent"] = notified

        # Price when added (to track price changes)
        if random.random() > 0.5:  # 50% have this
            if i % 30 == 0:
                price = None
            elif i % 40 == 0:  # String values
                price = random.choice(["N/A", "Unknown", "Free"])
            elif i % 50 == 0:  # Negative values
                price = round(random.uniform(-100, -1), 2)
            elif i % 60 == 0:  # Extreme values
                price = random.choice([0, 999999.99, 0.001])
            else:
                price = round(random.uniform(10, 500), 2)
            record["price_at_addition"] = price

        # Current price (for price drop alerts)
        if random.random() > 0.5:  # 50% have this
            if i % 35 == 0:
                current = None
            elif i % 45 == 0:  # String values
                current = random.choice(["Out of Stock", "Discontinued", "TBD"])
            elif (
                i % 55 == 0 and "price_at_addition" in record
            ):  # Business logic: current price before historical
                if isinstance(record["price_at_addition"], (int, float)):
                    current = record["price_at_addition"] * 0.5
                else:
                    current = round(random.uniform(5, 50), 2)
            else:
                current = round(random.uniform(10, 500), 2)
            record["current_price"] = current

        # Notes/comments
        if random.random() > 0.6:  # 40% have notes
            if i % 40 == 0:
                notes = None
            elif i % 50 == 0:
                notes = random.choice(["", "N/A", "None"])
            elif i % 60 == 0:  # Very long notes
                notes = fake.text(max_nb_chars=500)
            elif i % 70 == 0:  # Special characters
                notes = f"Gift for birthday! 🎁 #{i}"
            else:
                notes = random.choice(
                    [
                        "Birthday gift idea",
                        "Wait for sale",
                        "Check reviews first",
                        "Alternative to consider",
                        "Must have!",
                        "Compare with other options",
                        fake.sentence(nb_words=8),
                    ]
                )
            record["user_notes"] = notes

        # Quantity desired
        if random.random() > 0.7:  # 30% have quantity
            if i % 45 == 0:
                qty = None
            elif i % 55 == 0:  # String values
                qty = random.choice(["Multiple", "A few", "N/A"])
            elif i % 65 == 0:  # Negative or zero
                qty = random.choice([-1, 0, -10])
            elif i % 75 == 0:  # Extreme values
                qty = random.choice([999, 0.5, 10000])
            else:
                qty = random.randint(1, 5)
            record["desired_quantity"] = qty

        # Alert enabled (for price drops)
        if random.random() > 0.5:  # 50% have this
            if i % 40 == 0:
                alert = None
            elif i % 50 == 0:  # Various boolean representations
                alert = random.choice(["Y", "N", "Active", "Inactive"])
            else:
                alert = random.choice([True, False])
            record["price_alert_enabled"] = alert

        # Source/channel (how item was added)
        if random.random() > 0.6:  # 40% have this
            if i % 35 == 0:
                source = None
            elif i % 45 == 0:
                source = random.choice(["", "N/A", "Unknown"])
            else:
                source = random.choice(
                    [
                        "Web",
                        "Mobile App",
                        "iOS",
                        "Android",
                        "Browser Extension",
                        "Email Link",
                        "Social Media",
                        "QR Code",
                        "API",
                        "Import",
                    ]
                )
            record["add_source"] = source

        # List name (if user has multiple lists)
        if random.random() > 0.7:  # 30% have custom list names
            if i % 40 == 0:
                list_name = None
            elif i % 50 == 0:
                list_name = random.choice(["", "Default", "Main"])
            else:
                list_name = random.choice(
                    [
                        "Birthday Wishes",
                        "Christmas List",
                        "Dream Items",
                        "Future Purchases",
                        "Gift Ideas",
                        "Save for Later",
                        "Wedding Registry",
                        "Baby Registry",
                        "Home Improvement",
                    ]
                )
            record["list_name"] = list_name

        data.append(record)

    # Create DataFrame
    df = pd.DataFrame(data)

    # Add EXACT duplicate rows (every attribute is same)
    num_exact_duplicates = int(num_rows * 0.02)  # 2% exact duplicates
    for _ in range(num_exact_duplicates):
        if len(df) > 0:
            # Pick a random row to duplicate
            row_to_duplicate = df.sample(1)
            df = pd.concat([df, row_to_duplicate], ignore_index=True)

    # Add some completely empty rows
    for _ in range(int(num_rows * 0.005)):  # 0.5% empty rows
        empty_row = pd.Series([None] * len(df.columns), index=df.columns)
        df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)

    # Add some rows with all string 'NULL' or 'N/A' values
    for _ in range(int(num_rows * 0.01)):  # 1% NULL string rows
        null_values = ["NULL", "N/A", "null", "NA", "", " "]
        null_row = pd.Series(
            [random.choice(null_values) for _ in range(len(df.columns))],
            index=df.columns,
        )
        df = pd.concat([df, pd.DataFrame([null_row])], ignore_index=True)

    # Shuffle the dataframe to mix duplicates throughout
    df = df.sample(frac=1).reset_index(drop=True)

    return df


def add_more_messiness(df):
    """
    Add additional data quality issues to make the dataset more challenging
    """
    # Add trailing/leading spaces to some string columns
    string_cols = df.select_dtypes(include=["object"]).columns
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05  # 5% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    # Add case inconsistencies
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03  # 3% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    # Add special characters to some values
    for col in string_cols[:3]:  # Only first 3 string columns
        mask = np.random.random(len(df)) < 0.02  # 2% of values
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


# Generate the dataset
if __name__ == "__main__":
    # Set the number of rows you want
    NUM_ROWS = 2000  # Change this to your desired number

    # ID formats should match your customer and product datasets
    CUSTOMER_ID_FORMAT = "CUST"  # Options: 'CUST', 'CUSTOMER', or 'NUMBER'
    PRODUCT_ID_FORMAT = "PROD"  # Options: 'PROD', 'PRODUCT', or 'NUMBER'

    print(f"Generating {NUM_ROWS} rows of messy wishlist data...")
    df = generate_messy_wishlist_data(NUM_ROWS, CUSTOMER_ID_FORMAT, PRODUCT_ID_FORMAT)

    # Add more messiness
    df = add_more_messiness(df)

    # Display basic info
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumn names (realistic but challenging for mapping):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")

    print(f"\nFirst 10 rows:")
    print(df.head(10))

    # Show data quality issues summary
    print("\n" + "=" * 50)
    print("DATA QUALITY ISSUES SUMMARY:")
    print("=" * 50)
    print(f"Total null values: {df.isnull().sum().sum()}")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    print(f"Total rows: {len(df)}")

    # Check NOT NULL constraint violations
    print("\n" + "=" * 50)
    print("NOT NULL CONSTRAINT VIOLATIONS:")
    print("=" * 50)

    if "user_id" in df.columns:
        null_customers = df[df["user_id"].isnull()].shape[0]
        if null_customers > 0:
            print(
                f"✗ NULL user_id/customer_id (violates NOT NULL): {null_customers} rows"
            )

    if "item_id" in df.columns:
        null_products = df[df["item_id"].isnull()].shape[0]
        if null_products > 0:
            print(
                f"✗ NULL item_id/product_id (violates NOT NULL): {null_products} rows"
            )

    # Show business logic violations
    print("\n" + "=" * 50)
    print("BUSINESS LOGIC VIOLATIONS EXAMPLES:")
    print("=" * 50)

    # Check for duplicate wishlist items (same customer-product pair)
    if "user_id" in df.columns and "item_id" in df.columns:
        duplicate_items = df[df.duplicated(subset=["user_id", "item_id"], keep=False)]
        duplicate_items = duplicate_items[
            duplicate_items["user_id"].notna() & duplicate_items["item_id"].notna()
        ]
        if len(duplicate_items) > 0:
            print(
                f"✗ Duplicate wishlist items (same customer-product): {len(duplicate_items)} rows"
            )
            sample_dupes = (
                duplicate_items.groupby(["user_id", "item_id"]).size().head(3)
            )
            print(f"  Examples (customer, product): {sample_dupes.to_dict()}")

    # Check for purchased date before added date
    if "date_added" in df.columns and "purchase_date" in df.columns:
        added_dt = pd.to_datetime(df["date_added"], errors="coerce")
        purchased_dt = pd.to_datetime(df["purchase_date"], errors="coerce")
        purchased_before_added = df[purchased_dt < added_dt]
        if len(purchased_before_added) > 0:
            print(
                f"✗ Items purchased before being added: {len(purchased_before_added)} rows"
            )

    # Check for removed date before added date
    if "date_added" in df.columns and "removal_date" in df.columns:
        added_dt = pd.to_datetime(df["date_added"], errors="coerce")
        removed_dt = pd.to_datetime(df["removal_date"], errors="coerce")
        removed_before_added = df[removed_dt < added_dt]
        if len(removed_before_added) > 0:
            print(
                f"✗ Items removed before being added: {len(removed_before_added)} rows"
            )

    # Check for items both purchased and removed
    if "purchase_date" in df.columns and "removal_date" in df.columns:
        both_purchased_removed = df[
            df["purchase_date"].notna() & df["removal_date"].notna()
        ]
        if len(both_purchased_removed) > 0:
            print(
                f"✗ Items both purchased AND removed: {len(both_purchased_removed)} rows"
            )

    # Check for future dates
    if "date_added" in df.columns:
        added_dt = pd.to_datetime(df["date_added"], errors="coerce")
        future_added = df[added_dt > datetime.now()]
        if len(future_added) > 0:
            print(f"✗ Items added in the future: {len(future_added)} rows")

    # Check for invalid foreign keys
    if "user_id" in df.columns:
        invalid_customers = df[
            df["user_id"].isin(["INVALID", "NULL", "N/A", "", "UNKNOWN"])
        ]
        if len(invalid_customers) > 0:
            print(f"✗ Invalid customer references: {len(invalid_customers)} rows")

    if "item_id" in df.columns:
        invalid_products = df[
            df["item_id"].isin(["INVALID", "N/A", "NULL", "", "DISCONTINUED"])
        ]
        if len(invalid_products) > 0:
            print(f"✗ Invalid product references: {len(invalid_products)} rows")

    # Check for price anomalies
    if "price_at_addition" in df.columns and "current_price" in df.columns:
        price_added = pd.to_numeric(df["price_at_addition"], errors="coerce")
        price_current = pd.to_numeric(df["current_price"], errors="coerce")
        # Current price significantly lower than historical (possible data issue)
        price_drop_extreme = df[price_current < (price_added * 0.1)]
        if len(price_drop_extreme) > 0:
            print(f"✗ Extreme price drops (>90%): {len(price_drop_extreme)} items")

    # Show extreme values
    print("\n" + "=" * 50)
    print("EXTREME VALUES EXAMPLES:")
    print("=" * 50)

    # Check for negative prices
    if "price_at_addition" in df.columns:
        price_numeric = pd.to_numeric(df["price_at_addition"], errors="coerce")
        negative_prices = df[price_numeric < 0]
        if len(negative_prices) > 0:
            print(f"✗ Negative prices: {len(negative_prices)} items")
            print(
                f"  Examples: {negative_prices['price_at_addition'].unique()[:5].tolist()}"
            )

    # Check for invalid quantities
    if "desired_quantity" in df.columns:
        qty_numeric = pd.to_numeric(df["desired_quantity"], errors="coerce")
        invalid_qty = df[(qty_numeric <= 0) | (qty_numeric > 100)]
        if len(invalid_qty) > 0:
            print(f"✗ Invalid quantities (<=0 or >100): {len(invalid_qty)} items")
            print(
                f"  Examples: {invalid_qty['desired_quantity'].unique()[:5].tolist()}"
            )

    # Check for very old wishlist items
    if "date_added" in df.columns:
        added_dt = pd.to_datetime(df["date_added"], errors="coerce")
        very_old = df[added_dt < (datetime.now() - timedelta(days=5 * 365))]
        if len(very_old) > 0:
            print(f"✗ Items added >5 years ago: {len(very_old)} items")

    # Show customer statistics
    print("\n" + "=" * 50)
    print("CUSTOMER WISHLIST STATISTICS:")
    print("=" * 50)

    if "user_id" in df.columns:
        customer_stats = (
            df.groupby("user_id")
            .agg(
                {
                    "wish_id": "count",
                    "purchase_date": lambda x: (
                        x.notna().sum() if "purchase_date" in df.columns else 0
                    ),
                    "removal_date": lambda x: (
                        x.notna().sum() if "removal_date" in df.columns else 0
                    ),
                }
            )
            .rename(
                columns={
                    "wish_id": "total_items",
                    "purchase_date": "purchased_items",
                    "removal_date": "removed_items",
                }
            )
        )

        # Find customers with most items
        top_customers = customer_stats.nlargest(5, "total_items")
        print(f"Top 5 customers by wishlist size:")
        for idx, row in top_customers.iterrows():
            if idx not in ["INVALID", "NULL", "N/A", "", "UNKNOWN"]:
                print(
                    f"  {idx}: {row['total_items']} items ({row['purchased_items']} purchased, {row['removed_items']} removed)"
                )

    # Show main columns analysis
    print("\n" + "=" * 50)
    print("MAIN COLUMNS ANALYSIS:")
    print("=" * 50)

    main_columns = [
        "wish_id",
        "user_id",
        "item_id",
        "date_added",
        "priority",
        "purchase_date",
        "removal_date",
    ]

    for col in main_columns:
        if col in df.columns:
            null_count = df[col].isnull().sum()
            unique_count = df[col].nunique()
            print(f"\n{col}:")
            print(f"  - Null values: {null_count} ({null_count/len(df)*100:.1f}%)")
            print(f"  - Unique values: {unique_count}")
            if col == "priority" and unique_count > 0:
                priority_dist = df[col].value_counts().head(5)
                print(f"  - Top values: {priority_dist.to_dict()}")

    # Save to CSV
    output_file = "messy_wishlist_data.xlsx"
    df.to_excel(output_file, index=False)
    print(f"\n✅ Dataset saved to '{output_file}'")

    # Save wishlist summary for analysis
    wishlist_summary = []

    # Purchase conversion rate by priority
    if "priority" in df.columns and "purchase_date" in df.columns:
        for priority in df["priority"].dropna().unique()[:10]:
            priority_items = df[df["priority"] == priority]
            purchased = priority_items["purchase_date"].notna().sum()
            total = len(priority_items)
            wishlist_summary.append(
                {
                    "priority": priority,
                    "total_items": total,
                    "purchased_items": purchased,
                    "conversion_rate": (
                        round(purchased / total * 100, 2) if total > 0 else 0
                    ),
                }
            )

    if wishlist_summary:
        summary_df = pd.DataFrame(wishlist_summary)
        summary_df.to_csv("wishlist_priority_summary.csv", index=False)
        print(f"✅ Wishlist priority summary saved to 'wishlist_priority_summary.csv'")

Generating 2000 rows of messy wishlist data...

Dataset shape: (2070, 15)

Column names (realistic but challenging for mapping):
  1. wish_id
  2. user_id
  3. item_id
  4. date_added
  5. priority
  6. purchase_date
  7. removal_date
  8. user_notes
  9. price_alert_enabled
  10. notification_sent
  11. current_price
  12. desired_quantity
  13. list_name
  14. price_at_addition
  15. add_source

First 10 rows:
  wish_id     user_id        item_id                      date_added priority  \
0  100219  CUST_01117      PROD_0074      2024-12-02 22:44:45.998740   Medium   
1  100172  CUST_01192      PROD_0064      2024-02-21 06:06:33.267564      Low   
2  101366  CUST_01087    PROD_0052        2025-03-26 08:09:59.375121      Low   
3  101682  CUST_01022      PROD_0152      2025-08-04 08:16:50.728390      Low   
4  101393  CUST_01154      PROD_0269      2025-01-09 04:32:29.069223   Medium   
5  101584  CUST_01168      PROD_0272      2023-11-22 11:46:32.562486      Low   
6  101782  CUST_0

### Shopping Cart Table Generator

In [5]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import string
import uuid

# Initialize Faker
fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)


def generate_messy_shopping_cart_data(
    num_rows=3000, customer_id_format="CUST", product_id_format="PROD"
):
    """
    Generate a messy, realistic shopping cart dataset with realistic but challenging column names
    and various data quality issues for testing data mapping and cleaning.

    Args:
        num_rows: Number of shopping cart records to generate
        customer_id_format: Format of customer IDs ('CUST', 'CUSTOMER', or 'NUMBER')
        product_id_format: Format of product IDs ('PROD', 'PRODUCT', or 'NUMBER')
    """

    data = []

    # Track IDs for creating duplicates and relationships
    used_cart_ids = []
    used_session_ids = []
    used_order_ids = []

    # Generate pools of customer and product IDs
    num_customers = max(
        num_rows // 15, 50
    )  # Assuming ~15 cart items across different sessions
    num_products = max(num_rows // 10, 100)  # Pool of products

    customer_ids = []
    for i in range(num_customers):
        if customer_id_format == "CUST":
            cust_id = f"CUST_{str(i + 1000).zfill(5)}"
        elif customer_id_format == "CUSTOMER":
            cust_id = f"CUSTOMER-{str(i + 1000).zfill(5)}"
        else:  # NUMBER
            cust_id = str(10000 + i)
        customer_ids.append(cust_id)

    product_ids = []
    for i in range(num_products):
        if product_id_format == "PROD":
            prod_id = f"PROD_{str(i + 1).zfill(4)}"
        elif product_id_format == "PRODUCT":
            prod_id = f"PRODUCT-{str(i + 1).zfill(4)}"
        else:  # NUMBER
            prod_id = str(10000 + i)
        product_ids.append(prod_id)

    # Choose one consistent cart ID format
    cart_id_format_choice = random.choice(["CART", "BASKET", "NUMBER"])
    order_id_format_choice = random.choice(["ORD", "ORDER", "NUMBER"])

    # Track cart sessions for realistic grouping
    cart_sessions = {}

    for i in range(num_rows):
        record = {}

        # Cart ID with CONSISTENT format
        if i % 50 == 0 and used_cart_ids:  # 2% duplicates
            cart_id = random.choice(used_cart_ids)
        else:
            if cart_id_format_choice == "CART":
                cart_id = f"CART_{str(i + 1).zfill(7)}"
            elif cart_id_format_choice == "BASKET":
                cart_id = f"BASKET-{str(i + 1).zfill(7)}"
            else:  # NUMBER
                cart_id = str(1000000 + i)

            used_cart_ids.append(cart_id)

        record["cart_item_id"] = cart_id if i % 100 != 0 else None  # 1% null IDs

        # Decide if this is a guest or registered user cart
        is_guest = random.random() < 0.35  # 35% guest carts

        # Customer ID (NULL for guest carts)
        if is_guest:
            cust_id = None
        elif i % 60 == 0:  # Non-existent customer IDs (FK violation)
            if customer_id_format == "CUST":
                cust_id = f"CUST_{str(99999).zfill(5)}"
            elif customer_id_format == "CUSTOMER":
                cust_id = f"CUSTOMER-{str(99999).zfill(5)}"
            else:
                cust_id = "99999"
        elif i % 40 == 0:  # Invalid format
            cust_id = random.choice(["INVALID", "NULL", "N/A", "", "GUEST"])
        elif (
            i % 80 == 0
        ):  # Business logic violation: both customer_id and session_id null
            cust_id = None
            is_guest = False  # Will make session_id also null
        else:
            cust_id = random.choice(customer_ids)

        record["customer_ref"] = cust_id

        # Session ID (for guest carts or all carts)
        if is_guest and cust_id is None:
            # Guest cart must have session ID
            if i % 70 == 0:  # Invalid session format
                session_id = random.choice(["INVALID", "NULL", "", "000000"])
            elif (
                i % 50 == 0 and used_session_ids
            ):  # Reuse session (multiple items in cart)
                session_id = random.choice(used_session_ids[-10:])  # Recent sessions
            else:
                session_id = fake.uuid4()
                used_session_ids.append(session_id)
        elif not is_guest and cust_id:
            # Registered user might still have session ID
            if random.random() < 0.7:  # 70% have session ID
                if i % 60 == 0:  # Invalid format
                    session_id = random.choice(["USER_SESSION", "N/A"])
                else:
                    session_id = fake.uuid4()
                    used_session_ids.append(session_id)
            else:
                session_id = None
        elif i % 80 == 0:  # Business logic violation case
            session_id = None  # Both customer_id and session_id are null
        else:
            session_id = None

        record["session_identifier"] = session_id

        # Product ID - NOT NULL in schema
        if i % 75 == 0:  # Violates NOT NULL
            prod_id = None
        elif i % 55 == 0:  # Non-existent product IDs (FK violation)
            if product_id_format == "PROD":
                prod_id = f"PROD_{str(9999).zfill(4)}"
            elif product_id_format == "PRODUCT":
                prod_id = f"PRODUCT-{str(9999).zfill(4)}"
            else:
                prod_id = "99999"
        elif i % 45 == 0:  # Invalid format
            prod_id = random.choice(["INVALID", "N/A", "NULL", "", "OUT_OF_STOCK"])
        else:
            # Group products by session (realistic cart behavior)
            if session_id and session_id in cart_sessions:
                # 30% chance to add same product (quantity update scenario)
                if random.random() < 0.3 and cart_sessions[session_id]:
                    prod_id = random.choice(cart_sessions[session_id])
                else:
                    prod_id = random.choice(product_ids)
                    cart_sessions[session_id].append(prod_id)
            else:
                prod_id = random.choice(product_ids)
                if session_id:
                    cart_sessions[session_id] = [prod_id]

        record["product_ref"] = prod_id

        # Quantity - NOT NULL in schema
        if i % 80 == 0:  # Violates NOT NULL
            quantity = None
        elif i % 30 == 0:  # String values
            quantity = random.choice(["One", "Two", "Many", "N/A"])
        elif i % 40 == 0:  # Zero or negative (business logic violation)
            quantity = random.choice([0, -1, -10])
        elif i % 50 == 0:  # Extreme values
            quantity = random.choice([999, 10000, 0.5, -999])
        elif i % 60 == 0:  # Decimal quantities for non-decimal products
            quantity = random.choice([1.5, 2.3, 3.7])
        else:
            # Realistic quantities
            quantity = random.choices(
                [1, 2, 3, 4, 5, 10, 20],
                weights=[40, 25, 15, 10, 5, 3, 2],  # Most carts have 1-2 items
                k=1,
            )[0]

        record["item_quantity"] = quantity

        # Unit Price - NOT NULL in schema
        if i % 85 == 0:  # Violates NOT NULL
            price = None
        elif i % 35 == 0:  # String values
            price = random.choice(["Free", "N/A", "Contact for price", ""])
        elif i % 45 == 0:  # Negative prices (business logic violation)
            price = round(random.uniform(-100, -1), 2)
        elif i % 55 == 0:  # Zero price
            price = 0
        elif i % 65 == 0:  # Extreme prices
            price = random.choice([999999.99, 0.001, -9999])
        elif i % 75 == 0:  # Price with too many decimals
            price = round(random.uniform(10, 500), 5)
        else:
            price = round(random.uniform(5, 500), 2)

        record["price_per_unit"] = price

        # Added Date - should have DEFAULT CURRENT_TIMESTAMP
        if i % 25 == 0:
            added = None
        elif i % 35 == 0:  # String format variations
            added_date = fake.date_time_between(start_date="-30d", end_date="now")
            formats = ["%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%d-%m-%Y", "%Y%m%d"]
            added = added_date.strftime(random.choice(formats))
        elif i % 45 == 0:  # Unix timestamp
            added = int(
                fake.date_time_between(start_date="-30d", end_date="now").timestamp()
            )
        elif i % 55 == 0:  # Future date (business logic violation)
            added = fake.date_time_between(start_date="+1d", end_date="+7d")
        elif i % 65 == 0:  # Very old cart (should be abandoned)
            added = fake.date_time_between(start_date="-1y", end_date="-6m")
        else:
            # Recent carts (within 30 days)
            added = fake.date_time_between(start_date="-30d", end_date="now")

        record["date_added_to_cart"] = added

        # Cart Status
        if i % 20 == 0:
            status = None
        elif i % 30 == 0:  # Inconsistent values
            status = random.choice(
                ["active", "ACTIVE", "A", "1", "abandoned", "ABANDONED"]
            )
        elif i % 40 == 0:  # Invalid values
            status = random.choice(["Pending", "In Progress", "Expired", "Deleted"])
        elif i % 50 == 0:  # Typos
            status = random.choice(["Activ", "Abandond", "Convertd"])
        else:
            # Realistic status distribution
            # 70% abandoned, 20% converted, 10% still active
            if isinstance(added, datetime):
                days_old = (datetime.now() - added).days
                if days_old > 7:
                    status = random.choices(
                        ["Abandoned", "Converted"], weights=[70, 30], k=1
                    )[0]
                else:
                    status = random.choices(
                        ["Active", "Abandoned", "Converted"], weights=[30, 50, 20], k=1
                    )[0]
            else:
                status = random.choice(["Active", "Abandoned", "Converted"])

        record["status"] = status

        # Converted Order ID (only for converted carts)
        if status in ["Converted", "converted", "CONVERTED", "Convertd"]:
            if i % 60 == 0:  # Business logic violation: converted but no order ID
                order_id = None
            elif i % 70 == 0:  # Invalid order ID format
                order_id = random.choice(["INVALID", "N/A", "PENDING"])
            elif (
                i % 80 == 0 and used_order_ids
            ):  # Duplicate order ID (multiple carts same order)
                order_id = random.choice(used_order_ids)
            else:
                if order_id_format_choice == "ORD":
                    order_id = f"ORD_{str(len(used_order_ids) + 1000).zfill(6)}"
                elif order_id_format_choice == "ORDER":
                    order_id = f"ORDER-{str(len(used_order_ids) + 1000).zfill(6)}"
                else:
                    order_id = str(100000 + len(used_order_ids))
                used_order_ids.append(order_id)
        elif i % 90 == 0:  # Business logic violation: not converted but has order ID
            if order_id_format_choice == "ORD":
                order_id = f"ORD_{str(99999).zfill(6)}"
            else:
                order_id = "INVALID_ORDER"
        else:
            order_id = None

        record["order_reference"] = order_id

        # Additional realistic columns that might exist

        # Discount amount
        if random.random() > 0.6:  # 40% have discounts
            if i % 35 == 0:
                discount = None
            elif i % 45 == 0:  # String values
                discount = random.choice(["10%", "SALE", "N/A"])
            elif i % 55 == 0:  # Negative discount (surcharge?)
                discount = round(random.uniform(-50, -1), 2)
            elif i % 65 == 0:  # Discount greater than price (business logic violation)
                if isinstance(price, (int, float)) and price > 0:
                    discount = price * 1.5
                else:
                    discount = 1000
            else:
                if isinstance(price, (int, float)) and price > 0:
                    discount = round(
                        price * random.uniform(0, 0.3), 2
                    )  # 0-30% discount
                else:
                    discount = 0
            record["discount_amount"] = discount

        # Tax amount
        if random.random() > 0.5:  # 50% have tax calculated
            if i % 40 == 0:
                tax = None
            elif i % 50 == 0:  # String values
                tax = random.choice(["Included", "Exempt", "N/A"])
            elif i % 60 == 0:  # Negative tax
                tax = round(random.uniform(-10, -1), 2)
            elif i % 70 == 0:  # Extreme tax
                tax = random.choice([999, 0.001])
            else:
                if (
                    isinstance(price, (int, float))
                    and price > 0
                    and isinstance(quantity, (int, float))
                ):
                    tax = round(
                        price * quantity * random.uniform(0.05, 0.15), 2
                    )  # 5-15% tax
                else:
                    tax = 0
            record["tax_amount"] = tax

        # Updated date (for quantity changes)
        if random.random() > 0.7:  # 30% have been updated
            if i % 45 == 0:
                updated = None
            elif i % 55 == 0:  # Updated before added (business logic violation)
                if isinstance(added, datetime):
                    updated = added - timedelta(hours=random.randint(1, 24))
                else:
                    updated = fake.date_time_between(start_date="-35d", end_date="-31d")
            else:
                if isinstance(added, datetime):
                    updated = fake.date_time_between(start_date=added, end_date="now")
                else:
                    updated = fake.date_time_between(start_date="-29d", end_date="now")
            record["last_updated"] = updated

        # Device type
        if random.random() > 0.4:  # 60% have device info
            if i % 50 == 0:
                device = None
            elif i % 60 == 0:
                device = random.choice(["", "N/A", "Unknown"])
            else:
                device = random.choice(
                    [
                        "Desktop",
                        "Mobile",
                        "Tablet",
                        "iOS",
                        "Android",
                        "Windows",
                        "Mac",
                        "Chrome",
                        "Safari",
                        "App",
                    ]
                )
            record["device_type"] = device

        # Coupon code
        if random.random() > 0.7:  # 30% have coupon
            if i % 40 == 0:
                coupon = None
            elif i % 50 == 0:
                coupon = random.choice(["", "INVALID", "EXPIRED"])
            elif i % 60 == 0:  # Very long coupon code
                coupon = fake.text(max_nb_chars=100)
            else:
                coupon = random.choice(
                    [
                        "SAVE10",
                        "WELCOME20",
                        "FREESHIP",
                        "SUMMER2024",
                        "VIP15",
                        "FLASH50",
                        "CLEARANCE",
                        "LOYALTY",
                    ]
                )
            record["promo_code"] = coupon

        # Saved for later flag
        if random.random() > 0.8:  # 20% saved for later
            if i % 55 == 0:
                saved = None
            elif i % 65 == 0:  # Various boolean representations
                saved = random.choice(["Y", "N", "Yes", "No", "1", "0"])
            else:
                saved = random.choice([True, False])
            record["saved_for_later"] = saved

        # IP address (for fraud detection)
        if random.random() > 0.6:  # 40% have IP
            if i % 45 == 0:
                ip = None
            elif i % 55 == 0:  # Invalid IP
                ip = random.choice(["N/A", "0.0.0.0", "999.999.999.999", "localhost"])
            else:
                ip = fake.ipv4()
            record["ip_address"] = ip

        data.append(record)

    # Create DataFrame
    df = pd.DataFrame(data)

    # Add EXACT duplicate rows (every attribute is same)
    num_exact_duplicates = int(num_rows * 0.02)  # 2% exact duplicates
    for _ in range(num_exact_duplicates):
        if len(df) > 0:
            # Pick a random row to duplicate
            row_to_duplicate = df.sample(1)
            df = pd.concat([df, row_to_duplicate], ignore_index=True)

    # Add some completely empty rows
    for _ in range(int(num_rows * 0.005)):  # 0.5% empty rows
        empty_row = pd.Series([None] * len(df.columns), index=df.columns)
        df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)

    # Add some rows with all string 'NULL' or 'N/A' values
    for _ in range(int(num_rows * 0.01)):  # 1% NULL string rows
        null_values = ["NULL", "N/A", "null", "NA", "", " "]
        null_row = pd.Series(
            [random.choice(null_values) for _ in range(len(df.columns))],
            index=df.columns,
        )
        df = pd.concat([df, pd.DataFrame([null_row])], ignore_index=True)

    # Shuffle the dataframe to mix duplicates throughout
    df = df.sample(frac=1).reset_index(drop=True)

    return df


def add_more_messiness(df):
    """
    Add additional data quality issues to make the dataset more challenging
    """
    # Add trailing/leading spaces to some string columns
    string_cols = df.select_dtypes(include=["object"]).columns
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05  # 5% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    # Add case inconsistencies
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03  # 3% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    # Add special characters to some values
    for col in string_cols[:3]:  # Only first 3 string columns
        mask = np.random.random(len(df)) < 0.02  # 2% of values
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


# Generate the dataset
if __name__ == "__main__":
    # Set the number of rows you want
    NUM_ROWS = 3000  # Change this to your desired number

    # ID formats should match your customer and product datasets
    CUSTOMER_ID_FORMAT = "CUST"  # Options: 'CUST', 'CUSTOMER', or 'NUMBER'
    PRODUCT_ID_FORMAT = "PROD"  # Options: 'PROD', 'PRODUCT', or 'NUMBER'

    print(f"Generating {NUM_ROWS} rows of messy shopping cart data...")
    df = generate_messy_shopping_cart_data(
        NUM_ROWS, CUSTOMER_ID_FORMAT, PRODUCT_ID_FORMAT
    )

    # Add more messiness
    df = add_more_messiness(df)

    # Display basic info
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumn names (realistic but challenging for mapping):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")

    print(f"\nFirst 10 rows:")
    print(df.head(10))

    # Show data quality issues summary
    print("\n" + "=" * 50)
    print("DATA QUALITY ISSUES SUMMARY:")
    print("=" * 50)
    print(f"Total null values: {df.isnull().sum().sum()}")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    print(f"Total rows: {len(df)}")

    # Check NOT NULL constraint violations
    print("\n" + "=" * 50)
    print("NOT NULL CONSTRAINT VIOLATIONS:")
    print("=" * 50)

    if "product_ref" in df.columns:
        null_products = df[df["product_ref"].isnull()].shape[0]
        if null_products > 0:
            print(f"✗ NULL product_ref (violates NOT NULL): {null_products} rows")

    if "item_quantity" in df.columns:
        null_qty = df[df["item_quantity"].isnull()].shape[0]
        if null_qty > 0:
            print(f"✗ NULL item_quantity (violates NOT NULL): {null_qty} rows")

    if "price_per_unit" in df.columns:
        null_price = df[df["price_per_unit"].isnull()].shape[0]
        if null_price > 0:
            print(f"✗ NULL price_per_unit (violates NOT NULL): {null_price} rows")

    # Show business logic violations
    print("\n" + "=" * 50)
    print("BUSINESS LOGIC VIOLATIONS EXAMPLES:")
    print("=" * 50)

    # Check for carts with neither customer_id nor session_id
    if "customer_ref" in df.columns and "session_identifier" in df.columns:
        orphan_carts = df[
            df["customer_ref"].isnull() & df["session_identifier"].isnull()
        ]
        if len(orphan_carts) > 0:
            print(
                f"✗ Carts with neither customer_id nor session_id: {len(orphan_carts)} rows"
            )

    # Check for zero or negative quantities
    if "item_quantity" in df.columns:
        qty_numeric = pd.to_numeric(df["item_quantity"], errors="coerce")
        invalid_qty = df[qty_numeric <= 0]
        if len(invalid_qty) > 0:
            print(f"✗ Zero or negative quantities: {len(invalid_qty)} items")
            print(f"  Examples: {invalid_qty['item_quantity'].unique()[:5].tolist()}")

    # Check for negative prices
    if "price_per_unit" in df.columns:
        price_numeric = pd.to_numeric(df["price_per_unit"], errors="coerce")
        negative_prices = df[price_numeric < 0]
        if len(negative_prices) > 0:
            print(f"✗ Negative prices: {len(negative_prices)} items")

    # Check for converted carts without order ID
    if "status" in df.columns and "order_reference" in df.columns:
        converted_no_order = df[
            (df["status"].isin(["Converted", "converted", "CONVERTED"]))
            & (df["order_reference"].isnull())
        ]
        if len(converted_no_order) > 0:
            print(
                f"✗ Converted carts without order ID: {len(converted_no_order)} carts"
            )

    # Check for non-converted carts with order ID
    if "status" in df.columns and "order_reference" in df.columns:
        not_converted_with_order = df[
            (~df["status"].isin(["Converted", "converted", "CONVERTED"]))
            & (df["order_reference"].notna())
        ]
        if len(not_converted_with_order) > 0:
            print(
                f"✗ Non-converted carts with order ID: {len(not_converted_with_order)} carts"
            )

    # Check for very old active carts
    if "status" in df.columns and "date_added_to_cart" in df.columns:
        added_dt = pd.to_datetime(df["date_added_to_cart"], errors="coerce")
        old_active = df[
            (df["status"].isin(["Active", "active", "ACTIVE"]))
            & (added_dt < (datetime.now() - timedelta(days=30)))
        ]
        if len(old_active) > 0:
            print(f"✗ Active carts older than 30 days: {len(old_active)} carts")

    # Check for discount greater than price
    if "price_per_unit" in df.columns and "discount_amount" in df.columns:
        price_numeric = pd.to_numeric(df["price_per_unit"], errors="coerce")
        discount_numeric = pd.to_numeric(df["discount_amount"], errors="coerce")
        over_discount = df[discount_numeric > price_numeric]
        if len(over_discount) > 0:
            print(f"✗ Discount greater than price: {len(over_discount)} items")

    # Check for invalid foreign keys
    if "customer_ref" in df.columns:
        invalid_customers = df[
            df["customer_ref"].isin(["INVALID", "NULL", "N/A", "", "GUEST"])
        ]
        if len(invalid_customers) > 0:
            print(f"✗ Invalid customer references: {len(invalid_customers)} rows")

    if "product_ref" in df.columns:
        invalid_products = df[
            df["product_ref"].isin(["INVALID", "N/A", "NULL", "", "OUT_OF_STOCK"])
        ]
        if len(invalid_products) > 0:
            print(f"✗ Invalid product references: {len(invalid_products)} rows")

    # Show cart abandonment analysis
    print("\n" + "=" * 50)
    print("CART ABANDONMENT ANALYSIS:")
    print("=" * 50)

    if "status" in df.columns:
        status_counts = df["status"].value_counts()
        print(f"Cart status distribution:")
        for status, count in status_counts.head(10).items():
            print(f"  {status}: {count} ({count/len(df)*100:.1f}%)")

    # Calculate abandonment rate
    if "status" in df.columns:
        abandoned = df["status"].isin(["Abandoned", "abandoned", "ABANDONED"]).sum()
        converted = df["status"].isin(["Converted", "converted", "CONVERTED"]).sum()
        if (abandoned + converted) > 0:
            abandonment_rate = abandoned / (abandoned + converted) * 100
            print(f"\nAbandonment rate: {abandonment_rate:.1f}%")

    # Show session analysis
    print("\n" + "=" * 50)
    print("SESSION ANALYSIS:")
    print("=" * 50)

    if "session_identifier" in df.columns:
        session_stats = (
            df.groupby("session_identifier")
            .agg(
                {
                    "cart_item_id": "count",
                    "price_per_unit": lambda x: pd.to_numeric(x, errors="coerce").sum(),
                }
            )
            .rename(
                columns={
                    "cart_item_id": "items_in_session",
                    "price_per_unit": "session_value",
                }
            )
        )

        # Find sessions with most items
        top_sessions = session_stats.nlargest(5, "items_in_session")
        print(f"Top 5 sessions by item count:")
        for idx, row in top_sessions.iterrows():
            if idx and idx not in ["INVALID", "NULL", "", "000000"]:
                print(
                    f"  Session {str(idx)[:8]}...: {row['items_in_session']} items, value: ${row['session_value']:.2f}"
                )

    # Show guest vs registered analysis
    print("\n" + "=" * 50)
    print("GUEST VS REGISTERED ANALYSIS:")
    print("=" * 50)

    if "customer_ref" in df.columns:
        guest_carts = df[df["customer_ref"].isnull()]
        registered_carts = df[df["customer_ref"].notna()]
        print(f"Guest carts: {len(guest_carts)} ({len(guest_carts)/len(df)*100:.1f}%)")
        print(
            f"Registered user carts: {len(registered_carts)} ({len(registered_carts)/len(df)*100:.1f}%)"
        )

        # Conversion rates by type
        if "status" in df.columns:
            guest_converted = (
                guest_carts["status"]
                .isin(["Converted", "converted", "CONVERTED"])
                .sum()
            )
            reg_converted = (
                registered_carts["status"]
                .isin(["Converted", "converted", "CONVERTED"])
                .sum()
            )

            if len(guest_carts) > 0:
                print(
                    f"Guest conversion rate: {guest_converted/len(guest_carts)*100:.1f}%"
                )
            if len(registered_carts) > 0:
                print(
                    f"Registered conversion rate: {reg_converted/len(registered_carts)*100:.1f}%"
                )

    # Show main columns analysis
    print("\n" + "=" * 50)
    print("MAIN COLUMNS ANALYSIS:")
    print("=" * 50)

    main_columns = [
        "cart_item_id",
        "customer_ref",
        "session_identifier",
        "product_ref",
        "item_quantity",
        "price_per_unit",
        "date_added_to_cart",
        "status",
        "order_reference",
    ]

    for col in main_columns:
        if col in df.columns:
            null_count = df[col].isnull().sum()
            unique_count = df[col].nunique()
            print(f"\n{col}:")
            print(f"  - Null values: {null_count} ({null_count/len(df)*100:.1f}%)")
            print(f"  - Unique values: {unique_count}")
            if col == "status" and unique_count > 0:
                status_dist = df[col].value_counts().head(3)
                print(f"  - Top values: {status_dist.to_dict()}")

    # Save to CSV
    output_file = "messy_shopping_cart_data.xlsx"
    df.to_excel(output_file, index=False)
    print(f"\n✅ Dataset saved to '{output_file}'")

    # Save cart abandonment summary
    abandonment_summary = []

    # Analyze by time periods
    if "date_added_to_cart" in df.columns and "status" in df.columns:
        added_dt = pd.to_datetime(df["date_added_to_cart"], errors="coerce")
        df["days_old"] = (datetime.now() - added_dt).dt.days

        for days in [1, 3, 7, 14, 30]:
            period_carts = df[df["days_old"] <= days]
            if len(period_carts) > 0:
                abandoned = (
                    period_carts["status"].isin(["Abandoned", "abandoned"]).sum()
                )
                converted = (
                    period_carts["status"].isin(["Converted", "converted"]).sum()
                )
                active = period_carts["status"].isin(["Active", "active"]).sum()

                abandonment_summary.append(
                    {
                        "period_days": days,
                        "total_carts": len(period_carts),
                        "abandoned": abandoned,
                        "converted": converted,
                        "active": active,
                        "abandonment_rate": (
                            round(abandoned / len(period_carts) * 100, 2)
                            if len(period_carts) > 0
                            else 0
                        ),
                    }
                )

    if abandonment_summary:
        summary_df = pd.DataFrame(abandonment_summary)
        summary_df.to_csv("cart_abandonment_summary.csv", index=False)
        print(f"✅ Cart abandonment summary saved to 'cart_abandonment_summary.csv'")

Generating 3000 rows of messy shopping cart data...


/tmp/ipykernel_6241/2020148436.py:431: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)
/tmp/ipykernel_6241/2020148436.py:431: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)
/tmp/ipykernel_6241/2020148436.py:431: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or 


Dataset shape: (3105, 16)

Column names (realistic but challenging for mapping):
  1. cart_item_id
  2. customer_ref
  3. session_identifier
  4. product_ref
  5. item_quantity
  6. price_per_unit
  7. date_added_to_cart
  8. status
  9. order_reference
  10. device_type
  11. discount_amount
  12. promo_code
  13. ip_address
  14. tax_amount
  15. last_updated
  16. saved_for_later

First 10 rows:
  cart_item_id customer_ref                    session_identifier  \
0    1001338           None  3dabba61-8a69-4650-a9a2-e9cd7b6f65df   
1      1000419   CUST_01154  94fb09f0-53c9-4ae7-bcf0-7bb3f343d9eb   
2      1000887   CUST_01027                                  None   
3      1002843         None  db747f33-fdba-43eb-b8e1-524090771f71   
4      1000604   CUST_01090  b33422cb-9a3f-4ad1-a9a9-aff9235fc3d5   
5      1002798   CUST_01137                                  None   
6      1001452         None  15668483-c526-4dc8-bf93-672421d67c73   
7      1000062   CUST_01075                  

### Orders Table Generator

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import string
import uuid

# Initialize Faker
fake = Faker(['en_US', 'en_GB', 'fr_FR', 'de_DE'])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

def generate_messy_orders_data(num_rows=2500, customer_id_format='CUST'):
    """
    Generate a messy, realistic orders dataset with realistic but challenging column names
    and various data quality issues for testing data mapping and cleaning.
    
    Args:
        num_rows: Number of order records to generate
        customer_id_format: Format of customer IDs ('CUST', 'CUSTOMER', or 'NUMBER')
    """
    
    data = []
    
    # Track IDs for creating duplicates and relationships
    used_order_ids = []
    used_session_ids = []
    
    # Generate pool of customer IDs
    num_customers = max(num_rows // 5, 100)  # Assuming ~5 orders per customer average
    
    customer_ids = []
    for i in range(num_customers):
        if customer_id_format == 'CUST':
            cust_id = f"CUST_{str(i + 1000).zfill(5)}"
        elif customer_id_format == 'CUSTOMER':
            cust_id = f"CUSTOMER-{str(i + 1000).zfill(5)}"
        else:  # NUMBER
            cust_id = str(10000 + i)
        customer_ids.append(cust_id)
    
    # Choose one consistent order ID format
    order_id_format_choice = random.choice(['ORD', 'ORDER', 'NUMBER'])
    
    # Marketing channels
    channels = ['Google Ads', 'Facebook', 'Email', 'Direct', 'Organic Search', 
                'Instagram', 'Referral', 'Affiliate', 'Social Media', 'Marketplace']
    
    for i in range(num_rows):
        record = {}
        
        # Order ID with CONSISTENT format
        if i % 50 == 0 and used_order_ids:  # 2% duplicates
            order_id = random.choice(used_order_ids)
        else:
            if order_id_format_choice == 'ORD':
                order_id = f"ORD_{str(i + 100000).zfill(6)}"
            elif order_id_format_choice == 'ORDER':
                order_id = f"ORDER-{str(i + 100000).zfill(6)}"
            else:  # NUMBER
                order_id = str(1000000 + i)
            
            used_order_ids.append(order_id)
        
        record['order_ref'] = order_id if i % 100 != 0 else None  # 1% null IDs
        
        # Decide if this is a guest or registered user order
        is_guest = random.random() < 0.25  # 25% guest orders
        
        # Customer ID (NULL for guest orders)
        if is_guest:
            cust_id = None
        elif i % 60 == 0:  # Non-existent customer IDs (FK violation)
            if customer_id_format == 'CUST':
                cust_id = f"CUST_{str(99999).zfill(5)}"
            elif customer_id_format == 'CUSTOMER':
                cust_id = f"CUSTOMER-{str(99999).zfill(5)}"
            else:
                cust_id = "99999"
        elif i % 40 == 0:  # Invalid format
            cust_id = random.choice(['INVALID', 'NULL', 'N/A', '', 'GUEST'])
        elif i % 80 == 0:  # Business logic violation: both customer_id and session_id null
            cust_id = None
            is_guest = False  # Will make session_id also null
        else:
            cust_id = random.choice(customer_ids)
        
        record['customer_ref'] = cust_id
        
        # Session ID (for guest orders)
        if is_guest and cust_id is None:
            # Guest order must have session ID
            if i % 70 == 0:  # Invalid session format
                session_id = random.choice(['INVALID', 'NULL', '', '000000'])
            else:
                session_id = fake.uuid4()
                used_session_ids.append(session_id)
        elif not is_guest and cust_id:
            # Registered user might still have session ID
            if random.random() < 0.4:  # 40% have session ID
                session_id = fake.uuid4()
            else:
                session_id = None
        elif i % 80 == 0:  # Business logic violation case
            session_id = None  # Both customer_id and session_id are null
        else:
            session_id = None
        
        record['session_ref'] = session_id
        
        # Order Date - NOT NULL in schema
        if i % 90 == 0:  # Violates NOT NULL
            order_date = None
        elif i % 30 == 0:  # String format variations
            order_date_dt = fake.date_time_between(start_date='-2y', end_date='now')
            formats = ['%Y-%m-%d %H:%M:%S', '%m/%d/%Y %H:%M', '%d-%m-%Y', '%Y%m%d']
            order_date = order_date_dt.strftime(random.choice(formats))
        elif i % 40 == 0:  # Unix timestamp
            order_date = int(fake.date_time_between(start_date='-2y', end_date='now').timestamp())
        elif i % 50 == 0:  # Future date (business logic violation)
            order_date = fake.date_time_between(start_date='+1d', end_date='+30d')
        elif i % 60 == 0:  # Very old order
            order_date = fake.date_time_between(start_date='-10y', end_date='-5y')
        else:
            order_date = fake.date_time_between(start_date='-2y', end_date='now')
        
        record['purchase_date'] = order_date
        
        # Order Status
        if i % 25 == 0:
            status = None
        elif i % 35 == 0:  # Inconsistent values
            status = random.choice(['pending', 'PENDING', 'P', '1', 'shipped', 'SHIPPED'])
        elif i % 45 == 0:  # Invalid values
            status = random.choice(['In Transit', 'Complete', 'Failed', 'On Hold'])
        elif i % 55 == 0:  # Typos
            status = random.choice(['Pendng', 'Proccessing', 'Shiped', 'Deliverd', 'Cancled'])
        else:
            # Realistic status distribution
            status = random.choices(
                ['Pending', 'Processing', 'Shipped', 'Delivered', 'Cancelled', 'Returned'],
                weights=[10, 15, 20, 40, 10, 5],  # Most orders are delivered
                k=1
            )[0]
        
        record['order_status'] = status
        
        # Financial fields - Generate realistic amounts
        
        # Subtotal
        if i % 20 == 0:
            subtotal = None
        elif i % 30 == 0:  # String values
            subtotal = random.choice(['N/A', 'FREE', 'TBD', ''])
        elif i % 40 == 0:  # Negative values (business logic violation)
            subtotal = round(random.uniform(-500, -10), 2)
        elif i % 50 == 0:  # Zero subtotal
            subtotal = 0
        elif i % 60 == 0:  # Extreme values
            subtotal = random.choice([999999.99, 0.001, -9999])
        else:
            subtotal = round(random.uniform(10, 2000), 2)
        
        record['order_subtotal'] = subtotal
        
        # Tax Amount
        if i % 25 == 0:
            tax = None
        elif i % 35 == 0:  # String values
            tax = random.choice(['Included', 'Exempt', 'N/A'])
        elif i % 45 == 0:  # Negative tax
            tax = round(random.uniform(-50, -1), 2)
        elif i % 55 == 0:  # Tax greater than subtotal (business logic violation)
            if isinstance(subtotal, (int, float)) and subtotal > 0:
                tax = subtotal * 1.5
            else:
                tax = 999
        else:
            if isinstance(subtotal, (int, float)) and subtotal > 0:
                tax = round(subtotal * random.uniform(0.05, 0.15), 2)  # 5-15% tax
            else:
                tax = 0
        
        record['tax_total'] = tax
        
        # Shipping Cost
        if i % 30 == 0:
            shipping = None
        elif i % 40 == 0:  # String values
            shipping = random.choice(['Free', 'FREE SHIPPING', 'N/A'])
        elif i % 50 == 0:  # Negative shipping
            shipping = round(random.uniform(-20, -1), 2)
        elif i % 60 == 0:  # Extreme shipping
            shipping = random.choice([999, 0.001, -99])
        else:
            # Free shipping for large orders
            if isinstance(subtotal, (int, float)) and subtotal > 100:
                shipping = 0 if random.random() < 0.3 else round(random.uniform(5, 25), 2)
            else:
                shipping = round(random.uniform(5, 50), 2)
        
        record['shipping_fee'] = shipping
        
        # Discount Amount
        if i % 35 == 0:
            discount = None
        elif i % 45 == 0:  # String values
            discount = random.choice(['10%', 'SALE', 'N/A'])
        elif i % 55 == 0:  # Negative discount (surcharge?)
            discount = round(random.uniform(-100, -10), 2)
        elif i % 65 == 0:  # Discount greater than subtotal (business logic violation)
            if isinstance(subtotal, (int, float)) and subtotal > 0:
                discount = subtotal * 1.2
            else:
                discount = 1000
        else:
            if isinstance(subtotal, (int, float)) and subtotal > 0:
                discount = round(subtotal * random.uniform(0, 0.3), 2) if random.random() < 0.4 else 0
            else:
                discount = 0
        
        record['discount_total'] = discount
        
        # Total Amount (should be subtotal + tax + shipping - discount)
        if i % 25 == 0:
            total = None
        elif i % 35 == 0:  # String values
            total = random.choice(['PAID', 'PENDING', 'N/A'])
        elif i % 45 == 0:  # Business logic violation: total doesn't match calculation
            if all(isinstance(x, (int, float)) for x in [subtotal, tax, shipping, discount]):
                correct_total = subtotal + tax + shipping - discount
                total = correct_total * random.uniform(0.5, 1.5)  # Wrong calculation
            else:
                total = random.uniform(10, 1000)
        elif i % 55 == 0:  # Negative total
            total = round(random.uniform(-500, -1), 2)
        elif i % 65 == 0:  # Zero total
            total = 0
        else:
            # Correct calculation
            if all(isinstance(x, (int, float)) for x in [subtotal, tax, shipping, discount]):
                total = round(subtotal + tax + shipping - discount, 2)
            elif isinstance(subtotal, (int, float)):
                total = round(subtotal * 1.1, 2)  # Approximate
            else:
                total = round(random.uniform(10, 2000), 2)
        
        record['grand_total'] = total
        
        # Currency
        if i % 30 == 0:
            currency = None
        elif i % 40 == 0:  # Invalid currency codes
            currency = random.choice(['US', 'EURO', 'Dollar', '$', '€', 'N/A'])
        elif i % 50 == 0:  # Uncommon currencies
            currency = random.choice(['JPY', 'CNY', 'INR', 'BTC', 'DOGE'])
        else:
            currency = random.choices(
                ['USD', 'EUR', 'GBP', 'CAD', 'AUD'],
                weights=[60, 20, 10, 5, 5],
                k=1
            )[0]
        
        record['currency_code'] = currency
        
        # Acquisition Channel
        if i % 25 == 0:
            channel = None
        elif i % 35 == 0:  # Inconsistent values
            channel = random.choice(['google', 'GOOGLE', 'fb', 'FB', 'email', 'EMAIL'])
        elif i % 45 == 0:  # Invalid values
            channel = random.choice(['Unknown', 'N/A', '?', ''])
        else:
            channel = random.choice(channels)
        
        record['marketing_channel'] = channel
        
        # Device Type
        if i % 30 == 0:
            device = None
        elif i % 40 == 0:  # Inconsistent values
            device = random.choice(['mobile', 'MOBILE', 'desk', 'DESKTOP'])
        elif i % 50 == 0:  # Invalid values
            device = random.choice(['Unknown', 'Bot', 'API', ''])
        else:
            device = random.choices(
                ['Desktop', 'Mobile', 'Tablet', 'App'],
                weights=[40, 40, 10, 10],
                k=1
            )[0]
        
        record['device_category'] = device
        
        # Shipped Date (should be after order date)
        shipped_date = None
        if status in ['Shipped', 'Delivered', 'shipped', 'SHIPPED', 'Shiped', 'Deliverd']:
            if i % 40 == 0:  # Business logic violation: shipped before ordered
                if isinstance(order_date, datetime):
                    shipped_date = order_date - timedelta(days=random.randint(1, 5))
                else:
                    shipped_date = fake.date_time_between(start_date='-3y', end_date='-2y')
            elif i % 50 == 0:  # String format
                if isinstance(order_date, datetime):
                    shipped_date = (order_date + timedelta(days=random.randint(1, 3))).strftime('%Y-%m-%d')
                else:
                    shipped_date = fake.date_between(start_date='-1y', end_date='today')
            elif i % 60 == 0:  # Future shipping date
                shipped_date = fake.date_time_between(start_date='+1d', end_date='+30d')
            else:
                if isinstance(order_date, datetime):
                    shipped_date = fake.date_time_between(
                        start_date=order_date,
                        end_date=min(order_date + timedelta(days=7), datetime.now())
                    )
                else:
                    shipped_date = fake.date_time_between(start_date='-1y', end_date='now')
        elif i % 70 == 0:  # Business logic violation: not shipped but has date
            shipped_date = fake.date_time_between(start_date='-1y', end_date='now')
        
        record['shipment_date'] = shipped_date
        
        # Delivered Date (should be after shipped date)
        delivered_date = None
        if status in ['Delivered', 'delivered', 'Deliverd']:
            if i % 45 == 0:  # Business logic violation: delivered before shipped
                if isinstance(shipped_date, datetime):
                    delivered_date = shipped_date - timedelta(days=random.randint(1, 3))
                elif isinstance(order_date, datetime):
                    delivered_date = order_date - timedelta(days=random.randint(1, 5))
                else:
                    delivered_date = fake.date_time_between(start_date='-3y', end_date='-2y')
            elif i % 55 == 0:  # Same day delivery
                delivered_date = shipped_date
            elif i % 65 == 0:  # Future delivery
                delivered_date = fake.date_time_between(start_date='+1d', end_date='+30d')
            else:
                if isinstance(shipped_date, datetime):
                    delivered_date = fake.date_time_between(
                        start_date=shipped_date,
                        end_date=min(shipped_date + timedelta(days=10), datetime.now())
                    )
                elif isinstance(order_date, datetime):
                    delivered_date = fake.date_time_between(
                        start_date=order_date + timedelta(days=3),
                        end_date=min(order_date + timedelta(days=14), datetime.now())
                    )
                else:
                    delivered_date = fake.date_time_between(start_date='-6m', end_date='now')
        elif i % 75 == 0:  # Business logic violation: not delivered but has date
            delivered_date = fake.date_time_between(start_date='-1y', end_date='now')
        
        record['delivery_date'] = delivered_date
        
        # Created At timestamp
        if i % 30 == 0:
            created = None
        elif i % 40 == 0:  # String format
            created = fake.date_time_between(start_date='-2y', end_date='now').strftime('%Y-%m-%d %H:%M:%S')
        elif i % 50 == 0:  # Created after order date (business logic violation)
            if isinstance(order_date, datetime):
                created = order_date + timedelta(days=random.randint(1, 30))
            else:
                created = fake.date_time_between(start_date='+1d', end_date='+30d')
        else:
            if isinstance(order_date, datetime):
                created = order_date + timedelta(minutes=random.randint(0, 60))
            else:
                created = fake.date_time_between(start_date='-2y', end_date='now')
        
        record['record_created'] = created
        
        # Additional realistic columns that might exist
        
        # Payment method
        if random.random() > 0.3:  # 70% have payment method
            if i % 35 == 0:
                payment = None
            elif i % 45 == 0:
                payment = random.choice(['', 'N/A', 'Unknown'])
            else:
                payment = random.choice([
                    'Credit Card', 'Debit Card', 'PayPal', 'Apple Pay', 'Google Pay',
                    'Bank Transfer', 'Cash on Delivery', 'Bitcoin', 'Gift Card', 'Store Credit'
                ])
            record['payment_method'] = payment
        
        # Shipping method
        if random.random() > 0.4:  # 60% have shipping method
            if i % 40 == 0:
                ship_method = None
            elif i % 50 == 0:
                ship_method = random.choice(['', 'N/A', 'TBD'])
            else:
                ship_method = random.choice([
                    'Standard', 'Express', 'Overnight', 'Economy', 'Priority',
                    'Same Day', '2-Day', 'Free Shipping', 'White Glove'
                ])
            record['shipping_method'] = ship_method
        
        # Number of items
        if random.random() > 0.5:  # 50% have item count
            if i % 45 == 0:
                items = None
            elif i % 55 == 0:  # String values
                items = random.choice(['Multiple', 'Few', 'Many'])
            elif i % 65 == 0:  # Zero or negative
                items = random.choice([0, -1, -5])
            elif i % 75 == 0:  # Extreme values
                items = random.choice([999, 0.5, 10000])
            else:
                items = random.choices(
                    [1, 2, 3, 4, 5, 10, 20],
                    weights=[30, 25, 20, 10, 10, 3, 2],
                    k=1
                )[0]
            record['item_count'] = items
        
        # Coupon code used
        if random.random() > 0.6:  # 40% used coupons
            if i % 50 == 0:
                coupon = None
            elif i % 60 == 0:
                coupon = random.choice(['', 'INVALID', 'EXPIRED'])
            else:
                coupon = random.choice([
                    'SAVE10', 'WELCOME20', 'FREESHIP', 'SUMMER2024',
                    'VIP15', 'FLASH50', 'BLACKFRIDAY', 'LOYALTY'
                ])
            record['coupon_code'] = coupon
        
        # Customer IP
        if random.random() > 0.7:  # 30% have IP
            if i % 55 == 0:
                ip = None
            elif i % 65 == 0:  # Invalid IP
                ip = random.choice(['N/A', '0.0.0.0', '999.999.999.999'])
            else:
                ip = fake.ipv4()
            record['customer_ip'] = ip
        
        # Refund amount (for returned orders)
        if status in ['Returned', 'Cancelled', 'returned', 'RETURNED', 'Cancled']:
            if i % 60 == 0:
                refund = None
            elif i % 70 == 0:  # String values
                refund = random.choice(['Full', 'Partial', 'Pending'])
            elif i % 80 == 0:  # Refund greater than total (business logic violation)
                if isinstance(total, (int, float)) and total > 0:
                    refund = total * 1.5
                else:
                    refund = 1000
            else:
                if isinstance(total, (int, float)) and total > 0:
                    refund = round(total * random.uniform(0.5, 1.0), 2)
                else:
                    refund = round(random.uniform(10, 500), 2)
            record['refund_amount'] = refund
        
        data.append(record)
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Add EXACT duplicate rows (every attribute is same)
    num_exact_duplicates = int(num_rows * 0.02)  # 2% exact duplicates
    for _ in range(num_exact_duplicates):
        if len(df) > 0:
            # Pick a random row to duplicate
            row_to_duplicate = df.sample(1)
            df = pd.concat([df, row_to_duplicate], ignore_index=True)
    
    # Add some completely empty rows
    for _ in range(int(num_rows * 0.005)):  # 0.5% empty rows
        empty_row = pd.Series([None] * len(df.columns), index=df.columns)
        df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)
    
    # Add some rows with all string 'NULL' or 'N/A' values
    for _ in range(int(num_rows * 0.01)):  # 1% NULL string rows
        null_values = ['NULL', 'N/A', 'null', 'NA', '', ' ']
        null_row = pd.Series([random.choice(null_values) for _ in range(len(df.columns))], index=df.columns)
        df = pd.concat([df, pd.DataFrame([null_row])], ignore_index=True)
    
    # Shuffle the dataframe to mix duplicates throughout
    df = df.sample(frac=1).reset_index(drop=True)
    
    return df

def add_more_messiness(df):
    """
    Add additional data quality issues to make the dataset more challenging
    """
    # Add trailing/leading spaces to some string columns
    string_cols = df.select_dtypes(include=['object']).columns
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05  # 5% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: '  ' + str(x) + '  ' if pd.notna(x) else x
        )
    
    # Add case inconsistencies
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03  # 3% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x).upper() if pd.notna(x) and random.random() > 0.5 else str(x).lower() if pd.notna(x) else x
        )
    
    # Add special characters to some values
    for col in string_cols[:3]:  # Only first 3 string columns
        mask = np.random.random(len(df)) < 0.02  # 2% of values
        special_chars = ['@', '#', '!', '*', '&', '%']
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )
    
    return df

# Generate the dataset
if __name__ == "__main__":
    # Set the number of rows you want
    NUM_ROWS = 2500  # Change this to your desired number
    
    # ID format should match your customer dataset
    CUSTOMER_ID_FORMAT = 'CUST'  # Options: 'CUST', 'CUSTOMER', or 'NUMBER'
    
    print(f"Generating {NUM_ROWS} rows of messy orders data...")
    df = generate_messy_orders_data(NUM_ROWS, CUSTOMER_ID_FORMAT)
    
    # Add more messiness
    df = add_more_messiness(df)
    
    # Display basic info
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumn names (realistic but challenging for mapping):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")
    
    print(f"\nFirst 10 rows:")
    print(df.head(10))
    
    # Show data quality issues summary
    print("\n" + "="*50)
    print("DATA QUALITY ISSUES SUMMARY:")
    print("="*50)
    print(f"Total null values: {df.isnull().sum().sum()}")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    print(f"Total rows: {len(df)}")
    
    # Check NOT NULL constraint violations
    print("\n" + "="*50)
    print("NOT NULL CONSTRAINT VIOLATIONS:")
    print("="*50)
    
    if 'purchase_date' in df.columns:
        null_dates = df[df['purchase_date'].isnull()].shape[0]
        if null_dates > 0:
            print(f"✗ NULL purchase_date (violates NOT NULL): {null_dates} rows")
    
    # Show business logic violations
    print("\n" + "="*50)
    print("BUSINESS LOGIC VIOLATIONS EXAMPLES:")
    print("="*50)
    
    # Check for orders with neither customer_id nor session_id
    if 'customer_ref' in df.columns and 'session_ref' in df.columns:
        orphan_orders = df[df['customer_ref'].isnull() & df['session_ref'].isnull()]
        if len(orphan_orders) > 0:
            print(f"✗ Orders with neither customer_id nor session_id: {len(orphan_orders)} rows")
    
    # Check for negative amounts
    for col in ['order_subtotal', 'grand_total']:
        if col in df.columns:
            amount_numeric = pd.to_numeric(df[col], errors='coerce')
            negative_amounts = df[amount_numeric < 0]
            if len(negative_amounts) > 0:
                print(f"✗ Negative {col}: {len(negative_amounts)} orders")
    
    # Check for discount greater than subtotal
    if 'order_subtotal' in df.columns and 'discount_total' in df.columns:
        subtotal_numeric = pd.to_numeric(df['order_subtotal'], errors='coerce')
        discount_numeric = pd.to_numeric(df['discount_total'], errors='coerce')
        over_discount = df[discount_numeric > subtotal_numeric]
        if len(over_discount) > 0:
            print(f"✗ Discount greater than subtotal: {len(over_discount)} orders")
    
    # Check for tax greater than subtotal
    if 'order_subtotal' in df.columns and 'tax_total' in df.columns:
        subtotal_numeric = pd.to_numeric(df['order_subtotal'], errors='coerce')
        tax_numeric = pd.to_numeric(df['tax_total'], errors='coerce')
        over_tax = df[tax_numeric > subtotal_numeric]
        if len(over_tax) > 0:
            print(f"✗ Tax greater than subtotal: {len(over_tax)} orders")
    
    # Check for incorrect total calculation
    if all(col in df.columns for col in ['order_subtotal', 'tax_total', 'shipping_fee', 'discount_total', 'grand_total']):
        subtotal = pd.to_numeric(df['order_subtotal'], errors='coerce').fillna(0)
        tax = pd.to_numeric(df['tax_total'], errors='coerce').fillna(0)
        shipping = pd.to_numeric(df['shipping_fee'], errors='coerce').fillna(0)
        discount = pd.to_numeric(df['discount_total'], errors='coerce').fillna(0)
        total = pd.to_numeric(df['grand_total'], errors='coerce').fillna(0)
        
        calculated_total = subtotal + tax + shipping - discount
        wrong_total = df[abs(total - calculated_total) > 1]  # Allow $1 rounding difference
        if len(wrong_total) > 0:
            print(f"✗ Incorrect total calculation: {len(wrong_total)} orders")
    
    # Check date logic violations
    if 'purchase_date' in df.columns and 'shipment_date' in df.columns:
        order_dt = pd.to_datetime(df['purchase_date'], errors='coerce')
        shipped_dt = pd.to_datetime(df['shipment_date'], errors='coerce')
        shipped_before_order = df[shipped_dt < order_dt]
        if len(shipped_before_order) > 0:
            print(f"✗ Shipped before ordered: {len(shipped_before_order)} orders")
    
    if 'shipment_date' in df.columns and 'delivery_date' in df.columns:
        shipped_dt = pd.to_datetime(df['shipment_date'], errors='coerce')
        delivered_dt = pd.to_datetime(df['delivery_date'], errors='coerce')
        delivered_before_shipped = df[delivered_dt < shipped_dt]
        if len(delivered_before_shipped) > 0:
            print(f"✗ Delivered before shipped: {len(delivered_before_shipped)} orders")
    
    # Check status-date mismatches
    if 'order_status' in df.columns and 'shipment_date' in df.columns:
        not_shipped_with_date = df[
            (~df['order_status'].isin(['Shipped', 'Delivered', 'shipped', 'SHIPPED'])) &
            (df['shipment_date'].notna())
        ]
        if len(not_shipped_with_date) > 0:
            print(f"✗ Not shipped status but has shipment date: {len(not_shipped_with_date)} orders")
    
    # Check for refunds greater than total
    if 'grand_total' in df.columns and 'refund_amount' in df.columns:
        total_numeric = pd.to_numeric(df['grand_total'], errors='coerce')
        refund_numeric = pd.to_numeric(df['refund_amount'], errors='coerce')
        over_refund = df[refund_numeric > total_numeric]
        if len(over_refund) > 0:
            print(f"✗ Refund greater than total: {len(over_refund)} orders")
    
    # Show order statistics
    print("\n" + "="*50)
    print("ORDER STATISTICS:")
    print("="*50)
    
    if 'order_status' in df.columns:
        status_counts = df['order_status'].value_counts()
        print(f"Order status distribution:")
        for status, count in status_counts.head(10).items():
            print(f"  {status}: {count} ({count/len(df)*100:.1f}%)")
    
    # Revenue analysis
    if 'grand_total' in df.columns:
        total_numeric = pd.to_numeric(df['grand_total'], errors='coerce')
        valid_totals = total_numeric[total_numeric > 0]
        if len(valid_totals) > 0:
            print(f"\nRevenue metrics:")
            print(f"  Total revenue: ${valid_totals.sum():,.2f}")
            print(f"  Average order value: ${valid_totals.mean():.2f}")
            print(f"  Median order value: ${valid_totals.median():.2f}")
            print(f"  Max order: ${valid_totals.max():.2f}")
            print(f"  Min order: ${valid_totals.min():.2f}")
    
    # Channel analysis
    print("\n" + "="*50)
    print("CHANNEL ANALYSIS:")
    print("="*50)
    
    if 'marketing_channel' in df.columns:
        channel_stats = df.groupby('marketing_channel').agg({
            'order_ref': 'count',
            'grand_total': lambda x: pd.to_numeric(x, errors='coerce').sum()
        }).rename(columns={
            'order_ref': 'order_count',
            'grand_total': 'total_revenue'
        })
        
        top_channels = channel_stats.nlargest(5, 'total_revenue')
        print(f"Top 5 channels by revenue:")
        for idx, row in top_channels.iterrows():
            if idx and idx not in ['Unknown', 'N/A', '?', '']:
                print(f"  {idx}: {row['order_count']} orders, ${row['total_revenue']:,.2f}")
    
    # Device analysis
    if 'device_category' in df.columns:
        device_counts = df['device_category'].value_counts()
        print(f"\nDevice distribution:")
        for device, count in device_counts.head(5).items():
            if device:
                print(f"  {device}: {count} ({count/len(df)*100:.1f}%)")
    
    # Guest vs registered analysis
    print("\n" + "="*50)
    print("GUEST VS REGISTERED ANALYSIS:")
    print("="*50)
    
    if 'customer_ref' in df.columns:
        guest_orders = df[df['customer_ref'].isnull()]
        registered_orders = df[df['customer_ref'].notna()]
        print(f"Guest orders: {len(guest_orders)} ({len(guest_orders)/len(df)*100:.1f}%)")
        print(f"Registered orders: {len(registered_orders)} ({len(registered_orders)/len(df)*100:.1f}%)")
        
        # Average order value by type
        if 'grand_total' in df.columns:
            guest_aov = pd.to_numeric(guest_orders['grand_total'], errors='coerce').mean()
            reg_aov = pd.to_numeric(registered_orders['grand_total'], errors='coerce').mean()
            print(f"Guest AOV: ${guest_aov:.2f}")
            print(f"Registered AOV: ${reg_aov:.2f}")
    
    # Show main columns analysis
    print("\n" + "="*50)
    print("MAIN COLUMNS ANALYSIS:")
    print("="*50)
    
    main_columns = [
        'order_ref', 'customer_ref', 'session_ref', 'purchase_date',
        'order_status', 'order_subtotal', 'tax_total', 'shipping_fee',
        'discount_total', 'grand_total', 'currency_code'
    ]
    
    for col in main_columns:
        if col in df.columns:
            null_count = df[col].isnull().sum()
            unique_count = df[col].nunique()
            print(f"\n{col}:")
            print(f"  - Null values: {null_count} ({null_count/len(df)*100:.1f}%)")
            print(f"  - Unique values: {unique_count}")
            if col == 'currency_code' and unique_count > 0:
                currency_dist = df[col].value_counts().head(3)
                print(f"  - Top currencies: {currency_dist.to_dict()}")
    
    # Save to CSV
    output_file = 'messy_orders_data.xlsx'
    df.to_excel(output_file, index=False)
    print(f"\n✅ Dataset saved to '{output_file}'")
    
    # Save order summary by month
    monthly_summary = []
    
    if 'purchase_date' in df.columns and 'grand_total' in df.columns:
        order_dt = pd.to_datetime(df['purchase_date'], errors='coerce')
        df['order_month'] = order_dt.dt.to_period('M')
        
        for month in df['order_month'].dropna().unique()[:12]:
            month_orders = df[df['order_month'] == month]
            total_revenue = pd.to_numeric(month_orders['grand_total'], errors='coerce').sum()
            
            monthly_summary.append({
                'month': str(month),
                'order_count': len(month_orders),
                'total_revenue': round(total_revenue, 2),
                'avg_order_value': round(total_revenue/len(month_orders), 2) if len(month_orders) > 0 else 0,
                'delivered': month_orders['order_status'].isin(['Delivered', 'delivered']).sum() if 'order_status' in df.columns else 0,
                'cancelled': month_orders['order_status'].isin(['Cancelled', 'Cancled']).sum() if 'order_status' in df.columns else 0
            })
    
    if monthly_summary:
        summary_df = pd.DataFrame(monthly_summary)
        summary_df.to_csv('orders_monthly_summary.csv', index=False)
        print(f"✅ Monthly orders summary saved to 'orders_monthly_summary.csv'")

Generating 2500 rows of messy orders data...


/tmp/ipykernel_9007/3010401780.py:477: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)



Dataset shape: (2587, 22)

Column names (realistic but challenging for mapping):
  1. order_ref
  2. customer_ref
  3. session_ref
  4. purchase_date
  5. order_status
  6. order_subtotal
  7. tax_total
  8. shipping_fee
  9. discount_total
  10. grand_total
  11. currency_code
  12. marketing_channel
  13. device_category
  14. shipment_date
  15. delivery_date
  16. record_created
  17. coupon_code
  18. payment_method
  19. shipping_method
  20. item_count
  21. customer_ip
  22. refund_amount

First 10 rows:
  order_ref customer_ref                           session_ref  \
0        NA         null                                  NULL   
1   1001213         None  7880adc6-2af2-4718-a0b2-17f965f35984   
2   1001879   CUST_01363  f46976af-10be-43e8-9aba-c34bae9802fa   
3   1001233   CUST_01274                                  None   
4   1001907         None  4e9f67a7-16d4-4dff-b260-2a263025f041   
5      None         None                                  None   
6   1000068        

/tmp/ipykernel_9007/3010401780.py:610: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  order_dt = pd.to_datetime(df['purchase_date'], errors='coerce')
/tmp/ipykernel_9007/3010401780.py:611: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  shipped_dt = pd.to_datetime(df['shipment_date'], errors='coerce')
/tmp/ipykernel_9007/3010401780.py:617: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  shipped_dt = pd.to_datetime(df['shipment_date'], errors='coerce')
/tmp/ipykernel_9007/3010401780.py:618: UserWarning: Could not infer format, so each element will be parsed individually, falling 


✅ Dataset saved to 'messy_orders_data.xlsx'
✅ Monthly orders summary saved to 'orders_monthly_summary.csv'


/tmp/ipykernel_9007/3010401780.py:740: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  order_dt = pd.to_datetime(df['purchase_date'], errors='coerce')


### Orders Items Table Generator

In [9]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import string

# Initialize Faker
fake = Faker(['en_US', 'en_GB'])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

def generate_messy_order_items_data(num_rows=5000, order_id_format='ORD', product_id_format='PROD'):
    """
    Generate a messy, realistic order_items dataset with realistic but challenging column names
    and various data quality issues for testing data mapping and cleaning.
    
    Args:
        num_rows: Number of order item records to generate
        order_id_format: Format of order IDs ('ORD', 'ORDER', or 'NUMBER')
        product_id_format: Format of product IDs ('PROD', 'PRODUCT', or 'NUMBER')
    """
    
    data = []
    
    # Track IDs for creating duplicates and relationships
    used_order_item_ids = []
    
    # Generate pools of order and product IDs
    num_orders = max(num_rows // 3, 200)  # Assuming ~3 items per order average
    num_products = max(num_rows // 10, 100)  # Pool of products
    
    order_ids = []
    for i in range(num_orders):
        if order_id_format == 'ORD':
            order_id = f"ORD_{str(i + 100000).zfill(6)}"
        elif order_id_format == 'ORDER':
            order_id = f"ORDER-{str(i + 100000).zfill(6)}"
        else:  # NUMBER
            order_id = str(1000000 + i)
        order_ids.append(order_id)
    
    product_ids = []
    for i in range(num_products):
        if product_id_format == 'PROD':
            prod_id = f"PROD_{str(i + 1).zfill(4)}"
        elif product_id_format == 'PRODUCT':
            prod_id = f"PRODUCT-{str(i + 1).zfill(4)}"
        else:  # NUMBER
            prod_id = str(10000 + i)
        product_ids.append(prod_id)
    
    # Choose one consistent order item ID format
    item_id_format_choice = random.choice(['OI', 'ORDERITEM', 'NUMBER'])
    
    # Track order-product combinations for realistic grouping
    order_items_map = {}
    
    # Generate base prices for products (for consistency within same product)
    product_base_prices = {}
    product_base_costs = {}
    for prod_id in product_ids:
        product_base_prices[prod_id] = round(random.uniform(5, 500), 2)
        # Cost is typically 40-70% of price
        product_base_costs[prod_id] = round(product_base_prices[prod_id] * random.uniform(0.4, 0.7), 2)
    
    for i in range(num_rows):
        record = {}
        
        # Order Item ID with CONSISTENT format
        if i % 50 == 0 and used_order_item_ids:  # 2% duplicates
            item_id = random.choice(used_order_item_ids)
        else:
            if item_id_format_choice == 'OI':
                item_id = f"OI{str(i + 1).zfill(5)}"
            elif item_id_format_choice == 'ORDERITEM':
                item_id = f"ORDERITEM-{str(i + 1).zfill(5)}"
            else:  # NUMBER
                item_id = str(100000 + i)
            
            used_order_item_ids.append(item_id)
        
        record['line_item_id'] = item_id if i % 100 != 0 else None  # 1% null IDs
        
        # Order ID - NOT NULL in schema
        if i % 85 == 0:  # Violates NOT NULL
            order_id = None
        elif i % 60 == 0:  # Non-existent order IDs (FK violation)
            if order_id_format == 'ORD':
                order_id = f"ORD_{str(999999).zfill(6)}"
            elif order_id_format == 'ORDER':
                order_id = f"ORDER-{str(999999).zfill(6)}"
            else:
                order_id = "9999999"
        elif i % 40 == 0:  # Invalid format
            order_id = random.choice(['INVALID', 'NULL', 'N/A', '', 'MISSING'])
        else:
            # Group items by order (realistic basket behavior)
            if random.random() < 0.6 and order_items_map:  # 60% chance to add to existing order
                # Pick a recent order that doesn't have too many items
                recent_orders = [o for o, items in order_items_map.items() if len(items) < 10]
                if recent_orders:
                    order_id = random.choice(recent_orders[-20:])  # Recent 20 orders
                else:
                    order_id = random.choice(order_ids)
            else:
                order_id = random.choice(order_ids)
        
        record['order_ref'] = order_id
        
        # Product ID - NOT NULL in schema
        if i % 80 == 0:  # Violates NOT NULL
            prod_id = None
        elif i % 55 == 0:  # Non-existent product IDs (FK violation)
            if product_id_format == 'PROD':
                prod_id = f"PROD_{str(9999).zfill(4)}"
            elif product_id_format == 'PRODUCT':
                prod_id = f"PRODUCT-{str(9999).zfill(4)}"
            else:
                prod_id = "99999"
        elif i % 45 == 0:  # Invalid format
            prod_id = random.choice(['INVALID', 'N/A', 'NULL', '', 'DISCONTINUED'])
        else:
            # Check if this product is already in the order (duplicate item - different lines)
            if order_id in order_items_map and random.random() < 0.1:  # 10% chance of duplicate product
                if order_items_map[order_id]:
                    prod_id = random.choice(order_items_map[order_id])
                else:
                    prod_id = random.choice(product_ids)
            else:
                # Popular products appear more often
                popular_products = product_ids[:20]  # First 20 products are popular
                if random.random() < 0.3:  # 30% chance of popular product
                    prod_id = random.choice(popular_products)
                else:
                    prod_id = random.choice(product_ids)
        
        # Track order-product mapping
        if order_id and prod_id:
            if order_id not in order_items_map:
                order_items_map[order_id] = []
            order_items_map[order_id].append(prod_id)
        
        record['product_ref'] = prod_id
        
        # Quantity - NOT NULL in schema
        if i % 90 == 0:  # Violates NOT NULL
            quantity = None
        elif i % 30 == 0:  # String values
            quantity = random.choice(['One', 'Two', 'Many', 'N/A', 'A few'])
        elif i % 40 == 0:  # Zero or negative (business logic violation)
            quantity = random.choice([0, -1, -10, -100])
        elif i % 50 == 0:  # Extreme values
            quantity = random.choice([999, 10000, 0.5, -999])
        elif i % 60 == 0:  # Decimal quantities for non-decimal products
            quantity = random.choice([1.5, 2.3, 3.7, 10.25])
        else:
            # Realistic quantity distribution
            quantity = random.choices(
                [1, 2, 3, 4, 5, 10, 20, 50, 100],
                weights=[50, 20, 10, 5, 5, 5, 3, 1, 1],  # Most items have quantity 1-2
                k=1
            )[0]
        
        record['qty_ordered'] = quantity
        
        # Unit Price - NOT NULL in schema
        if prod_id in product_base_prices:
            base_price = product_base_prices[prod_id]
        else:
            base_price = round(random.uniform(5, 500), 2)
        
        if i % 95 == 0:  # Violates NOT NULL
            unit_price = None
        elif i % 35 == 0:  # String values
            unit_price = random.choice(['Free', 'N/A', 'Contact for price', '', 'TBD'])
        elif i % 45 == 0:  # Negative prices (business logic violation)
            unit_price = round(random.uniform(-100, -1), 2)
        elif i % 55 == 0:  # Zero price
            unit_price = 0
        elif i % 65 == 0:  # Extreme prices
            unit_price = random.choice([999999.99, 0.001, -9999])
        elif i % 75 == 0:  # Price with too many decimals
            unit_price = round(base_price, 5)
        elif i % 25 == 0:  # Price variation (sale price, different from base)
            unit_price = round(base_price * random.uniform(0.5, 0.9), 2)  # 10-50% discount
        else:
            # Small variations around base price
            unit_price = round(base_price * random.uniform(0.95, 1.05), 2)
        
        record['unit_selling_price'] = unit_price
        
        # Discount Amount
        if i % 30 == 0:
            discount = None
        elif i % 40 == 0:  # String values
            discount = random.choice(['10%', 'SALE', 'N/A', 'Free shipping'])
        elif i % 50 == 0:  # Negative discount (surcharge?)
            discount = round(random.uniform(-50, -1), 2)
        elif i % 60 == 0:  # Discount greater than price (business logic violation)
            if isinstance(unit_price, (int, float)) and isinstance(quantity, (int, float)):
                discount = unit_price * quantity * 1.2  # 120% discount
            else:
                discount = 1000
        else:
            if isinstance(unit_price, (int, float)) and unit_price > 0:
                # Discount based on quantity or random promotion
                if isinstance(quantity, (int, float)) and quantity >= 10:
                    discount = round(unit_price * quantity * 0.1, 2)  # 10% bulk discount
                elif random.random() < 0.3:  # 30% have some discount
                    discount = round(unit_price * random.uniform(0.05, 0.25), 2)
                else:
                    discount = 0
            else:
                discount = 0
        
        record['item_discount'] = discount
        
        # Total Price - NOT NULL (should be quantity × unit_price - discount)
        if i % 100 == 0:  # Violates NOT NULL
            total_price = None
        elif i % 35 == 0:  # String values
            total_price = random.choice(['PAID', 'PENDING', 'N/A', 'Calculate'])
        elif i % 45 == 0:  # Business logic violation: incorrect calculation
            if all(isinstance(x, (int, float)) for x in [quantity, unit_price, discount]):
                correct_total = (quantity * unit_price) - discount
                total_price = round(correct_total * random.uniform(0.5, 1.5), 2)  # Wrong calculation
            else:
                total_price = random.uniform(10, 1000)
        elif i % 55 == 0:  # Negative total
            total_price = round(random.uniform(-500, -1), 2)
        elif i % 65 == 0:  # Zero total (but has quantity and price)
            total_price = 0
        else:
            # Correct calculation
            if all(isinstance(x, (int, float)) for x in [quantity, unit_price]):
                discount_val = discount if isinstance(discount, (int, float)) else 0
                total_price = round((quantity * unit_price) - discount_val, 2)
                # Ensure non-negative
                if total_price < 0:
                    total_price = 0
            else:
                total_price = round(random.uniform(10, 1000), 2)
        
        record['line_total'] = total_price
        
        # Product Cost (for margin calculation)
        if prod_id in product_base_costs:
            base_cost = product_base_costs[prod_id]
        else:
            base_cost = round(base_price * 0.6, 2) if isinstance(base_price, (int, float)) else 10
        
        if i % 25 == 0:
            product_cost = None
        elif i % 35 == 0:  # String values
            product_cost = random.choice(['N/A', 'Unknown', 'TBD'])
        elif i % 45 == 0:  # Negative cost
            product_cost = round(random.uniform(-50, -1), 2)
        elif i % 55 == 0:  # Cost greater than price (business logic violation - negative margin)
            if isinstance(unit_price, (int, float)) and unit_price > 0:
                product_cost = unit_price * 1.5  # 150% of selling price
            else:
                product_cost = 999
        elif i % 65 == 0:  # Zero cost
            product_cost = 0
        else:
            # Normal cost with small variations
            product_cost = round(base_cost * random.uniform(0.95, 1.05), 2)
        
        record['unit_cost'] = product_cost
        
        # Created At timestamp
        if i % 30 == 0:
            created = None
        elif i % 40 == 0:  # String format
            created = fake.date_time_between(start_date='-2y', end_date='now').strftime('%Y-%m-%d %H:%M:%S')
        elif i % 50 == 0:  # Unix timestamp
            created = int(fake.date_time_between(start_date='-2y', end_date='now').timestamp())
        elif i % 60 == 0:  # Future date (business logic violation)
            created = fake.date_time_between(start_date='+1d', end_date='+30d')
        else:
            created = fake.date_time_between(start_date='-2y', end_date='now')
        
        record['created_timestamp'] = created
        
        # Additional realistic columns that might exist
        
        # Tax amount per item
        if random.random() > 0.5:  # 50% have tax calculated
            if i % 40 == 0:
                tax = None
            elif i % 50 == 0:  # String values
                tax = random.choice(['Included', 'Exempt', 'N/A'])
            elif i % 60 == 0:  # Negative tax
                tax = round(random.uniform(-10, -1), 2)
            else:
                if isinstance(total_price, (int, float)) and total_price > 0:
                    tax = round(total_price * random.uniform(0.05, 0.15), 2)  # 5-15% tax
                else:
                    tax = 0
            record['tax_amount'] = tax
        
        # SKU (product code)
        if random.random() > 0.6:  # 40% have SKU
            if i % 45 == 0:
                sku = None
            elif i % 55 == 0:
                sku = random.choice(['N/A', 'NULL', 'SKU'])
            else:
                if prod_id:
                    # Generate SKU based on product ID
                    sku = f"SKU-{prod_id[-4:]}-{random.choice(['BLK', 'WHT', 'RED', 'BLU'])}"
                else:
                    sku = f"SKU-{random.randint(1000, 9999)}"
            record['product_sku'] = sku
        
        # Warehouse/location
        if random.random() > 0.7:  # 30% have warehouse info
            if i % 50 == 0:
                warehouse = None
            elif i % 60 == 0:
                warehouse = random.choice(['', 'N/A', 'TBD'])
            else:
                warehouse = random.choice([
                    'WH-EAST-01', 'WH-WEST-01', 'WH-CENTRAL-01', 'WH-NORTH-01',
                    'DROPSHIP', 'VENDOR-DIRECT', 'STORE-001', 'STORE-002'
                ])
            record['fulfillment_location'] = warehouse
        
        # Return status
        if random.random() > 0.85:  # 15% have return info
            if i % 55 == 0:
                return_status = None
            elif i % 65 == 0:
                return_status = random.choice(['', 'N/A'])
            else:
                return_status = random.choice([
                    'Not Returned', 'Returned', 'Return Pending', 'Return Approved',
                    'Refunded', 'Exchanged', 'Return Rejected'
                ])
            record['return_status'] = return_status
        
        # Margin amount (calculated field)
        if random.random() > 0.6:  # 40% have margin
            if all(isinstance(x, (int, float)) for x in [unit_price, product_cost, quantity]):
                margin = round(((unit_price - product_cost) * quantity), 2)
                # Business logic violation: negative margin flag
                if margin < 0 and random.random() < 0.2:  # 20% of negative margins are flagged
                    margin = f"LOSS: ${abs(margin)}"
            else:
                margin = None
            record['profit_margin'] = margin
        
        # Shipping weight
        if random.random() > 0.7:  # 30% have weight
            if i % 60 == 0:
                weight = None
            elif i % 70 == 0:  # String values
                weight = random.choice(['Light', 'Heavy', 'N/A'])
            elif i % 80 == 0:  # Negative weight
                weight = round(random.uniform(-5, -0.1), 2)
            else:
                if isinstance(quantity, (int, float)) and quantity > 0:
                    # Weight per item * quantity
                    item_weight = random.uniform(0.1, 10)
                    weight = round(item_weight * quantity, 2)
                else:
                    weight = round(random.uniform(0.1, 50), 2)
            record['total_weight_kg'] = weight
        
        # Gift wrap flag
        if random.random() > 0.9:  # 10% are gifts
            if i % 65 == 0:
                gift = None
            elif i % 75 == 0:  # Various boolean representations
                gift = random.choice(['Y', 'N', 'Yes', 'No', '1', '0'])
            else:
                gift = random.choice([True, False])
            record['is_gift'] = gift
        
        data.append(record)
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Add EXACT duplicate rows (every attribute is same)
    num_exact_duplicates = int(num_rows * 0.02)  # 2% exact duplicates
    for _ in range(num_exact_duplicates):
        if len(df) > 0:
            # Pick a random row to duplicate
            row_to_duplicate = df.sample(1)
            df = pd.concat([df, row_to_duplicate], ignore_index=True)
    
    # Add some completely empty rows
    for _ in range(int(num_rows * 0.005)):  # 0.5% empty rows
        empty_row = pd.Series([None] * len(df.columns), index=df.columns)
        df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)
    
    # Add some rows with all string 'NULL' or 'N/A' values
    for _ in range(int(num_rows * 0.01)):  # 1% NULL string rows
        null_values = ['NULL', 'N/A', 'null', 'NA', '', ' ']
        null_row = pd.Series([random.choice(null_values) for _ in range(len(df.columns))], index=df.columns)
        df = pd.concat([df, pd.DataFrame([null_row])], ignore_index=True)
    
    # Shuffle the dataframe to mix duplicates throughout
    df = df.sample(frac=1).reset_index(drop=True)
    
    return df

def add_more_messiness(df):
    """
    Add additional data quality issues to make the dataset more challenging
    """
    # Add trailing/leading spaces to some string columns
    string_cols = df.select_dtypes(include=['object']).columns
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05  # 5% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: '  ' + str(x) + '  ' if pd.notna(x) else x
        )
    
    # Add case inconsistencies
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03  # 3% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x).upper() if pd.notna(x) and random.random() > 0.5 else str(x).lower() if pd.notna(x) else x
        )
    
    # Add special characters to some values
    for col in string_cols[:3]:  # Only first 3 string columns
        mask = np.random.random(len(df)) < 0.02  # 2% of values
        special_chars = ['@', '#', '!', '*', '&', '%']
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )
    
    return df

# Generate the dataset
if __name__ == "__main__":
    # Set the number of rows you want
    NUM_ROWS = 5000  # Change this to your desired number
    
    # ID formats should match your orders and products datasets
    ORDER_ID_FORMAT = 'ORD'     # Options: 'ORD', 'ORDER', or 'NUMBER'
    PRODUCT_ID_FORMAT = 'PROD'   # Options: 'PROD', 'PRODUCT', or 'NUMBER'
    
    print(f"Generating {NUM_ROWS} rows of messy order items data...")
    df = generate_messy_order_items_data(NUM_ROWS, ORDER_ID_FORMAT, PRODUCT_ID_FORMAT)
    
    # Add more messiness
    df = add_more_messiness(df)
    
    # Display basic info
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumn names (realistic but challenging for mapping):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")
    
    print(f"\nFirst 10 rows:")
    print(df.head(10))
    
    # Show data quality issues summary
    print("\n" + "="*50)
    print("DATA QUALITY ISSUES SUMMARY:")
    print("="*50)
    print(f"Total null values: {df.isnull().sum().sum()}")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    print(f"Total rows: {len(df)}")
    
    # Check NOT NULL constraint violations
    print("\n" + "="*50)
    print("NOT NULL CONSTRAINT VIOLATIONS:")
    print("="*50)
    
    not_null_columns = ['order_ref', 'product_ref', 'qty_ordered', 'unit_selling_price', 'line_total']
    for col in not_null_columns:
        if col in df.columns:
            null_count = df[df[col].isnull()].shape[0]
            if null_count > 0:
                print(f"✗ NULL {col} (violates NOT NULL): {null_count} rows")
    
    # Show business logic violations
    print("\n" + "="*50)
    print("BUSINESS LOGIC VIOLATIONS EXAMPLES:")
    print("="*50)
    
    # Check for zero or negative quantities
    if 'qty_ordered' in df.columns:
        qty_numeric = pd.to_numeric(df['qty_ordered'], errors='coerce')
        invalid_qty = df[qty_numeric <= 0]
        if len(invalid_qty) > 0:
            print(f"✗ Zero or negative quantities: {len(invalid_qty)} items")
            print(f"  Examples: {invalid_qty['qty_ordered'].unique()[:5].tolist()}")
    
    # Check for negative prices
    if 'unit_selling_price' in df.columns:
        price_numeric = pd.to_numeric(df['unit_selling_price'], errors='coerce')
        negative_prices = df[price_numeric < 0]
        if len(negative_prices) > 0:
            print(f"✗ Negative unit prices: {len(negative_prices)} items")
    
    # Check for discount greater than total
    if 'item_discount' in df.columns and 'line_total' in df.columns:
        discount_numeric = pd.to_numeric(df['item_discount'], errors='coerce')
        total_numeric = pd.to_numeric(df['line_total'], errors='coerce')
        
        # Calculate what total should be before discount
        if 'qty_ordered' in df.columns and 'unit_selling_price' in df.columns:
            qty_numeric = pd.to_numeric(df['qty_ordered'], errors='coerce')
            price_numeric = pd.to_numeric(df['unit_selling_price'], errors='coerce')
            pre_discount_total = qty_numeric * price_numeric
            
            over_discount = df[discount_numeric > pre_discount_total]
            if len(over_discount) > 0:
                print(f"✗ Discount greater than pre-discount total: {len(over_discount)} items")
    
    # Check for incorrect total calculation
    if all(col in df.columns for col in ['qty_ordered', 'unit_selling_price', 'item_discount', 'line_total']):
        qty = pd.to_numeric(df['qty_ordered'], errors='coerce').fillna(0)
        price = pd.to_numeric(df['unit_selling_price'], errors='coerce').fillna(0)
        discount = pd.to_numeric(df['item_discount'], errors='coerce').fillna(0)
        total = pd.to_numeric(df['line_total'], errors='coerce').fillna(0)
        
        calculated_total = (qty * price) - discount
        wrong_total = df[abs(total - calculated_total) > 0.01]  # Allow 1 cent rounding difference
        if len(wrong_total) > 0:
            print(f"✗ Incorrect total calculation: {len(wrong_total)} items")
    
    # Check for cost greater than price (negative margin)
    if 'unit_selling_price' in df.columns and 'unit_cost' in df.columns:
        price_numeric = pd.to_numeric(df['unit_selling_price'], errors='coerce')
        cost_numeric = pd.to_numeric(df['unit_cost'], errors='coerce')
        negative_margin = df[(cost_numeric > price_numeric) & (price_numeric > 0)]
        if len(negative_margin) > 0:
            print(f"✗ Negative margin (cost > price): {len(negative_margin)} items")
            margin_loss = ((cost_numeric - price_numeric) * pd.to_numeric(df['qty_ordered'], errors='coerce')).sum()
            print(f"  Total margin loss: ${margin_loss:,.2f}")
    
    # Check for duplicate products in same order
    if 'order_ref' in df.columns and 'product_ref' in df.columns:
        duplicate_products = df[df.duplicated(subset=['order_ref', 'product_ref'], keep=False)]
        duplicate_products = duplicate_products[
            duplicate_products['order_ref'].notna() & 
            duplicate_products['product_ref'].notna()
        ]
        if len(duplicate_products) > 0:
            print(f"✗ Duplicate products in same order: {len(duplicate_products)} items")
            sample_dupes = duplicate_products.groupby(['order_ref', 'product_ref']).size().head(3)
            print(f"  Examples (order, product): {sample_dupes.to_dict()}")
    
    # Check for invalid foreign keys
    if 'order_ref' in df.columns:
        invalid_orders = df[df['order_ref'].isin(['INVALID', 'NULL', 'N/A', '', 'MISSING'])]
        if len(invalid_orders) > 0:
            print(f"✗ Invalid order references: {len(invalid_orders)} rows")
    
    if 'product_ref' in df.columns:
        invalid_products = df[df['product_ref'].isin(['INVALID', 'N/A', 'NULL', '', 'DISCONTINUED'])]
        if len(invalid_products) > 0:
            print(f"✗ Invalid product references: {len(invalid_products)} rows")
    
    # Show order statistics
    print("\n" + "="*50)
    print("ORDER STATISTICS:")
    print("="*50)
    
    if 'order_ref' in df.columns:
        # Items per order
        items_per_order = df.groupby('order_ref')['line_item_id'].count()
        print(f"Items per order:")
        print(f"  Average: {items_per_order.mean():.1f}")
        print(f"  Median: {items_per_order.median():.0f}")
        print(f"  Max: {items_per_order.max()}")
        print(f"  Orders with 1 item: {(items_per_order == 1).sum()} ({(items_per_order == 1).sum()/len(items_per_order)*100:.1f}%)")
        
        # Order values
        if 'line_total' in df.columns:
            order_values = df.groupby('order_ref')['line_total'].apply(
                lambda x: pd.to_numeric(x, errors='coerce').sum()
            )
            valid_values = order_values[order_values > 0]
            if len(valid_values) > 0:
                print(f"\nOrder values:")
                print(f"  Average order value: ${valid_values.mean():.2f}")
                print(f"  Median order value: ${valid_values.median():.2f}")
                print(f"  Max order value: ${valid_values.max():.2f}")
    
    # Show product statistics
    print("\n" + "="*50)
    print("PRODUCT STATISTICS:")
    print("="*50)
    
    if 'product_ref' in df.columns:
        # Top selling products
        product_sales = df.groupby('product_ref').agg({
            'qty_ordered': lambda x: pd.to_numeric(x, errors='coerce').sum(),
            'line_total': lambda x: pd.to_numeric(x, errors='coerce').sum()
        })
        
        # Filter out invalid products
        valid_products = product_sales[
            ~product_sales.index.isin(['INVALID', 'N/A', 'NULL', '', 'DISCONTINUED'])
        ]
        
        if len(valid_products) > 0:
            top_products_qty = valid_products.nlargest(5, 'qty_ordered')
            print(f"Top 5 products by quantity:")
            for idx, row in top_products_qty.iterrows():
                print(f"  {idx}: {row['qty_ordered']:.0f} units, ${row['line_total']:,.2f}")
            
            top_products_revenue = valid_products.nlargest(5, 'line_total')
            print(f"\nTop 5 products by revenue:")
            for idx, row in top_products_revenue.iterrows():
                print(f"  {idx}: ${row['line_total']:,.2f} ({row['qty_ordered']:.0f} units)")
    
    # Show margin analysis
    print("\n" + "="*50)
    print("MARGIN ANALYSIS:")
    print("="*50)
    
    if 'unit_selling_price' in df.columns and 'unit_cost' in df.columns and 'qty_ordered' in df.columns:
        price = pd.to_numeric(df['unit_selling_price'], errors='coerce')
        cost = pd.to_numeric(df['unit_cost'], errors='coerce')
        qty = pd.to_numeric(df['qty_ordered'], errors='coerce')
        
        # Calculate margins
        revenue = (price * qty).sum()
        total_cost = (cost * qty).sum()
        gross_margin = revenue - total_cost
        
        if revenue > 0:
            margin_pct = (gross_margin / revenue) * 100
            print(f"Overall margin metrics:")
            print(f"  Total revenue: ${revenue:,.2f}")
            print(f"  Total cost: ${total_cost:,.2f}")
            print(f"  Gross margin: ${gross_margin:,.2f}")
            print(f"  Margin %: {margin_pct:.1f}%")
            
            # Products with negative margin
            item_margin = (price - cost) * qty
            negative_margin_items = df[item_margin < 0]
            if len(negative_margin_items) > 0:
                print(f"  Items with negative margin: {len(negative_margin_items)}")
                total_loss = abs(item_margin[item_margin < 0].sum())
                print(f"  Total loss from negative margins: ${total_loss:,.2f}")
    
    # Show main columns analysis
    print("\n" + "="*50)
    print("MAIN COLUMNS ANALYSIS:")
    print("="*50)
    
    main_columns = [
        'line_item_id', 'order_ref', 'product_ref', 'qty_ordered',
        'unit_selling_price', 'line_total', 'item_discount', 'unit_cost'
    ]
    
    for col in main_columns:
        if col in df.columns:
            null_count = df[col].isnull().sum()
            unique_count = df[col].nunique()
            print(f"\n{col}:")
            print(f"  - Null values: {null_count} ({null_count/len(df)*100:.1f}%)")
            print(f"  - Unique values: {unique_count}")
            if col in ['qty_ordered'] and unique_count > 0:
                qty_dist = df[col].value_counts().head(5)
                print(f"  - Top quantities: {qty_dist.to_dict()}")
    
    # Save to CSV
    output_file = 'messy_order_items_data.xlsx'
    df.to_excel(output_file, index=False)
    print(f"\n✅ Dataset saved to '{output_file}'")
    
    # Save order items summary
    order_summary = []
    
    if 'order_ref' in df.columns and 'line_total' in df.columns:
        for order_id in df['order_ref'].dropna().unique()[:100]:  # Top 100 orders
            order_items = df[df['order_ref'] == order_id]
            total_items = len(order_items)
            total_value = pd.to_numeric(order_items['line_total'], errors='coerce').sum()
            unique_products = order_items['product_ref'].nunique() if 'product_ref' in df.columns else 0
            
            order_summary.append({
                'order_id': order_id,
                'total_items': total_items,
                'unique_products': unique_products,
                'order_value': round(total_value, 2),
                'avg_item_value': round(total_value/total_items, 2) if total_items > 0 else 0
            })
    
    if order_summary:
        summary_df = pd.DataFrame(order_summary)
        summary_df.to_csv('order_items_summary.csv', index=False)
        print(f"✅ Order items summary saved to 'order_items_summary.csv'")

Generating 5000 rows of messy order items data...

Dataset shape: (5175, 16)

Column names (realistic but challenging for mapping):
  1. line_item_id
  2. order_ref
  3. product_ref
  4. qty_ordered
  5. unit_selling_price
  6. item_discount
  7. line_total
  8. unit_cost
  9. created_timestamp
  10. product_sku
  11. total_weight_kg
  12. is_gift
  13. tax_amount
  14. profit_margin
  15. fulfillment_location
  16. return_status

First 10 rows:
  line_item_id    order_ref product_ref qty_ordered unit_selling_price  \
0       102777   ORD_100833   PROD_0237           3              34.45   
1       101421   ORD_100001   PROD_0053         100             400.55   
2      101346#   ORD_100169   PROD_0085           1             280.14   
3       104741   ORD_101425   PROD_0045           2             386.82   
4       101793   ORD_101215   PROD_0195           1             192.35   
5       101271   ORD_100794   PROD_0436          20             326.62   
6       102387   ORD_101324   PR

### Payments Table Generator

In [8]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import string

# Initialize Faker
fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)


def generate_messy_payments_data(num_rows=3500, order_id_format="ORD"):
    """
    Generate a messy, realistic payments dataset with realistic but challenging column names
    and various data quality issues for testing data mapping and cleaning.

    Args:
        num_rows: Number of payment records to generate
        order_id_format: Format of order IDs ('ORD', 'ORDER', or 'NUMBER')
    """

    data = []

    # Track IDs for creating duplicates and relationships
    used_payment_ids = []
    used_transaction_ids = []

    # Generate pool of order IDs
    num_orders = max(num_rows // 2, 500)  # Some orders have multiple payment attempts

    order_ids = []
    for i in range(num_orders):
        if order_id_format == "ORD":
            order_id = f"ORD_{str(i + 100000).zfill(6)}"
        elif order_id_format == "ORDER":
            order_id = f"ORDER-{str(i + 100000).zfill(6)}"
        else:  # NUMBER
            order_id = str(1000000 + i)
        order_ids.append(order_id)

    # Choose one consistent payment ID format
    payment_id_format_choice = random.choice(["PAY", "PAYMENT", "NUMBER"])

    # Payment methods and providers mapping
    payment_methods = {
        "Credit Card": ["Stripe", "Square", "Authorize.net", "Braintree"],
        "Debit Card": ["Stripe", "Square", "Authorize.net"],
        "PayPal": ["PayPal"],
        "Apple Pay": ["Stripe", "Square"],
        "Google Pay": ["Stripe", "Google"],
        "Bank Transfer": ["Plaid", "ACH", "Wire"],
        "Cryptocurrency": ["Coinbase", "BitPay"],
        "Buy Now Pay Later": ["Klarna", "Afterpay", "Affirm"],
        "Gift Card": ["Internal", "Store Credit"],
        "Cash on Delivery": ["COD", "Manual"],
    }

    # Track order payment status for business logic
    order_payment_status = {}

    # Generate order amounts for consistency
    order_amounts = {}
    for order_id in order_ids:
        order_amounts[order_id] = round(random.uniform(10, 2000), 2)

    for i in range(num_rows):
        record = {}

        # Payment ID with CONSISTENT format
        if i % 50 == 0 and used_payment_ids:  # 2% duplicates
            payment_id = random.choice(used_payment_ids)
        else:
            if payment_id_format_choice == "PAY":
                payment_id = f"PAY{str(i + 1).zfill(6)}"
            elif payment_id_format_choice == "PAYMENT":
                payment_id = f"PAYMENT-{str(i + 1).zfill(6)}"
            else:  # NUMBER
                payment_id = str(1000000 + i)

            used_payment_ids.append(payment_id)

        record["payment_ref"] = payment_id if i % 100 != 0 else None  # 1% null IDs

        # Order ID - NOT NULL in schema
        if i % 90 == 0:  # Violates NOT NULL
            order_id = None
        elif i % 60 == 0:  # Non-existent order IDs (FK violation)
            if order_id_format == "ORD":
                order_id = f"ORD_{str(999999).zfill(6)}"
            elif order_id_format == "ORDER":
                order_id = f"ORDER-{str(999999).zfill(6)}"
            else:
                order_id = "9999999"
        elif i % 40 == 0:  # Invalid format
            order_id = random.choice(["INVALID", "NULL", "N/A", "", "MISSING"])
        else:
            # Some orders have multiple payment attempts (failed then successful)
            if random.random() < 0.2 and order_payment_status:  # 20% are retry payments
                # Pick an order that had a failed payment
                failed_orders = [
                    o for o, s in order_payment_status.items() if s == "Failed"
                ]
                if failed_orders:
                    order_id = random.choice(failed_orders)
                else:
                    order_id = random.choice(order_ids)
            else:
                order_id = random.choice(order_ids)

        record["order_ref"] = order_id

        # Payment Method
        if i % 25 == 0:
            method = None
        elif i % 35 == 0:  # Inconsistent values
            method = random.choice(
                ["CC", "credit card", "CREDIT_CARD", "Card", "Visa", "Mastercard"]
            )
        elif i % 45 == 0:  # Invalid values
            method = random.choice(["Unknown", "N/A", "Cash", "Check", ""])
        elif i % 55 == 0:  # Typos
            method = random.choice(
                ["Credt Card", "PayPall", "Banck Transfer", "Appel Pay"]
            )
        else:
            method = random.choice(list(payment_methods.keys()))

        record["payment_type"] = method

        # Payment Provider (should match payment method)
        if i % 30 == 0:
            provider = None
        elif i % 40 == 0:  # Mismatched provider (business logic violation)
            if method == "Credit Card":
                provider = "PayPal"  # Wrong provider for method
            elif method == "PayPal":
                provider = "Stripe"  # Wrong provider for method
            else:
                provider = "Unknown"
        elif i % 50 == 0:  # Invalid values
            provider = random.choice(["N/A", "NULL", "Internal", ""])
        elif i % 60 == 0:  # Typos
            provider = random.choice(["Strpe", "Sqaure", "PayPl", "Klarrna"])
        else:
            if method in payment_methods:
                provider = random.choice(payment_methods[method])
            else:
                provider = random.choice(["Stripe", "PayPal", "Square"])

        record["gateway_provider"] = provider

        # Payment Status
        if i % 20 == 0:
            status = None
        elif i % 30 == 0:  # Inconsistent values
            status = random.choice(
                ["completed", "COMPLETED", "Complete", "1", "Success", "SUCCESS"]
            )
        elif i % 40 == 0:  # Invalid values
            status = random.choice(
                ["In Progress", "Processing", "Approved", "Declined"]
            )
        elif i % 50 == 0:  # Typos
            status = random.choice(["Complted", "Pendng", "Faild", "Refnded"])
        else:
            # Realistic status distribution
            status = random.choices(
                [
                    "Completed",
                    "Pending",
                    "Failed",
                    "Refunded",
                    "Cancelled",
                    "Partially Refunded",
                ],
                weights=[60, 10, 10, 10, 5, 5],
                k=1,
            )[0]

        # Track order payment status
        if order_id:
            order_payment_status[order_id] = status

        record["payment_status"] = status

        # Payment Date
        if i % 25 == 0:
            payment_date = None
        elif i % 35 == 0:  # String format variations
            payment_date_dt = fake.date_time_between(start_date="-1y", end_date="now")
            formats = ["%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%d-%m-%Y", "%Y%m%d"]
            payment_date = payment_date_dt.strftime(random.choice(formats))
        elif i % 45 == 0:  # Unix timestamp
            payment_date = int(
                fake.date_time_between(start_date="-1y", end_date="now").timestamp()
            )
        elif i % 55 == 0:  # Future date (business logic violation)
            payment_date = fake.date_time_between(start_date="+1d", end_date="+30d")
        elif i % 65 == 0:  # Very old payment
            payment_date = fake.date_time_between(start_date="-5y", end_date="-2y")
        else:
            payment_date = fake.date_time_between(start_date="-1y", end_date="now")

        record["transaction_date"] = payment_date

        # Amount - NOT NULL in schema
        if order_id in order_amounts:
            expected_amount = order_amounts[order_id]
        else:
            expected_amount = round(random.uniform(10, 2000), 2)

        if i % 95 == 0:  # Violates NOT NULL
            amount = None
        elif i % 35 == 0:  # String values
            amount = random.choice(["Free", "N/A", "Pending", "", "TBD"])
        elif i % 45 == 0:  # Negative amount (business logic violation)
            amount = round(random.uniform(-500, -10), 2)
        elif i % 55 == 0:  # Zero amount
            amount = 0
        elif i % 65 == 0:  # Amount doesn't match order (business logic violation)
            amount = round(expected_amount * random.uniform(0.5, 1.5), 2)
        elif i % 75 == 0:  # Extreme amounts
            amount = random.choice([999999.99, 0.01, -9999])
        elif i % 30 == 0:  # Partial payment
            amount = round(expected_amount * random.uniform(0.3, 0.9), 2)
        else:
            amount = expected_amount

        record["payment_amount"] = amount

        # Transaction ID (from payment provider)
        if i % 30 == 0:
            trans_id = None
        elif i % 40 == 0:  # Invalid format
            trans_id = random.choice(["N/A", "NULL", "PENDING", ""])
        elif (
            i % 50 == 0 and used_transaction_ids
        ):  # Duplicate transaction ID (business logic violation)
            trans_id = random.choice(used_transaction_ids)
        else:
            # Generate provider-specific transaction ID
            if provider == "Stripe":
                trans_id = f"pi_{fake.uuid4()[:24]}"
            elif provider == "PayPal":
                trans_id = f"PP-{fake.uuid4()[:20].upper()}"
            elif provider == "Square":
                trans_id = f"sq_{fake.uuid4()[:22]}"
            else:
                trans_id = f"TXN-{fake.uuid4()[:16].upper()}"
            used_transaction_ids.append(trans_id)

        record["gateway_transaction_id"] = trans_id

        # Processing Fee
        if i % 25 == 0:
            fee = None
        elif i % 35 == 0:  # String values
            fee = random.choice(["Included", "N/A", "Waived"])
        elif i % 45 == 0:  # Negative fee (credit?)
            fee = round(random.uniform(-10, -1), 2)
        elif i % 55 == 0:  # Fee greater than payment (business logic violation)
            if isinstance(amount, (int, float)) and amount > 0:
                fee = amount * 1.2
            else:
                fee = 999
        elif i % 65 == 0:  # Zero fee
            fee = 0
        else:
            # Typical processing fees (2-3% + fixed fee)
            if isinstance(amount, (int, float)) and amount > 0:
                if method in ["Credit Card", "Debit Card"]:
                    fee = round(amount * 0.029 + 0.30, 2)  # 2.9% + $0.30
                elif method == "PayPal":
                    fee = round(amount * 0.0349 + 0.49, 2)  # 3.49% + $0.49
                elif method == "Bank Transfer":
                    fee = round(amount * 0.008, 2)  # 0.8%
                else:
                    fee = round(amount * 0.025, 2)  # 2.5%
            else:
                fee = 0

        record["transaction_fee"] = fee

        # Refund Amount (only for refunded/partially refunded status)
        refund_amount = 0
        refund_date = None

        if status in ["Refunded", "Partially Refunded", "refunded", "Refnded"]:
            if i % 60 == 0:  # String values
                refund_amount = random.choice(["Full", "Partial", "Pending"])
            elif i % 70 == 0:  # Refund greater than payment (business logic violation)
                if isinstance(amount, (int, float)) and amount > 0:
                    refund_amount = amount * 1.5
                else:
                    refund_amount = 1000
            elif i % 80 == 0:  # Negative refund
                refund_amount = round(random.uniform(-100, -10), 2)
            else:
                if isinstance(amount, (int, float)) and amount > 0:
                    if status in ["Refunded", "refunded"]:
                        refund_amount = amount  # Full refund
                    else:  # Partial refund
                        refund_amount = round(amount * random.uniform(0.1, 0.9), 2)
                else:
                    refund_amount = round(random.uniform(10, 500), 2)

            # Refund Date (should be after payment date)
            if i % 50 == 0:  # Business logic violation: refund before payment
                if isinstance(payment_date, datetime):
                    refund_date = payment_date - timedelta(days=random.randint(1, 10))
                else:
                    refund_date = fake.date_time_between(
                        start_date="-2y", end_date="-1y"
                    )
            elif i % 60 == 0:  # String format
                refund_date = fake.date_between(start_date="-6m", end_date="today")
            elif i % 70 == 0:  # Future refund
                refund_date = fake.date_time_between(start_date="+1d", end_date="+30d")
            else:
                if isinstance(payment_date, datetime):
                    refund_date = fake.date_time_between(
                        start_date=payment_date,
                        end_date=min(payment_date + timedelta(days=90), datetime.now()),
                    )
                else:
                    refund_date = fake.date_time_between(
                        start_date="-6m", end_date="now"
                    )
        elif (
            i % 85 == 0
        ):  # Business logic violation: not refunded but has refund amount
            refund_amount = round(random.uniform(10, 500), 2)
            refund_date = fake.date_time_between(start_date="-6m", end_date="now")

        record["refund_total"] = refund_amount
        record["refund_processed_date"] = refund_date

        # Created At timestamp
        if i % 30 == 0:
            created = None
        elif i % 40 == 0:  # String format
            created = fake.date_time_between(start_date="-1y", end_date="now").strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        elif i % 50 == 0:  # Created before payment (business logic violation)
            if isinstance(payment_date, datetime):
                created = payment_date - timedelta(days=random.randint(1, 30))
            else:
                created = fake.date_time_between(start_date="-2y", end_date="-1y")
        else:
            if isinstance(payment_date, datetime):
                created = payment_date + timedelta(minutes=random.randint(0, 60))
            else:
                created = fake.date_time_between(start_date="-1y", end_date="now")

        record["record_created"] = created

        # Additional realistic columns that might exist

        # Currency
        if random.random() > 0.6:  # 40% have currency
            if i % 45 == 0:
                currency = None
            elif i % 55 == 0:  # Invalid codes
                currency = random.choice(["US", "EURO", "Dollar", "$"])
            else:
                currency = random.choices(
                    ["USD", "EUR", "GBP", "CAD", "AUD"], weights=[70, 15, 5, 5, 5], k=1
                )[0]
            record["currency_code"] = currency

        # Card last 4 digits (for card payments)
        if method in ["Credit Card", "Debit Card", "CC", "credit card"]:
            if i % 50 == 0:
                card_last4 = None
            elif i % 60 == 0:  # Invalid format
                card_last4 = random.choice(["XXXX", "N/A", "****"])
            elif i % 70 == 0:  # Wrong length
                card_last4 = random.choice(["123", "12345", "1"])
            else:
                card_last4 = str(random.randint(1000, 9999))
            record["card_last_four"] = card_last4

        # Card brand
        if method in ["Credit Card", "Debit Card", "CC", "credit card"]:
            if i % 55 == 0:
                brand = None
            elif i % 65 == 0:
                brand = random.choice(["Unknown", "N/A", "Card"])
            else:
                brand = random.choice(
                    ["Visa", "Mastercard", "Amex", "Discover", "JCB", "Diners"]
                )
            record["card_brand"] = brand

        # Authorization code
        if status in ["Completed", "completed", "COMPLETED"]:
            if i % 60 == 0:
                auth_code = None
            elif i % 70 == 0:
                auth_code = random.choice(["N/A", "PENDING", ""])
            else:
                auth_code = f"AUTH-{fake.uuid4()[:12].upper()}"
            record["authorization_code"] = auth_code

        # Risk score (fraud detection)
        if random.random() > 0.7:  # 30% have risk score
            if i % 65 == 0:
                risk = None
            elif i % 75 == 0:  # String values
                risk = random.choice(["Low", "Medium", "High", "N/A"])
            elif i % 85 == 0:  # Out of range
                risk = random.choice([-10, 150, 999])
            else:
                # Higher risk for failed payments
                if status in ["Failed", "failed"]:
                    risk = random.randint(60, 99)
                else:
                    risk = random.randint(1, 50)
            record["risk_score"] = risk

        # Customer IP
        if random.random() > 0.6:  # 40% have IP
            if i % 70 == 0:
                ip = None
            elif i % 80 == 0:  # Invalid IP
                ip = random.choice(["N/A", "0.0.0.0", "999.999.999.999"])
            else:
                ip = fake.ipv4()
            record["customer_ip"] = ip

        # Billing country
        if random.random() > 0.5:  # 50% have billing country
            if i % 75 == 0:
                country = None
            elif i % 85 == 0:  # Invalid codes
                country = random.choice(["USA", "United States", "N/A"])
            else:
                country = fake.country_code()
            record["billing_country"] = country

        # Retry attempt number
        if order_id in order_payment_status and random.random() > 0.7:
            # Count previous attempts for this order
            retry_count = len([1 for o in data if o.get("order_ref") == order_id])
            if retry_count > 0:
                record["retry_attempt"] = retry_count

        data.append(record)

    # Create DataFrame
    df = pd.DataFrame(data)

    # Add EXACT duplicate rows (every attribute is same)
    num_exact_duplicates = int(num_rows * 0.02)  # 2% exact duplicates
    for _ in range(num_exact_duplicates):
        if len(df) > 0:
            # Pick a random row to duplicate
            row_to_duplicate = df.sample(1)
            df = pd.concat([df, row_to_duplicate], ignore_index=True)

    # Add some completely empty rows
    for _ in range(int(num_rows * 0.005)):  # 0.5% empty rows
        empty_row = pd.Series([None] * len(df.columns), index=df.columns)
        df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)

    # Add some rows with all string 'NULL' or 'N/A' values
    for _ in range(int(num_rows * 0.01)):  # 1% NULL string rows
        null_values = ["NULL", "N/A", "null", "NA", "", " "]
        null_row = pd.Series(
            [random.choice(null_values) for _ in range(len(df.columns))],
            index=df.columns,
        )
        df = pd.concat([df, pd.DataFrame([null_row])], ignore_index=True)

    # Shuffle the dataframe to mix duplicates throughout
    df = df.sample(frac=1).reset_index(drop=True)

    return df


def add_more_messiness(df):
    """
    Add additional data quality issues to make the dataset more challenging
    """
    # Add trailing/leading spaces to some string columns
    string_cols = df.select_dtypes(include=["object"]).columns
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05  # 5% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    # Add case inconsistencies
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03  # 3% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    # Add special characters to some values
    for col in string_cols[:3]:  # Only first 3 string columns
        mask = np.random.random(len(df)) < 0.02  # 2% of values
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


# Generate the dataset
if __name__ == "__main__":
    # Set the number of rows you want
    NUM_ROWS = 3500  # Change this to your desired number

    # ID format should match your orders dataset
    ORDER_ID_FORMAT = "ORD"  # Options: 'ORD', 'ORDER', or 'NUMBER'

    print(f"Generating {NUM_ROWS} rows of messy payments data...")
    df = generate_messy_payments_data(NUM_ROWS, ORDER_ID_FORMAT)

    # Add more messiness
    df = add_more_messiness(df)

    # Display basic info
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumn names (realistic but challenging for mapping):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")

    print(f"\nFirst 10 rows:")
    print(df.head(10))

    # Show data quality issues summary
    print("\n" + "=" * 50)
    print("DATA QUALITY ISSUES SUMMARY:")
    print("=" * 50)
    print(f"Total null values: {df.isnull().sum().sum()}")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    print(f"Total rows: {len(df)}")

    # Check NOT NULL constraint violations
    print("\n" + "=" * 50)
    print("NOT NULL CONSTRAINT VIOLATIONS:")
    print("=" * 50)

    if "order_ref" in df.columns:
        null_orders = df[df["order_ref"].isnull()].shape[0]
        if null_orders > 0:
            print(f"✗ NULL order_ref (violates NOT NULL): {null_orders} rows")

    if "payment_amount" in df.columns:
        null_amounts = df[df["payment_amount"].isnull()].shape[0]
        if null_amounts > 0:
            print(f"✗ NULL payment_amount (violates NOT NULL): {null_amounts} rows")

    # Show business logic violations
    print("\n" + "=" * 50)
    print("BUSINESS LOGIC VIOLATIONS EXAMPLES:")
    print("=" * 50)

    # Check for negative amounts
    if "payment_amount" in df.columns:
        amount_numeric = pd.to_numeric(df["payment_amount"], errors="coerce")
        negative_amounts = df[amount_numeric < 0]
        if len(negative_amounts) > 0:
            print(f"✗ Negative payment amounts: {len(negative_amounts)} payments")

    # Check for processing fee greater than payment
    if "payment_amount" in df.columns and "transaction_fee" in df.columns:
        amount_numeric = pd.to_numeric(df["payment_amount"], errors="coerce")
        fee_numeric = pd.to_numeric(df["transaction_fee"], errors="coerce")
        over_fee = df[fee_numeric > amount_numeric]
        if len(over_fee) > 0:
            print(f"✗ Processing fee greater than payment: {len(over_fee)} payments")

    # Check for refund greater than payment
    if "payment_amount" in df.columns and "refund_total" in df.columns:
        amount_numeric = pd.to_numeric(df["payment_amount"], errors="coerce")
        refund_numeric = pd.to_numeric(df["refund_total"], errors="coerce")
        over_refund = df[refund_numeric > amount_numeric]
        if len(over_refund) > 0:
            print(f"✗ Refund greater than payment: {len(over_refund)} payments")

    # Check for refunds without refunded status
    if "payment_status" in df.columns and "refund_total" in df.columns:
        refund_numeric = pd.to_numeric(df["refund_total"], errors="coerce")
        wrong_status_refund = df[
            (refund_numeric > 0)
            & (
                ~df["payment_status"].isin(
                    ["Refunded", "Partially Refunded", "refunded"]
                )
            )
        ]
        if len(wrong_status_refund) > 0:
            print(
                f"✗ Has refund but status not 'Refunded': {len(wrong_status_refund)} payments"
            )

    # Check for refund date before payment date
    if "transaction_date" in df.columns and "refund_processed_date" in df.columns:
        payment_dt = pd.to_datetime(df["transaction_date"], errors="coerce")
        refund_dt = pd.to_datetime(df["refund_processed_date"], errors="coerce")
        refund_before_payment = df[refund_dt < payment_dt]
        if len(refund_before_payment) > 0:
            print(f"✗ Refund before payment: {len(refund_before_payment)} payments")

    # Check for duplicate transaction IDs
    if "gateway_transaction_id" in df.columns:
        trans_duplicates = df[
            df.duplicated(subset=["gateway_transaction_id"], keep=False)
            & df["gateway_transaction_id"].notna()
            & (~df["gateway_transaction_id"].isin(["N/A", "NULL", "PENDING", ""]))
        ]
        if len(trans_duplicates) > 0:
            print(f"✗ Duplicate transaction IDs: {len(trans_duplicates)} payments")

    # Check for provider-method mismatches
    if "payment_type" in df.columns and "gateway_provider" in df.columns:
        paypal_wrong = df[
            (df["payment_type"] == "PayPal")
            & (df["gateway_provider"].isin(["Stripe", "Square", "Authorize.net"]))
        ]
        if len(paypal_wrong) > 0:
            print(
                f"✗ PayPal payments with wrong provider: {len(paypal_wrong)} payments"
            )

    # Check for invalid foreign keys
    if "order_ref" in df.columns:
        invalid_orders = df[
            df["order_ref"].isin(["INVALID", "NULL", "N/A", "", "MISSING"])
        ]
        if len(invalid_orders) > 0:
            print(f"✗ Invalid order references: {len(invalid_orders)} rows")

    # Show payment statistics
    print("\n" + "=" * 50)
    print("PAYMENT STATISTICS:")
    print("=" * 50)

    if "payment_status" in df.columns:
        status_counts = df["payment_status"].value_counts()
        print(f"Payment status distribution:")
        for status, count in status_counts.head(10).items():
            print(f"  {status}: {count} ({count/len(df)*100:.1f}%)")

    # Calculate success rate
    if "payment_status" in df.columns:
        completed = (
            df["payment_status"].isin(["Completed", "completed", "COMPLETED"]).sum()
        )
        failed = df["payment_status"].isin(["Failed", "failed", "Faild"]).sum()
        if (completed + failed) > 0:
            success_rate = completed / (completed + failed) * 100
            print(f"\nPayment success rate: {success_rate:.1f}%")

    # Revenue analysis
    if "payment_amount" in df.columns and "payment_status" in df.columns:
        amount_numeric = pd.to_numeric(df["payment_amount"], errors="coerce")
        completed_payments = df[
            df["payment_status"].isin(["Completed", "completed", "COMPLETED"])
        ]
        completed_amount = pd.to_numeric(
            completed_payments["payment_amount"], errors="coerce"
        )

        if len(completed_amount) > 0:
            print(f"\nRevenue metrics (completed payments):")
            print(f"  Total revenue: ${completed_amount.sum():,.2f}")
            print(f"  Average payment: ${completed_amount.mean():.2f}")
            print(f"  Median payment: ${completed_amount.median():.2f}")

    # Processing fees analysis
    if "transaction_fee" in df.columns:
        fee_numeric = pd.to_numeric(df["transaction_fee"], errors="coerce")
        total_fees = fee_numeric[fee_numeric > 0].sum()
        if total_fees > 0:
            print(f"\nProcessing fees:")
            print(f"  Total fees: ${total_fees:,.2f}")
            print(f"  Average fee: ${fee_numeric[fee_numeric > 0].mean():.2f}")

            if "payment_amount" in df.columns:
                amount_numeric = pd.to_numeric(df["payment_amount"], errors="coerce")
                avg_fee_rate = (fee_numeric / amount_numeric * 100).mean()
                print(f"  Average fee rate: {avg_fee_rate:.2f}%")

    # Refund analysis
    if "refund_total" in df.columns:
        refund_numeric = pd.to_numeric(df["refund_total"], errors="coerce")
        refunded_payments = df[refund_numeric > 0]
        if len(refunded_payments) > 0:
            total_refunds = refund_numeric[refund_numeric > 0].sum()
            print(f"\nRefund metrics:")
            print(f"  Total refunded: ${total_refunds:,.2f}")
            print(f"  Refund rate: {len(refunded_payments)/len(df)*100:.1f}%")
            print(f"  Average refund: ${refund_numeric[refund_numeric > 0].mean():.2f}")

    # Payment method analysis
    print("\n" + "=" * 50)
    print("PAYMENT METHOD ANALYSIS:")
    print("=" * 50)

    if "payment_type" in df.columns:
        method_stats = (
            df.groupby("payment_type")
            .agg(
                {
                    "payment_ref": "count",
                    "payment_amount": lambda x: pd.to_numeric(x, errors="coerce").sum(),
                }
            )
            .rename(
                columns={
                    "payment_ref": "payment_count",
                    "payment_amount": "total_amount",
                }
            )
        )

        # Filter out invalid methods
        valid_methods = method_stats[
            ~method_stats.index.isin(["Unknown", "N/A", "Cash", "Check", ""])
        ]

        if len(valid_methods) > 0:
            top_methods = valid_methods.nlargest(5, "total_amount")
            print(f"Top 5 payment methods by volume:")
            for idx, row in top_methods.iterrows():
                if idx:
                    print(
                        f"  {idx}: {row['payment_count']} payments, ${row['total_amount']:,.2f}"
                    )

    # Provider analysis
    if "gateway_provider" in df.columns:
        provider_counts = df["gateway_provider"].value_counts()
        print(f"\nPayment provider distribution:")
        for provider, count in provider_counts.head(5).items():
            if provider and provider not in ["N/A", "NULL", "Internal", ""]:
                print(f"  {provider}: {count} ({count/len(df)*100:.1f}%)")

    # Multiple payment attempts analysis
    print("\n" + "=" * 50)
    print("PAYMENT RETRY ANALYSIS:")
    print("=" * 50)

    if "order_ref" in df.columns:
        payments_per_order = df.groupby("order_ref")["payment_ref"].count()
        multiple_attempts = payments_per_order[payments_per_order > 1]
        if len(multiple_attempts) > 0:
            print(f"Orders with multiple payment attempts: {len(multiple_attempts)}")
            print(f"  Max attempts for single order: {multiple_attempts.max()}")
            print(
                f"  Average attempts (for retried orders): {multiple_attempts.mean():.1f}"
            )

    # Show main columns analysis
    print("\n" + "=" * 50)
    print("MAIN COLUMNS ANALYSIS:")
    print("=" * 50)

    main_columns = [
        "payment_ref",
        "order_ref",
        "payment_type",
        "gateway_provider",
        "payment_status",
        "transaction_date",
        "payment_amount",
        "gateway_transaction_id",
        "transaction_fee",
        "refund_total",
    ]

    for col in main_columns:
        if col in df.columns:
            null_count = df[col].isnull().sum()
            unique_count = df[col].nunique()
            print(f"\n{col}:")
            print(f"  - Null values: {null_count} ({null_count/len(df)*100:.1f}%)")
            print(f"  - Unique values: {unique_count}")

    # Save to CSV
    output_file = "messy_payments_data.xlsx"
    df.to_excel(output_file, index=False)
    print(f"\n✅ Dataset saved to '{output_file}'")

    # Save payment summary by method and status
    payment_summary = []

    if "payment_type" in df.columns and "payment_status" in df.columns:
        for method in df["payment_type"].dropna().unique()[:10]:
            method_payments = df[df["payment_type"] == method]

            completed = (
                method_payments["payment_status"].isin(["Completed", "completed"]).sum()
            )
            failed = method_payments["payment_status"].isin(["Failed", "failed"]).sum()
            refunded = (
                method_payments["payment_status"].isin(["Refunded", "refunded"]).sum()
            )

            total_amount = pd.to_numeric(
                method_payments["payment_amount"], errors="coerce"
            ).sum()

            payment_summary.append(
                {
                    "payment_method": method,
                    "total_payments": len(method_payments),
                    "completed": completed,
                    "failed": failed,
                    "refunded": refunded,
                    "success_rate": (
                        round(completed / len(method_payments) * 100, 2)
                        if len(method_payments) > 0
                        else 0
                    ),
                    "total_volume": round(total_amount, 2),
                }
            )

    if payment_summary:
        summary_df = pd.DataFrame(payment_summary)
        summary_df.to_csv("payments_method_summary.csv", index=False)
        print(f"✅ Payment method summary saved to 'payments_method_summary.csv'")

Generating 3500 rows of messy payments data...


/tmp/ipykernel_9007/529757749.py:468: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)



Dataset shape: (3622, 20)

Column names (realistic but challenging for mapping):
  1. payment_ref
  2. order_ref
  3. payment_type
  4. gateway_provider
  5. payment_status
  6. transaction_date
  7. payment_amount
  8. gateway_transaction_id
  9. transaction_fee
  10. refund_total
  11. refund_processed_date
  12. record_created
  13. risk_score
  14. billing_country
  15. authorization_code
  16. customer_ip
  17. currency_code
  18. card_last_four
  19. card_brand
  20. retry_attempt

First 10 rows:
  payment_ref       order_ref       payment_type gateway_provider  \
0     1002257      ORD_100814          Gift Card     Store Credit   
1     1002725      ORD_101391               None           PayPal   
2     1001497      ORD_101115     Cryptocurrency           BitPay   
3     1001060      ORD_101303        Credit Card        Braintree   
4     1000295      ORD_100051             PayPal           PayPal   
5     1002584    ORD_101463          Credit Card           Stripe   
6     10

### Inventory Table Generator

In [7]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import string

# Initialize Faker
fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)


def generate_messy_inventory_data(
    num_rows=1500, product_id_format="PROD", supplier_id_format="SUPP"
):
    """
    Generate a messy, realistic inventory dataset with realistic but challenging column names
    and various data quality issues for testing data mapping and cleaning.

    Args:
        num_rows: Number of inventory records to generate
        product_id_format: Format of product IDs ('PROD', 'PRODUCT', or 'NUMBER')
        supplier_id_format: Format of supplier IDs ('SUPP', 'SUPPLIER', or 'NUMBER')
    """

    data = []

    # Track IDs for creating duplicates and relationships
    used_inventory_ids = []

    # Generate pools of product and supplier IDs
    num_products = num_rows  # One inventory record per product typically
    num_suppliers = max(num_rows // 20, 30)  # Fewer suppliers than products

    product_ids = []
    for i in range(num_products):
        if product_id_format == "PROD":
            prod_id = f"PROD_{str(i + 1).zfill(4)}"
        elif product_id_format == "PRODUCT":
            prod_id = f"PRODUCT-{str(i + 1).zfill(4)}"
        else:  # NUMBER
            prod_id = str(10000 + i)
        product_ids.append(prod_id)

    supplier_ids = []
    for i in range(num_suppliers):
        if supplier_id_format == "SUPP":
            supp_id = f"SUPP_{str(i + 1).zfill(3)}"
        elif supplier_id_format == "SUPPLIER":
            supp_id = f"SUPPLIER-{str(i + 1).zfill(3)}"
        else:  # NUMBER
            supp_id = str(1000 + i)
        supplier_ids.append(supp_id)

    # Choose one consistent inventory ID format
    inv_id_format_choice = random.choice(["INV", "INVENTORY", "NUMBER"])

    # Track product inventory to avoid duplicates (mostly)
    product_inventory_map = {}

    # Product categories for realistic stock patterns
    product_categories = {
        "high_turnover": product_ids[: int(len(product_ids) * 0.2)],  # 20% fast-moving
        "medium_turnover": product_ids[
            int(len(product_ids) * 0.2) : int(len(product_ids) * 0.6)
        ],  # 40% medium
        "low_turnover": product_ids[
            int(len(product_ids) * 0.6) : int(len(product_ids) * 0.9)
        ],  # 30% slow
        "obsolete": product_ids[
            int(len(product_ids) * 0.9) :
        ],  # 10% obsolete/discontinued
    }

    for i in range(num_rows):
        record = {}

        # Inventory ID with CONSISTENT format
        if i % 50 == 0 and used_inventory_ids:  # 2% duplicates
            inv_id = random.choice(used_inventory_ids)
        else:
            if inv_id_format_choice == "INV":
                inv_id = f"INV{str(i + 1).zfill(5)}"
            elif inv_id_format_choice == "INVENTORY":
                inv_id = f"INVENTORY-{str(i + 1).zfill(5)}"
            else:  # NUMBER
                inv_id = str(100000 + i)

            used_inventory_ids.append(inv_id)

        record["inv_id"] = inv_id if i % 100 != 0 else None  # 1% null IDs

        # Product ID - NOT NULL in schema
        if i % 85 == 0:  # Violates NOT NULL
            prod_id = None
        elif i % 60 == 0:  # Non-existent product IDs (FK violation)
            if product_id_format == "PROD":
                prod_id = f"PROD_{str(9999).zfill(4)}"
            elif product_id_format == "PRODUCT":
                prod_id = f"PRODUCT-{str(9999).zfill(4)}"
            else:
                prod_id = "99999"
        elif i % 40 == 0:  # Invalid format
            prod_id = random.choice(["INVALID", "NULL", "N/A", "", "DISCONTINUED"])
        elif (
            i % 70 == 0 and product_inventory_map
        ):  # Duplicate product (business logic violation)
            # Same product with multiple inventory records
            prod_id = random.choice(list(product_inventory_map.keys())[:10])
        else:
            # Pick products that haven't been assigned yet
            available_products = [
                p for p in product_ids if p not in product_inventory_map
            ]
            if available_products:
                prod_id = random.choice(available_products)
            else:
                prod_id = random.choice(product_ids)

        product_inventory_map[prod_id] = True
        record["product_ref"] = prod_id

        # Determine product category for realistic patterns
        product_category = "medium_turnover"  # default
        for category, products in product_categories.items():
            if prod_id in products:
                product_category = category
                break

        # Supplier ID
        if i % 25 == 0:
            supp_id = None
        elif i % 35 == 0:  # Non-existent supplier IDs (FK violation)
            if supplier_id_format == "SUPP":
                supp_id = f"SUPP_{str(999).zfill(3)}"
            elif supplier_id_format == "SUPPLIER":
                supp_id = f"SUPPLIER-{str(999).zfill(3)}"
            else:
                supp_id = "9999"
        elif i % 45 == 0:  # Invalid format
            supp_id = random.choice(["INVALID", "N/A", "NULL", "", "UNKNOWN"])
        else:
            supp_id = random.choice(supplier_ids)

        record["vendor_id"] = supp_id

        # Stock Quantity - NOT NULL in schema
        if i % 90 == 0:  # Violates NOT NULL
            stock_qty = None
        elif i % 30 == 0:  # String values
            stock_qty = random.choice(["Out of Stock", "Many", "Few", "N/A", "Unknown"])
        elif i % 40 == 0:  # Negative stock (business logic violation)
            stock_qty = random.randint(-100, -1)
        elif i % 50 == 0:  # Extreme values
            stock_qty = random.choice([99999, 1000000, -9999])
        elif i % 60 == 0:  # Decimal stock for physical items
            stock_qty = random.choice([10.5, 25.3, 100.75])
        else:
            # Realistic stock based on product category
            if product_category == "high_turnover":
                stock_qty = random.choices(
                    [
                        0,
                        random.randint(1, 10),
                        random.randint(11, 50),
                        random.randint(51, 200),
                        random.randint(201, 1000),
                    ],
                    weights=[5, 10, 30, 40, 15],  # Often low stock due to high sales
                    k=1,
                )[0]
            elif product_category == "medium_turnover":
                stock_qty = random.choices(
                    [
                        0,
                        random.randint(1, 20),
                        random.randint(21, 100),
                        random.randint(101, 500),
                    ],
                    weights=[3, 20, 50, 27],
                    k=1,
                )[0]
            elif product_category == "low_turnover":
                stock_qty = random.choices(
                    [
                        0,
                        random.randint(1, 50),
                        random.randint(51, 200),
                        random.randint(201, 1000),
                    ],
                    weights=[2, 15, 40, 43],  # Higher stock due to low sales
                    k=1,
                )[0]
            else:  # obsolete
                stock_qty = random.choices(
                    [0, random.randint(1, 10), random.randint(11, 100)],
                    weights=[60, 30, 10],  # Mostly out of stock
                    k=1,
                )[0]

        record["current_stock"] = stock_qty

        # Reserved Quantity (should be <= stock_quantity)
        if i % 25 == 0:
            reserved_qty = None
        elif i % 35 == 0:  # String values
            reserved_qty = random.choice(["Some", "None", "All", "N/A"])
        elif i % 45 == 0:  # Negative reserved (business logic violation)
            reserved_qty = random.randint(-50, -1)
        elif i % 55 == 0:  # Reserved > stock (business logic violation)
            if isinstance(stock_qty, (int, float)) and stock_qty > 0:
                reserved_qty = int(stock_qty * random.uniform(1.1, 2.0))
            else:
                reserved_qty = random.randint(100, 500)
        elif i % 65 == 0:  # Decimal reserved
            reserved_qty = random.choice([5.5, 10.25, 15.75])
        else:
            if isinstance(stock_qty, (int, float)) and stock_qty > 0:
                # Realistic reservation based on stock
                if stock_qty == 0:
                    reserved_qty = 0
                else:
                    reserved_qty = random.choices(
                        [
                            0,
                            int(stock_qty * 0.1),
                            int(stock_qty * 0.3),
                            int(stock_qty * 0.5),
                            stock_qty,
                        ],
                        weights=[30, 25, 25, 15, 5],  # Most have some reservation
                        k=1,
                    )[0]
            else:
                reserved_qty = 0

        record["reserved_stock"] = reserved_qty

        # Reorder Level (threshold for restocking)
        if i % 30 == 0:
            reorder_level = None
        elif i % 40 == 0:  # String values
            reorder_level = random.choice(["Low", "Medium", "High", "Auto"])
        elif i % 50 == 0:  # Negative reorder level
            reorder_level = random.randint(-50, -1)
        elif i % 60 == 0:  # Extreme reorder levels
            reorder_level = random.choice([0, 99999, -999])
        elif i % 70 == 0:  # Reorder level > current stock (always reordering)
            if isinstance(stock_qty, (int, float)):
                reorder_level = int(abs(stock_qty) * 2 + 100)
            else:
                reorder_level = 1000
        else:
            # Realistic reorder level based on category
            if product_category == "high_turnover":
                reorder_level = random.randint(50, 200)
            elif product_category == "medium_turnover":
                reorder_level = random.randint(20, 100)
            elif product_category == "low_turnover":
                reorder_level = random.randint(5, 30)
            else:  # obsolete
                reorder_level = 0  # Don't reorder

        record["min_stock_level"] = reorder_level

        # Last Restocked Date
        if i % 25 == 0:
            restocked = None
        elif i % 35 == 0:  # String format variations
            restocked_dt = fake.date_time_between(start_date="-6m", end_date="now")
            formats = ["%Y-%m-%d %H:%M:%S", "%m/%d/%Y", "%d-%m-%Y", "%Y%m%d"]
            restocked = restocked_dt.strftime(random.choice(formats))
        elif i % 45 == 0:  # Future restock (business logic violation)
            restocked = fake.date_time_between(start_date="+1d", end_date="+30d")
        elif i % 55 == 0:  # Very old restock (inventory issues)
            restocked = fake.date_time_between(start_date="-5y", end_date="-2y")
        else:
            # Realistic restock based on category
            if product_category == "high_turnover":
                restocked = fake.date_time_between(start_date="-7d", end_date="now")
            elif product_category == "medium_turnover":
                restocked = fake.date_time_between(start_date="-30d", end_date="now")
            elif product_category == "low_turnover":
                restocked = fake.date_time_between(start_date="-90d", end_date="now")
            else:  # obsolete
                restocked = fake.date_time_between(start_date="-2y", end_date="-6m")

        record["last_restock_date"] = restocked

        # Last Sold Date
        if i % 30 == 0:
            last_sold = None
        elif (
            i % 40 == 0
        ):  # Business logic violation: sold after restock but before restock date
            if isinstance(restocked, datetime):
                last_sold = restocked - timedelta(days=random.randint(1, 30))
            else:
                last_sold = fake.date_time_between(start_date="-1y", end_date="-6m")
        elif i % 50 == 0:  # Future sale (business logic violation)
            last_sold = fake.date_time_between(start_date="+1d", end_date="+30d")
        elif i % 60 == 0:  # Never sold but low stock (business logic issue)
            if isinstance(stock_qty, (int, float)) and stock_qty < 10:
                last_sold = None  # Low stock but never sold
            else:
                last_sold = fake.date_time_between(start_date="-30d", end_date="now")
        else:
            # Realistic last sold based on category and stock
            if product_category == "high_turnover":
                last_sold = fake.date_time_between(start_date="-1d", end_date="now")
            elif product_category == "medium_turnover":
                last_sold = fake.date_time_between(start_date="-7d", end_date="now")
            elif product_category == "low_turnover":
                last_sold = fake.date_time_between(start_date="-30d", end_date="now")
            else:  # obsolete
                last_sold = fake.date_time_between(start_date="-1y", end_date="-3m")

        record["last_sale_date"] = last_sold

        # Storage Cost per unit per month
        if i % 25 == 0:
            storage_cost = None
        elif i % 35 == 0:  # String values
            storage_cost = random.choice(["Free", "Included", "N/A", "Variable"])
        elif i % 45 == 0:  # Negative cost
            storage_cost = round(random.uniform(-5, -0.1), 2)
        elif i % 55 == 0:  # Extreme costs
            storage_cost = random.choice([0, 999.99, 0.001, -99])
        elif i % 65 == 0:  # Zero cost
            storage_cost = 0
        else:
            # Realistic storage cost based on product type
            storage_cost = round(random.uniform(0.10, 5.00), 2)

        record["monthly_storage_cost"] = storage_cost

        # Created At timestamp
        if i % 30 == 0:
            created = None
        elif i % 40 == 0:  # String format
            created = fake.date_time_between(start_date="-2y", end_date="now").strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        elif i % 50 == 0:  # Future creation (business logic violation)
            created = fake.date_time_between(start_date="+1d", end_date="+30d")
        else:
            created = fake.date_time_between(start_date="-2y", end_date="now")

        record["created_date"] = created

        # Additional realistic columns that might exist

        # Warehouse location
        if random.random() > 0.4:  # 60% have warehouse info
            if i % 45 == 0:
                warehouse = None
            elif i % 55 == 0:
                warehouse = random.choice(["", "N/A", "Multiple"])
            else:
                warehouse = random.choice(
                    [
                        "WH-EAST-01",
                        "WH-WEST-01",
                        "WH-CENTRAL-01",
                        "WH-NORTH-01",
                        "WH-SOUTH-01",
                        "DC-01",
                        "DC-02",
                        "STORE-001",
                        "STORE-002",
                        "DROPSHIP",
                    ]
                )
            record["warehouse_location"] = warehouse

        # Available quantity (stock - reserved)
        if random.random() > 0.5:  # 50% have this calculated field
            if all(isinstance(x, (int, float)) for x in [stock_qty, reserved_qty]):
                available = stock_qty - reserved_qty
                # Business logic violation: negative available
                if i % 75 == 0:
                    available = -abs(available)
            elif i % 65 == 0:
                available = random.choice(["Calculate", "N/A", "Check system"])
            else:
                available = None
            record["available_qty"] = available

        # Stock value (quantity * cost)
        if random.random() > 0.6:  # 40% have stock value
            if isinstance(stock_qty, (int, float)) and stock_qty > 0:
                unit_cost = round(random.uniform(5, 200), 2)
                stock_value = round(stock_qty * unit_cost, 2)
                if i % 80 == 0:  # Wrong calculation
                    stock_value = stock_value * random.uniform(0.5, 1.5)
            else:
                stock_value = None
            record["total_stock_value"] = stock_value

        # Days since last sale
        if random.random() > 0.7:  # 30% have this
            if isinstance(last_sold, datetime):
                days_since = (datetime.now() - last_sold).days
                if i % 85 == 0:  # Negative days (future sale)
                    days_since = -random.randint(1, 30)
            elif i % 75 == 0:
                days_since = random.choice(["Never", "N/A", "Unknown"])
            else:
                days_since = None
            record["days_since_last_sale"] = days_since

        # Stock status (derived field)
        if random.random() > 0.6:  # 40% have status
            if isinstance(stock_qty, (int, float)):
                if stock_qty <= 0:
                    status = "Out of Stock"
                elif (
                    isinstance(reorder_level, (int, float))
                    and stock_qty <= reorder_level
                ):
                    status = "Low Stock"
                elif (
                    isinstance(reserved_qty, (int, float)) and reserved_qty >= stock_qty
                ):
                    status = "Fully Reserved"
                else:
                    status = "In Stock"

                # Add some inconsistencies
                if i % 90 == 0:
                    status = random.choice(["Available", "Unavailable", "Check"])
            else:
                status = None
            record["stock_status"] = status

        # Lead time (days to restock)
        if random.random() > 0.7:  # 30% have lead time
            if i % 65 == 0:
                lead_time = None
            elif i % 75 == 0:  # String values
                lead_time = random.choice(["Immediate", "Variable", "TBD"])
            elif i % 85 == 0:  # Negative lead time
                lead_time = random.randint(-10, -1)
            elif i % 95 == 0:  # Extreme lead time
                lead_time = random.choice([0, 999, 365])
            else:
                # Realistic lead time based on supplier
                lead_time = random.choices(
                    [1, 3, 7, 14, 30, 60, 90], weights=[10, 20, 30, 20, 10, 5, 5], k=1
                )[0]
            record["restock_lead_time_days"] = lead_time

        # Expiry date (for perishable items)
        if random.random() > 0.85:  # 15% are perishable
            if i % 70 == 0:
                expiry = None
            elif i % 80 == 0:  # Already expired
                expiry = fake.date_between(start_date="-30d", end_date="-1d")
            elif i % 90 == 0:  # String format
                expiry = random.choice(["N/A", "Non-perishable", "Check package"])
            else:
                expiry = fake.date_between(start_date="+1d", end_date="+2y")
            record["expiry_date"] = expiry

        data.append(record)

    # Create DataFrame
    df = pd.DataFrame(data)

    # Add EXACT duplicate rows (every attribute is same)
    num_exact_duplicates = int(num_rows * 0.02)  # 2% exact duplicates
    for _ in range(num_exact_duplicates):
        if len(df) > 0:
            # Pick a random row to duplicate
            row_to_duplicate = df.sample(1)
            df = pd.concat([df, row_to_duplicate], ignore_index=True)

    # Add some completely empty rows
    for _ in range(int(num_rows * 0.005)):  # 0.5% empty rows
        empty_row = pd.Series([None] * len(df.columns), index=df.columns)
        df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)

    # Add some rows with all string 'NULL' or 'N/A' values
    for _ in range(int(num_rows * 0.01)):  # 1% NULL string rows
        null_values = ["NULL", "N/A", "null", "NA", "", " "]
        null_row = pd.Series(
            [random.choice(null_values) for _ in range(len(df.columns))],
            index=df.columns,
        )
        df = pd.concat([df, pd.DataFrame([null_row])], ignore_index=True)

    # Shuffle the dataframe to mix duplicates throughout
    df = df.sample(frac=1).reset_index(drop=True)

    return df


def add_more_messiness(df):
    """
    Add additional data quality issues to make the dataset more challenging
    """
    # Add trailing/leading spaces to some string columns
    string_cols = df.select_dtypes(include=["object"]).columns
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05  # 5% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    # Add case inconsistencies
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03  # 3% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    # Add special characters to some values
    for col in string_cols[:3]:  # Only first 3 string columns
        mask = np.random.random(len(df)) < 0.02  # 2% of values
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


# Generate the dataset
if __name__ == "__main__":
    # Set the number of rows you want
    NUM_ROWS = 1500  # Change this to your desired number

    # ID formats should match your products and suppliers datasets
    PRODUCT_ID_FORMAT = "PROD"  # Options: 'PROD', 'PRODUCT', or 'NUMBER'
    SUPPLIER_ID_FORMAT = "SUPP"  # Options: 'SUPP', 'SUPPLIER', or 'NUMBER'

    print(f"Generating {NUM_ROWS} rows of messy inventory data...")
    df = generate_messy_inventory_data(NUM_ROWS, PRODUCT_ID_FORMAT, SUPPLIER_ID_FORMAT)

    # Add more messiness
    df = add_more_messiness(df)

    # Display basic info
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumn names (realistic but challenging for mapping):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")

    print(f"\nFirst 10 rows:")
    print(df.head(10))

    # Show data quality issues summary
    print("\n" + "=" * 50)
    print("DATA QUALITY ISSUES SUMMARY:")
    print("=" * 50)
    print(f"Total null values: {df.isnull().sum().sum()}")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    print(f"Total rows: {len(df)}")

    # Check NOT NULL constraint violations
    print("\n" + "=" * 50)
    print("NOT NULL CONSTRAINT VIOLATIONS:")
    print("=" * 50)

    if "product_ref" in df.columns:
        null_products = df[df["product_ref"].isnull()].shape[0]
        if null_products > 0:
            print(f"✗ NULL product_ref (violates NOT NULL): {null_products} rows")

    if "current_stock" in df.columns:
        null_stock = df[df["current_stock"].isnull()].shape[0]
        if null_stock > 0:
            print(f"✗ NULL current_stock (violates NOT NULL): {null_stock} rows")

    # Show business logic violations
    print("\n" + "=" * 50)
    print("BUSINESS LOGIC VIOLATIONS EXAMPLES:")
    print("=" * 50)

    # Check for negative stock
    if "current_stock" in df.columns:
        stock_numeric = pd.to_numeric(df["current_stock"], errors="coerce")
        negative_stock = df[stock_numeric < 0]
        if len(negative_stock) > 0:
            print(f"✗ Negative stock quantities: {len(negative_stock)} products")
            print(
                f"  Total negative units: {stock_numeric[stock_numeric < 0].sum():.0f}"
            )

    # Check for reserved > stock
    if "current_stock" in df.columns and "reserved_stock" in df.columns:
        stock_numeric = pd.to_numeric(df["current_stock"], errors="coerce")
        reserved_numeric = pd.to_numeric(df["reserved_stock"], errors="coerce")
        over_reserved = df[reserved_numeric > stock_numeric]
        if len(over_reserved) > 0:
            print(
                f"✗ Reserved quantity > stock quantity: {len(over_reserved)} products"
            )

    # Check for negative available quantity
    if "available_qty" in df.columns:
        available_numeric = pd.to_numeric(df["available_qty"], errors="coerce")
        negative_available = df[available_numeric < 0]
        if len(negative_available) > 0:
            print(f"✗ Negative available quantity: {len(negative_available)} products")

    # Check for duplicate products (same product multiple inventory records)
    if "product_ref" in df.columns:
        duplicate_products = df[
            df.duplicated(subset=["product_ref"], keep=False)
            & df["product_ref"].notna()
            & (~df["product_ref"].isin(["INVALID", "NULL", "N/A", "", "DISCONTINUED"]))
        ]
        if len(duplicate_products) > 0:
            print(
                f"✗ Products with multiple inventory records: {len(duplicate_products)} rows"
            )
            dup_count = duplicate_products["product_ref"].value_counts().head(3)
            print(f"  Top duplicated products: {dup_count.to_dict()}")

    # Check for last sold before last restocked
    if "last_restock_date" in df.columns and "last_sale_date" in df.columns:
        restock_dt = pd.to_datetime(df["last_restock_date"], errors="coerce")
        sold_dt = pd.to_datetime(df["last_sale_date"], errors="coerce")
        impossible_sale = df[
            sold_dt < restock_dt - timedelta(days=1)
        ]  # Sold before last restock
        if len(impossible_sale) > 0:
            print(
                f"✗ Products sold before last restock: {len(impossible_sale)} products"
            )

    # Check for obsolete items with high stock
    if "current_stock" in df.columns and "last_sale_date" in df.columns:
        stock_numeric = pd.to_numeric(df["current_stock"], errors="coerce")
        sold_dt = pd.to_datetime(df["last_sale_date"], errors="coerce")
        days_since = (datetime.now() - sold_dt).dt.days
        obsolete_high_stock = df[(days_since > 180) & (stock_numeric > 100)]
        if len(obsolete_high_stock) > 0:
            print(
                f"✗ Obsolete items (>180 days) with high stock (>100): {len(obsolete_high_stock)} products"
            )

    # Check for invalid foreign keys
    if "product_ref" in df.columns:
        invalid_products = df[
            df["product_ref"].isin(["INVALID", "NULL", "N/A", "", "DISCONTINUED"])
        ]
        if len(invalid_products) > 0:
            print(f"✗ Invalid product references: {len(invalid_products)} rows")

    if "vendor_id" in df.columns:
        invalid_suppliers = df[
            df["vendor_id"].isin(["INVALID", "N/A", "NULL", "", "UNKNOWN"])
        ]
        if len(invalid_suppliers) > 0:
            print(f"✗ Invalid supplier references: {len(invalid_suppliers)} rows")

    # Show inventory statistics
    print("\n" + "=" * 50)
    print("INVENTORY STATISTICS:")
    print("=" * 50)

    if "current_stock" in df.columns:
        stock_numeric = pd.to_numeric(df["current_stock"], errors="coerce")
        valid_stock = stock_numeric[stock_numeric >= 0]

        if len(valid_stock) > 0:
            print(f"Stock levels:")
            print(f"  Total units in stock: {valid_stock.sum():,.0f}")
            print(f"  Average stock per product: {valid_stock.mean():.1f}")
            print(f"  Median stock per product: {valid_stock.median():.0f}")

            # Stock distribution
            out_of_stock = (stock_numeric == 0).sum()
            low_stock = ((stock_numeric > 0) & (stock_numeric <= 10)).sum()
            normal_stock = ((stock_numeric > 10) & (stock_numeric <= 100)).sum()
            high_stock = (stock_numeric > 100).sum()

            print(f"\nStock distribution:")
            print(f"  Out of stock: {out_of_stock} ({out_of_stock/len(df)*100:.1f}%)")
            print(f"  Low (1-10): {low_stock} ({low_stock/len(df)*100:.1f}%)")
            print(
                f"  Normal (11-100): {normal_stock} ({normal_stock/len(df)*100:.1f}%)"
            )
            print(f"  High (>100): {high_stock} ({high_stock/len(df)*100:.1f}%)")

    # Show reservation analysis
    if "current_stock" in df.columns and "reserved_stock" in df.columns:
        stock_numeric = pd.to_numeric(df["current_stock"], errors="coerce")
        reserved_numeric = pd.to_numeric(df["reserved_stock"], errors="coerce")

        reservation_rate = (reserved_numeric / stock_numeric * 100).replace(
            [np.inf, -np.inf], np.nan
        )

        print(f"\nReservation analysis:")
        print(
            f"  Total reserved units: {reserved_numeric[reserved_numeric > 0].sum():,.0f}"
        )
        print(f"  Average reservation rate: {reservation_rate.mean():.1f}%")
        print(f"  Products with reservations: {(reserved_numeric > 0).sum()}")
        print(f"  Fully reserved products: {(reserved_numeric == stock_numeric).sum()}")

    # Show warehouse distribution
    if "warehouse_location" in df.columns:
        warehouse_dist = df["warehouse_location"].value_counts()
        print(f"\nWarehouse distribution:")
        for warehouse, count in warehouse_dist.head(5).items():
            if warehouse and warehouse not in ["", "N/A", "Multiple"]:
                print(f"  {warehouse}: {count} products")

    # Show storage cost analysis
    if "monthly_storage_cost" in df.columns and "current_stock" in df.columns:
        storage_numeric = pd.to_numeric(df["monthly_storage_cost"], errors="coerce")
        stock_numeric = pd.to_numeric(df["current_stock"], errors="coerce")

        monthly_cost = (storage_numeric * stock_numeric).sum()
        if monthly_cost > 0:
            print(f"\nStorage cost analysis:")
            print(f"  Total monthly storage cost: ${monthly_cost:,.2f}")
            print(
                f"  Average cost per unit: ${storage_numeric[storage_numeric > 0].mean():.2f}"
            )

    # Show turnover analysis
    if "last_sale_date" in df.columns:
        sold_dt = pd.to_datetime(df["last_sale_date"], errors="coerce")
        days_since = (datetime.now() - sold_dt).dt.days

        print(f"\nTurnover analysis:")
        fast_moving = (days_since <= 7).sum()
        medium_moving = ((days_since > 7) & (days_since <= 30)).sum()
        slow_moving = ((days_since > 30) & (days_since <= 90)).sum()
        dead_stock = (days_since > 90).sum()

        print(f"  Fast moving (≤7 days): {fast_moving} products")
        print(f"  Medium moving (8-30 days): {medium_moving} products")
        print(f"  Slow moving (31-90 days): {slow_moving} products")
        print(f"  Dead stock (>90 days): {dead_stock} products")

    # Show main columns analysis
    print("\n" + "=" * 50)
    print("MAIN COLUMNS ANALYSIS:")
    print("=" * 50)

    main_columns = [
        "inv_id",
        "product_ref",
        "vendor_id",
        "current_stock",
        "reserved_stock",
        "min_stock_level",
        "last_restock_date",
        "last_sale_date",
        "monthly_storage_cost",
    ]

    for col in main_columns:
        if col in df.columns:
            null_count = df[col].isnull().sum()
            unique_count = df[col].nunique()
            print(f"\n{col}:")
            print(f"  - Null values: {null_count} ({null_count/len(df)*100:.1f}%)")
            print(f"  - Unique values: {unique_count}")

    # Save to CSV
    output_file = "messy_inventory_data.xlsx"
    df.to_excel(output_file, index=False)
    print(f"\n✅ Dataset saved to '{output_file}'")

    # Save inventory summary
    inventory_summary = []

    if "current_stock" in df.columns and "reserved_stock" in df.columns:
        stock_numeric = pd.to_numeric(df["current_stock"], errors="coerce")
        reserved_numeric = pd.to_numeric(df["reserved_stock"], errors="coerce")

        # Group by stock status
        for status in ["Out of Stock", "Low Stock", "Normal Stock", "High Stock"]:
            if status == "Out of Stock":
                status_items = df[stock_numeric == 0]
            elif status == "Low Stock":
                status_items = df[(stock_numeric > 0) & (stock_numeric <= 10)]
            elif status == "Normal Stock":
                status_items = df[(stock_numeric > 10) & (stock_numeric <= 100)]
            else:  # High Stock
                status_items = df[stock_numeric > 100]

            if len(status_items) > 0:
                inventory_summary.append(
                    {
                        "stock_status": status,
                        "product_count": len(status_items),
                        "total_units": stock_numeric[status_items.index].sum(),
                        "reserved_units": reserved_numeric[status_items.index].sum(),
                        "avg_stock": stock_numeric[status_items.index].mean(),
                    }
                )

    if inventory_summary:
        summary_df = pd.DataFrame(inventory_summary)
        summary_df.to_csv("inventory_status_summary.csv", index=False)
        print(f"✅ Inventory status summary saved to 'inventory_status_summary.csv'")

Generating 1500 rows of messy inventory data...


/tmp/ipykernel_9007/1269812433.py:483: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)
/tmp/ipykernel_9007/1269812433.py:483: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)
/tmp/ipykernel_9007/1269812433.py:483: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or 


Dataset shape: (1552, 17)

Column names (realistic but challenging for mapping):
  1. inv_id
  2. product_ref
  3. vendor_id
  4. current_stock
  5. reserved_stock
  6. min_stock_level
  7. last_restock_date
  8. last_sale_date
  9. monthly_storage_cost
  10. created_date
  11. available_qty
  12. days_since_last_sale
  13. stock_status
  14. warehouse_location
  15. total_stock_value
  16. restock_lead_time_days
  17. expiry_date

First 10 rows:
   inv_id product_ref     vendor_id current_stock reserved_stock  \
0  100555   PROD_0781      SUPP_017            72             21   
1  100938   PROD_0373      SUPP_003             8              4   
2  100060   PROD_9999    SUPP_029          Many                0   
3  100693   PROD_0244      SUPP_022         240               24   
4  100407   PROD_1172      SUPP_073            33              3   
5  100592   PROD_0311      SUPP_011            29              0   
6  101068   PROD_0409          None       1000000           None   
7  1

### Reviews Table Generator

In [6]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import string

# Initialize Faker
fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)


def generate_messy_reviews_data(
    num_rows=3000, product_id_format="PROD", customer_id_format="CUST"
):
    """
    Generate a messy, realistic reviews dataset with realistic but challenging column names
    and various data quality issues for testing data mapping and cleaning.

    Args:
        num_rows: Number of review records to generate
        product_id_format: Format of product IDs ('PROD', 'PRODUCT', or 'NUMBER')
        customer_id_format: Format of customer IDs ('CUST', 'CUSTOMER', or 'NUMBER')
    """

    data = []

    # Track IDs for creating duplicates and relationships
    used_review_ids = []

    # Generate pools of product and customer IDs
    num_products = max(num_rows // 10, 100)  # Average 10 reviews per product
    num_customers = max(num_rows // 5, 200)  # Average 5 reviews per customer

    product_ids = []
    for i in range(num_products):
        if product_id_format == "PROD":
            prod_id = f"PROD_{str(i + 1).zfill(4)}"
        elif product_id_format == "PRODUCT":
            prod_id = f"PRODUCT-{str(i + 1).zfill(4)}"
        else:  # NUMBER
            prod_id = str(10000 + i)
        product_ids.append(prod_id)

    customer_ids = []
    for i in range(num_customers):
        if customer_id_format == "CUST":
            cust_id = f"CUST_{str(i + 1000).zfill(5)}"
        elif customer_id_format == "CUSTOMER":
            cust_id = f"CUSTOMER-{str(i + 1000).zfill(5)}"
        else:  # NUMBER
            cust_id = str(10000 + i)
        customer_ids.append(cust_id)

    # Choose one consistent review ID format
    review_id_format_choice = random.choice(["REV", "REVIEW", "NUMBER"])

    # Track customer-product pairs to detect duplicate reviews
    customer_product_pairs = {}

    # Popular products get more reviews
    popular_products = product_ids[:20]  # Top 20 products

    # Review templates for fake/spam reviews
    spam_titles = [
        "AMAZING!!!",
        "BEST EVER",
        "DO NOT BUY",
        "SCAM!!!",
        "Five Stars",
        "Good",
        "OK",
        "Nice",
        "👍",
        "⭐⭐⭐⭐⭐",
    ]

    generic_reviews = [
        "Good product",
        "As expected",
        "Nice quality",
        "Fast shipping",
        "Would buy again",
        "Recommended",
        "Not bad",
        "Pretty good",
        "Satisfied with purchase",
        "Met expectations",
    ]

    for i in range(num_rows):
        record = {}

        # Review ID with CONSISTENT format
        if i % 50 == 0 and used_review_ids:  # 2% duplicates
            review_id = random.choice(used_review_ids)
        else:
            if review_id_format_choice == "REV":
                review_id = f"REV{str(i + 1).zfill(5)}"
            elif review_id_format_choice == "REVIEW":
                review_id = f"REVIEW-{str(i + 1).zfill(5)}"
            else:  # NUMBER
                review_id = str(100000 + i)

            used_review_ids.append(review_id)

        record["review_ref"] = review_id if i % 100 != 0 else None  # 1% null IDs

        # Product ID - NOT NULL in schema
        if i % 85 == 0:  # Violates NOT NULL
            prod_id = None
        elif i % 60 == 0:  # Non-existent product IDs (FK violation)
            if product_id_format == "PROD":
                prod_id = f"PROD_{str(9999).zfill(4)}"
            elif product_id_format == "PRODUCT":
                prod_id = f"PRODUCT-{str(9999).zfill(4)}"
            else:
                prod_id = "99999"
        elif i % 40 == 0:  # Invalid format
            prod_id = random.choice(["INVALID", "NULL", "N/A", "", "REMOVED"])
        else:
            # Popular products get more reviews
            if random.random() < 0.4:  # 40% chance for popular product
                prod_id = random.choice(popular_products)
            else:
                prod_id = random.choice(product_ids)

        record["product_ref"] = prod_id

        # Customer ID (can be NULL for anonymous reviews)
        if i % 15 == 0:  # Anonymous review
            cust_id = None
        elif i % 55 == 0:  # Non-existent customer IDs (FK violation)
            if customer_id_format == "CUST":
                cust_id = f"CUST_{str(99999).zfill(5)}"
            elif customer_id_format == "CUSTOMER":
                cust_id = f"CUSTOMER-{str(99999).zfill(5)}"
            else:
                cust_id = "99999"
        elif i % 45 == 0:  # Invalid format
            cust_id = random.choice(["INVALID", "GUEST", "ANONYMOUS", ""])
        else:
            cust_id = random.choice(customer_ids)

        # Check for duplicate review (same customer-product)
        if cust_id and prod_id:
            pair_key = f"{cust_id}_{prod_id}"
            if i % 30 == 0 and pair_key in customer_product_pairs:
                # Business logic violation: duplicate review
                pass  # Will create duplicate
            customer_product_pairs[pair_key] = True

        record["customer_ref"] = cust_id

        # Rating (should be 1-5)
        if i % 25 == 0:
            rating = None
        elif i % 35 == 0:  # String values
            rating = random.choice(["Five stars", "Good", "Bad", "N/A", "****"])
        elif i % 45 == 0:  # Out of range (violates CHECK constraint)
            rating = random.choice([0, 6, 10, -1, 100])
        elif i % 55 == 0:  # Decimal ratings
            rating = random.choice([3.5, 4.5, 2.7, 1.8])
        else:
            # Realistic rating distribution (J-shaped: more 5s and 1s)
            rating = random.choices(
                [1, 2, 3, 4, 5],
                weights=[15, 5, 10, 25, 45],  # Most reviews are positive
                k=1,
            )[0]

        record["star_rating"] = rating

        # Review Title
        if i % 30 == 0:
            title = None
        elif i % 40 == 0:  # Empty or minimal
            title = random.choice(["", " ", ".", "?", "N/A"])
        elif i % 50 == 0:  # Spam/fake titles
            title = random.choice(spam_titles)
        elif i % 60 == 0:  # Very long title
            title = fake.text(max_nb_chars=500)[:255]  # Truncated
        elif i % 70 == 0:  # Special characters/emojis
            title = random.choice(
                [
                    "⭐⭐⭐⭐⭐ AMAZING!",
                    "👍 Great product",
                    "❌ Terrible",
                    "💯 Perfect!",
                    "🔥 Hot item",
                    "💰 Worth the price",
                ]
            )
        elif i % 80 == 0:  # All caps
            title = fake.sentence(nb_words=4).upper()
        else:
            # Generate realistic title based on rating
            if isinstance(rating, int):
                if rating >= 4:
                    title = random.choice(
                        [
                            "Great product!",
                            "Excellent quality",
                            "Highly recommend",
                            "Love it!",
                            "Perfect!",
                            "Exceeded expectations",
                            "Amazing value",
                            "Very satisfied",
                            fake.sentence(nb_words=4),
                        ]
                    )
                elif rating == 3:
                    title = random.choice(
                        [
                            "Decent product",
                            "It's okay",
                            "Average quality",
                            "Not bad",
                            "Could be better",
                            "Mixed feelings",
                            fake.sentence(nb_words=3),
                        ]
                    )
                else:
                    title = random.choice(
                        [
                            "Disappointed",
                            "Not worth it",
                            "Poor quality",
                            "Waste of money",
                            "Do not recommend",
                            "Terrible experience",
                            "Buyer beware",
                            fake.sentence(nb_words=3),
                        ]
                    )
            else:
                title = fake.sentence(nb_words=4)

        record["review_headline"] = title

        # Review Text
        if i % 20 == 0:
            text = None
        elif i % 30 == 0:  # Empty or minimal
            text = random.choice(["", " ", "Good", "Bad", "OK", "."])
        elif i % 40 == 0:  # Generic/fake review
            text = random.choice(generic_reviews)
        elif i % 50 == 0:  # Spam review
            if random.random() < 0.5:
                # Promotional spam
                text = f"Check out {fake.url()} for amazing deals! Contact {fake.email()} for more info."
            else:
                # Repetitive text
                word = random.choice(["GREAT", "BAD", "LOVE", "HATE"])
                text = f"{word} " * random.randint(10, 50)
        elif i % 60 == 0:  # Very long review
            text = fake.text(max_nb_chars=5000)
        elif i % 70 == 0:  # Non-English or special characters
            text = random.choice(
                [
                    "很好的产品！强烈推荐。",
                    "Très bon produit, je recommande!",
                    "отличный продукт",
                    "素晴らしい製品です",
                ]
            )
        elif i % 80 == 0:  # Review doesn't match rating (business logic violation)
            if isinstance(rating, int):
                if rating >= 4:
                    # Positive rating but negative review
                    text = (
                        fake.paragraph(nb_sentences=3)
                        + " However, I'm very disappointed with this product. Would not buy again."
                    )
                else:
                    # Negative rating but positive review
                    text = (
                        "This is the best product I've ever purchased! "
                        + fake.paragraph(nb_sentences=3)
                    )
            else:
                text = fake.paragraph(nb_sentences=5)
        else:
            # Generate realistic review based on rating
            if isinstance(rating, int):
                if rating >= 4:
                    text = fake.paragraph(nb_sentences=random.randint(2, 5))
                    text += random.choice(
                        [
                            " Highly recommend!",
                            " Would buy again.",
                            " Great value for money.",
                            " Exceeded my expectations.",
                            " Very happy with this purchase.",
                            "",
                        ]
                    )
                elif rating == 3:
                    text = fake.paragraph(nb_sentences=random.randint(2, 4))
                    text += random.choice(
                        [
                            " It's okay for the price.",
                            " Has pros and cons.",
                            " Average product.",
                            " Meets basic needs.",
                            "",
                        ]
                    )
                else:
                    text = fake.paragraph(nb_sentences=random.randint(1, 3))
                    text += random.choice(
                        [
                            " Very disappointed.",
                            " Not worth the money.",
                            " Quality is poor.",
                            " Would not recommend.",
                            " Returned immediately.",
                            "",
                        ]
                    )
            else:
                text = fake.paragraph(nb_sentences=random.randint(2, 5))

        record["review_content"] = text

        # Review Date
        if i % 25 == 0:
            review_date = None
        elif i % 35 == 0:  # String format variations
            review_date_dt = fake.date_time_between(start_date="-2y", end_date="now")
            formats = ["%Y-%m-%d %H:%M:%S", "%m/%d/%Y", "%d-%m-%Y", "%Y%m%d"]
            review_date = review_date_dt.strftime(random.choice(formats))
        elif i % 45 == 0:  # Future date (business logic violation)
            review_date = fake.date_time_between(start_date="+1d", end_date="+30d")
        elif i % 55 == 0:  # Very old review
            review_date = fake.date_time_between(start_date="-10y", end_date="-5y")
        elif i % 65 == 0:  # Suspicious pattern: many reviews same day (fake reviews)
            review_date = datetime.now() - timedelta(days=random.choice([1, 7, 30]))
        else:
            # Realistic distribution: more recent reviews
            days_ago = random.choices(
                [
                    random.randint(1, 7),
                    random.randint(8, 30),
                    random.randint(31, 180),
                    random.randint(181, 730),
                ],
                weights=[40, 30, 20, 10],
                k=1,
            )[0]
            review_date = datetime.now() - timedelta(days=days_ago)

        record["submitted_date"] = review_date

        # Is Verified Purchase
        if i % 30 == 0:
            verified = None
        elif i % 40 == 0:  # Various boolean representations
            verified = random.choice(["Y", "N", "Yes", "No", "1", "0", "true", "false"])
        elif i % 50 == 0:  # Invalid values
            verified = random.choice(["Maybe", "Unknown", "Pending", ""])
        elif i % 60 == 0:  # Business logic violation: anonymous but verified
            if cust_id is None:
                verified = True  # Anonymous verified purchase (unusual)
            else:
                verified = False
        else:
            # Realistic: 70% of reviews are from verified purchases
            verified = random.choices([True, False], weights=[70, 30], k=1)[0]

        record["verified_purchase"] = verified

        # Additional realistic columns that might exist

        # Helpful votes
        if random.random() > 0.4:  # 60% have helpful votes
            if i % 45 == 0:
                helpful = None
            elif i % 55 == 0:  # String values
                helpful = random.choice(["Many", "Few", "None", "N/A"])
            elif i % 65 == 0:  # Negative votes (shouldn't happen)
                helpful = random.randint(-10, -1)
            elif i % 75 == 0:  # Suspicious: too many votes for new review
                if (
                    isinstance(review_date, datetime)
                    and (datetime.now() - review_date).days < 7
                ):
                    helpful = random.randint(100, 1000)  # Many votes on new review
                else:
                    helpful = random.randint(0, 50)
            else:
                # Older reviews have more votes
                if isinstance(review_date, datetime):
                    days_old = (datetime.now() - review_date).days
                    max_votes = min(days_old // 10, 100)
                    helpful = random.randint(0, max(max_votes, 1))
                else:
                    helpful = random.randint(0, 20)
            record["helpful_count"] = helpful

        # Total votes (helpful + not helpful)
        if "helpful_count" in record and random.random() > 0.5:
            if (
                isinstance(record["helpful_count"], int)
                and record["helpful_count"] >= 0
            ):
                total = record["helpful_count"] + random.randint(0, 20)
                if i % 85 == 0:  # Business logic violation: total < helpful
                    total = max(0, record["helpful_count"] - random.randint(1, 5))
            else:
                total = random.randint(0, 30)
            record["total_votes"] = total

        # Review status (moderation)
        if random.random() > 0.6:  # 40% have status
            if i % 50 == 0:
                status = None
            elif i % 60 == 0:
                status = random.choice(["", "N/A", "Unknown"])
            else:
                status = random.choices(
                    ["Approved", "Pending", "Rejected", "Flagged", "Hidden"],
                    weights=[80, 10, 3, 5, 2],
                    k=1,
                )[0]
            record["moderation_status"] = status

        # Response from seller
        if random.random() > 0.9:  # 10% have seller response
            if i % 70 == 0:
                response = None
            elif i % 80 == 0:
                response = ""
            else:
                if isinstance(rating, int) and rating <= 2:
                    response = f"We're sorry to hear about your experience. Please contact {fake.email()} for assistance."
                else:
                    response = "Thank you for your review! We appreciate your feedback."
            record["seller_response"] = response

        # Review images count
        if random.random() > 0.7:  # 30% have images
            if i % 60 == 0:
                images = None
            elif i % 70 == 0:  # String value
                images = random.choice(["Yes", "Multiple", "None"])
            elif i % 80 == 0:  # Negative count
                images = random.randint(-5, -1)
            else:
                images = random.choices(
                    [0, 1, 2, 3, 4, 5], weights=[40, 30, 15, 10, 3, 2], k=1
                )[0]
            record["image_count"] = images

        # Reviewer name (might be different from customer name)
        if random.random() > 0.5:  # 50% have reviewer name
            if i % 55 == 0:
                name = None
            elif i % 65 == 0:
                name = random.choice(["Anonymous", "A Customer", "Verified Buyer", ""])
            elif i % 75 == 0:  # Suspicious: all same name
                name = "John Smith"
            else:
                name = fake.name()
            record["reviewer_name"] = name

        # Device used for review
        if random.random() > 0.7:  # 30% have device info
            if i % 65 == 0:
                device = None
            elif i % 75 == 0:
                device = random.choice(["", "N/A", "Unknown"])
            else:
                device = random.choice(
                    ["Desktop", "Mobile Web", "iOS App", "Android App", "Tablet"]
                )
            record["submission_device"] = device

        # Language
        if random.random() > 0.8:  # 20% have language
            if i % 70 == 0:
                lang = None
            elif i % 80 == 0:
                lang = random.choice(["", "N/A", "Unknown"])
            else:
                lang = random.choices(
                    ["en", "es", "fr", "de", "zh", "ja", "pt", "ru"],
                    weights=[70, 10, 5, 5, 3, 3, 2, 2],
                    k=1,
                )[0]
            record["review_language"] = lang

        data.append(record)

    # Create DataFrame
    df = pd.DataFrame(data)

    # Add EXACT duplicate rows (every attribute is same)
    num_exact_duplicates = int(num_rows * 0.02)  # 2% exact duplicates
    for _ in range(num_exact_duplicates):
        if len(df) > 0:
            # Pick a random row to duplicate
            row_to_duplicate = df.sample(1)
            df = pd.concat([df, row_to_duplicate], ignore_index=True)

    # Add some completely empty rows
    for _ in range(int(num_rows * 0.005)):  # 0.5% empty rows
        empty_row = pd.Series([None] * len(df.columns), index=df.columns)
        df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)

    # Add some rows with all string 'NULL' or 'N/A' values
    for _ in range(int(num_rows * 0.01)):  # 1% NULL string rows
        null_values = ["NULL", "N/A", "null", "NA", "", " "]
        null_row = pd.Series(
            [random.choice(null_values) for _ in range(len(df.columns))],
            index=df.columns,
        )
        df = pd.concat([df, pd.DataFrame([null_row])], ignore_index=True)

    # Shuffle the dataframe to mix duplicates throughout
    df = df.sample(frac=1).reset_index(drop=True)

    return df


def add_more_messiness(df):
    """
    Add additional data quality issues to make the dataset more challenging
    """
    # Add trailing/leading spaces to some string columns
    string_cols = df.select_dtypes(include=["object"]).columns
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05  # 5% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    # Add case inconsistencies
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03  # 3% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    # Add special characters to some values
    for col in string_cols[:3]:  # Only first 3 string columns
        mask = np.random.random(len(df)) < 0.02  # 2% of values
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


# Generate the dataset
if __name__ == "__main__":
    # Set the number of rows you want
    NUM_ROWS = 3000  # Change this to your desired number

    # ID formats should match your products and customers datasets
    PRODUCT_ID_FORMAT = "PROD"  # Options: 'PROD', 'PRODUCT', or 'NUMBER'
    CUSTOMER_ID_FORMAT = "CUST"  # Options: 'CUST', 'CUSTOMER', or 'NUMBER'

    print(f"Generating {NUM_ROWS} rows of messy reviews data...")
    df = generate_messy_reviews_data(NUM_ROWS, PRODUCT_ID_FORMAT, CUSTOMER_ID_FORMAT)

    # Add more messiness
    df = add_more_messiness(df)

    # Display basic info
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumn names (realistic but challenging for mapping):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")

    print(f"\nFirst 10 rows:")
    print(df.head(10))

    # Show data quality issues summary
    print("\n" + "=" * 50)
    print("DATA QUALITY ISSUES SUMMARY:")
    print("=" * 50)
    print(f"Total null values: {df.isnull().sum().sum()}")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    print(f"Total rows: {len(df)}")

    # Check NOT NULL constraint violations
    print("\n" + "=" * 50)
    print("NOT NULL CONSTRAINT VIOLATIONS:")
    print("=" * 50)

    if "product_ref" in df.columns:
        null_products = df[df["product_ref"].isnull()].shape[0]
        if null_products > 0:
            print(f"✗ NULL product_ref (violates NOT NULL): {null_products} rows")

    # Check CHECK constraint violations
    print("\n" + "=" * 50)
    print("CHECK CONSTRAINT VIOLATIONS:")
    print("=" * 50)

    if "star_rating" in df.columns:
        rating_numeric = pd.to_numeric(df["star_rating"], errors="coerce")
        invalid_ratings = df[(rating_numeric < 1) | (rating_numeric > 5)]
        if len(invalid_ratings) > 0:
            print(f"✗ Ratings outside 1-5 range: {len(invalid_ratings)} reviews")
            print(
                f"  Invalid values: {invalid_ratings['star_rating'].unique()[:10].tolist()}"
            )

    # Show business logic violations
    print("\n" + "=" * 50)
    print("BUSINESS LOGIC VIOLATIONS EXAMPLES:")
    print("=" * 50)

    # Check for duplicate reviews (same customer-product)
    if "customer_ref" in df.columns and "product_ref" in df.columns:
        duplicate_reviews = df[
            df.duplicated(subset=["customer_ref", "product_ref"], keep=False)
            & df["customer_ref"].notna()
            & df["product_ref"].notna()
        ]
        if len(duplicate_reviews) > 0:
            print(
                f"✗ Duplicate reviews (same customer-product): {len(duplicate_reviews)} reviews"
            )

    # Check for anonymous verified purchases
    if "customer_ref" in df.columns and "verified_purchase" in df.columns:
        anonymous_verified = df[
            df["customer_ref"].isna()
            & df["verified_purchase"].isin([True, "Y", "Yes", "1", "true"])
        ]
        if len(anonymous_verified) > 0:
            print(
                f"✗ Anonymous but verified purchases: {len(anonymous_verified)} reviews"
            )

    # Check for rating-text mismatches
    if "star_rating" in df.columns and "review_content" in df.columns:
        rating_numeric = pd.to_numeric(df["star_rating"], errors="coerce")
        high_rating_negative = df[
            (rating_numeric >= 4)
            & df["review_content"].str.contains(
                "disappointed|terrible|waste|poor|bad", case=False, na=False
            )
        ]
        if len(high_rating_negative) > 0:
            print(
                f"✗ High rating but negative text: {len(high_rating_negative)} reviews"
            )

        low_rating_positive = df[
            (rating_numeric <= 2)
            & df["review_content"].str.contains(
                "excellent|amazing|perfect|love|great", case=False, na=False
            )
        ]
        if len(low_rating_positive) > 0:
            print(f"✗ Low rating but positive text: {len(low_rating_positive)} reviews")

    # Check for helpful > total votes
    if "helpful_count" in df.columns and "total_votes" in df.columns:
        helpful_numeric = pd.to_numeric(df["helpful_count"], errors="coerce")
        total_numeric = pd.to_numeric(df["total_votes"], errors="coerce")
        invalid_votes = df[helpful_numeric > total_numeric]
        if len(invalid_votes) > 0:
            print(f"✗ Helpful votes > total votes: {len(invalid_votes)} reviews")

    # Check for future review dates
    if "submitted_date" in df.columns:
        review_dt = pd.to_datetime(df["submitted_date"], errors="coerce")
        future_reviews = df[review_dt > datetime.now()]
        if len(future_reviews) > 0:
            print(f"✗ Reviews with future dates: {len(future_reviews)} reviews")

    # Check for suspicious review patterns
    if "submitted_date" in df.columns and "product_ref" in df.columns:
        review_dt = pd.to_datetime(df["submitted_date"], errors="coerce")
        df["review_date_only"] = review_dt.dt.date

        # Products with many reviews on same day
        same_day_reviews = df.groupby(["product_ref", "review_date_only"]).size()
        suspicious_products = same_day_reviews[same_day_reviews > 5]
        if len(suspicious_products) > 0:
            print(
                f"✗ Products with >5 reviews on same day (suspicious): {len(suspicious_products.index.get_level_values(0).unique())} products"
            )

    # Check for invalid foreign keys
    if "product_ref" in df.columns:
        invalid_products = df[
            df["product_ref"].isin(["INVALID", "NULL", "N/A", "", "REMOVED"])
        ]
        if len(invalid_products) > 0:
            print(f"✗ Invalid product references: {len(invalid_products)} rows")

    if "customer_ref" in df.columns:
        invalid_customers = df[
            df["customer_ref"].isin(["INVALID", "GUEST", "ANONYMOUS", ""])
        ]
        if len(invalid_customers) > 0:
            print(f"✗ Invalid customer references: {len(invalid_customers)} rows")

    # Show review statistics
    print("\n" + "=" * 50)
    print("REVIEW STATISTICS:")
    print("=" * 50)

    if "star_rating" in df.columns:
        rating_numeric = pd.to_numeric(df["star_rating"], errors="coerce")
        valid_ratings = rating_numeric[(rating_numeric >= 1) & (rating_numeric <= 5)]

        if len(valid_ratings) > 0:
            print(f"Rating distribution:")
            for rating in [5, 4, 3, 2, 1]:
                count = (valid_ratings == rating).sum()
                print(
                    f"  {rating} stars: {count} ({count/len(valid_ratings)*100:.1f}%)"
                )

            print(f"\nRating metrics:")
            print(f"  Average rating: {valid_ratings.mean():.2f}")
            print(f"  Median rating: {valid_ratings.median():.1f}")

    # Verified purchase analysis
    if "verified_purchase" in df.columns:
        verified = df["verified_purchase"].isin([True, "Y", "Yes", "1", "true"]).sum()
        not_verified = (
            df["verified_purchase"].isin([False, "N", "No", "0", "false"]).sum()
        )
        if (verified + not_verified) > 0:
            print(
                f"\nVerified purchase rate: {verified/(verified+not_verified)*100:.1f}%"
            )

    # Review length analysis
    if "review_content" in df.columns:
        review_lengths = df["review_content"].str.len()
        valid_lengths = review_lengths[review_lengths > 0]

        if len(valid_lengths) > 0:
            print(f"\nReview text analysis:")
            print(f"  Average length: {valid_lengths.mean():.0f} characters")
            print(f"  Shortest review: {valid_lengths.min()} characters")
            print(f"  Longest review: {valid_lengths.max()} characters")
            print(f"  Empty/minimal reviews: {(review_lengths <= 10).sum()}")

    # Product review analysis
    print("\n" + "=" * 50)
    print("PRODUCT REVIEW ANALYSIS:")
    print("=" * 50)

    if "product_ref" in df.columns:
        product_reviews = (
            df.groupby("product_ref")
            .agg(
                {
                    "review_ref": "count",
                    "star_rating": lambda x: pd.to_numeric(x, errors="coerce").mean(),
                }
            )
            .rename(columns={"review_ref": "review_count", "star_rating": "avg_rating"})
        )

        # Filter out invalid products
        valid_products = product_reviews[
            ~product_reviews.index.isin(["INVALID", "NULL", "N/A", "", "REMOVED"])
        ]

        if len(valid_products) > 0:
            top_reviewed = valid_products.nlargest(5, "review_count")
            print(f"Top 5 most reviewed products:")
            for idx, row in top_reviewed.iterrows():
                print(
                    f"  {idx}: {row['review_count']} reviews, {row['avg_rating']:.1f} avg rating"
                )

            # Products with suspicious ratings
            all_5_star = valid_products[
                (valid_products["avg_rating"] == 5)
                & (valid_products["review_count"] > 5)
            ]
            if len(all_5_star) > 0:
                print(
                    f"\nSuspicious: Products with all 5-star ratings (>5 reviews): {len(all_5_star)}"
                )

    # Helpfulness analysis
    if "helpful_count" in df.columns:
        helpful_numeric = pd.to_numeric(df["helpful_count"], errors="coerce")
        helpful_reviews = helpful_numeric[helpful_numeric > 0]

        if len(helpful_reviews) > 0:
            print(f"\nHelpfulness metrics:")
            print(f"  Reviews with helpful votes: {len(helpful_reviews)}")
            print(f"  Average helpful votes: {helpful_reviews.mean():.1f}")
            print(f"  Most helpful: {helpful_reviews.max()} votes")

    # Show main columns analysis
    print("\n" + "=" * 50)
    print("MAIN COLUMNS ANALYSIS:")
    print("=" * 50)

    main_columns = [
        "review_ref",
        "product_ref",
        "customer_ref",
        "star_rating",
        "review_headline",
        "review_content",
        "submitted_date",
        "verified_purchase",
    ]

    for col in main_columns:
        if col in df.columns:
            null_count = df[col].isnull().sum()
            unique_count = df[col].nunique()
            print(f"\n{col}:")
            print(f"  - Null values: {null_count} ({null_count/len(df)*100:.1f}%)")
            print(f"  - Unique values: {unique_count}")

    # Save to CSV
    output_file = "messy_reviews_data.xlsx"
    df.to_excel(output_file, index=False)
    print(f"\n✅ Dataset saved to '{output_file}'")

    # Save review summary by rating
    review_summary = []

    if "star_rating" in df.columns and "verified_purchase" in df.columns:
        rating_numeric = pd.to_numeric(df["star_rating"], errors="coerce")

        for rating in [5, 4, 3, 2, 1]:
            rating_reviews = df[rating_numeric == rating]

            verified_count = (
                rating_reviews["verified_purchase"]
                .isin([True, "Y", "Yes", "1", "true"])
                .sum()
            )
            avg_length = (
                rating_reviews["review_content"].str.len().mean()
                if "review_content" in df.columns
                else 0
            )

            review_summary.append(
                {
                    "rating": rating,
                    "count": len(rating_reviews),
                    "verified_count": verified_count,
                    "verified_pct": (
                        round(verified_count / len(rating_reviews) * 100, 2)
                        if len(rating_reviews) > 0
                        else 0
                    ),
                    "avg_text_length": round(avg_length, 0),
                }
            )

    if review_summary:
        summary_df = pd.DataFrame(review_summary)
        summary_df.to_csv("reviews_rating_summary.csv", index=False)
        print(f"✅ Review rating summary saved to 'reviews_rating_summary.csv'")

Generating 3000 rows of messy reviews data...


/tmp/ipykernel_9007/3094860077.py:513: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)



Dataset shape: (3105, 16)

Column names (realistic but challenging for mapping):
  1. review_ref
  2. product_ref
  3. customer_ref
  4. star_rating
  5. review_headline
  6. review_content
  7. submitted_date
  8. verified_purchase
  9. moderation_status
  10. submission_device
  11. helpful_count
  12. total_votes
  13. image_count
  14. review_language
  15. reviewer_name
  16. seller_response

First 10 rows:
   review_ref product_ref customer_ref star_rating        review_headline  \
0    101338     PROD_0016   CUST_01336           4         Great product!   
1      100419   PROD_0020   CUST_01253         1             Not worth it   
2      100887   PROD_0103   CUST_01390           1           Disappointed   
3      102843   PROD_0002   CUST_01169           4      Excellent quality   
4      100604   PROD_0223   CUST_01272           1           At nesciunt.   
5      102798   PROD_0239   CUST_01459           1           Poor quality   
6      101452   PROD_0001   CUST_01289      

### Marketing Campaigns Generator

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta, date
import random
import string

# Initialize Faker
fake = Faker(['en_US', 'en_GB'])
Faker.seed(42)
np.random.seed(42)
random.seed(42)

def generate_messy_marketing_campaigns_data(num_rows=500):
    """
    Generate a messy, realistic marketing campaigns dataset with realistic but challenging column names
    and various data quality issues for testing data mapping and cleaning.
    
    Args:
        num_rows: Number of campaign records to generate
    """
    
    data = []
    
    # Track IDs for creating duplicates
    used_campaign_ids = []
    
    # Choose one consistent campaign ID format
    campaign_id_format_choice = random.choice(['CMP', 'CAMPAIGN', 'NUMBER'])
    
    # Campaign types and channels
    campaign_types = ['Email', 'Social Media', 'PPC', 'Display', 'SEO', 'Content', 'Affiliate', 
                     'Influencer', 'Video', 'SMS', 'Direct Mail', 'Radio', 'TV', 'Podcast']
    
    # Campaign name templates
    campaign_themes = ['Summer Sale', 'Black Friday', 'Christmas Special', 'New Year Deal',
                      'Spring Collection', 'Back to School', 'Flash Sale', 'Clearance',
                      'Product Launch', 'Brand Awareness', 'Customer Retention', 'Lead Generation',
                      'Holiday Special', 'Anniversary Sale', 'VIP Exclusive', 'Referral Program']
    
    # Target audience templates
    audience_segments = [
        'Women, 25-40, USA', 'Men, 18-35, Urban', 'Parents with children', 'High income households',
        'College students', 'Senior citizens 65+', 'Millennials, Tech-savvy', 'Gen Z, Social media users',
        'B2B Decision makers', 'Small business owners', 'Fitness enthusiasts', 'Fashion conscious',
        'Budget shoppers', 'Premium customers', 'First-time buyers', 'Loyal customers'
    ]
    
    for i in range(num_rows):
        record = {}
        
        # Campaign ID with CONSISTENT format
        if i % 50 == 0 and used_campaign_ids:  # 2% duplicates
            campaign_id = random.choice(used_campaign_ids)
        else:
            if campaign_id_format_choice == 'CMP':
                campaign_id = f"CMP{str(i + 1).zfill(3)}"
            elif campaign_id_format_choice == 'CAMPAIGN':
                campaign_id = f"CAMPAIGN-{str(i + 1).zfill(3)}"
            else:  # NUMBER
                campaign_id = str(10000 + i)
            
            used_campaign_ids.append(campaign_id)
        
        record['campaign_ref'] = campaign_id if i % 100 != 0 else None  # 1% null IDs
        
        # Campaign Name
        if i % 25 == 0:
            name = None
        elif i % 35 == 0:  # Empty or minimal
            name = random.choice(['', ' ', 'Test', 'Campaign', 'N/A'])
        elif i % 45 == 0:  # Very long name
            name = fake.text(max_nb_chars=500)[:255]
        elif i % 55 == 0:  # Special characters
            name = random.choice(['Campaign #1', 'Sale!!!', '50% OFF', '⭐ Special ⭐', 'MEGA SALE'])
        elif i % 65 == 0:  # Duplicate names
            name = 'Summer Sale 2025'
        else:
            theme = random.choice(campaign_themes)
            year = random.choice(['2024', '2025', '2023'])
            suffix = random.choice(['', ' - Phase 1', ' - Test', ' - Final', ' v2'])
            name = f"{theme} {year}{suffix}"
        
        record['campaign_title'] = name
        
        # Campaign Type
        if i % 20 == 0:
            camp_type = None
        elif i % 30 == 0:  # Inconsistent values
            camp_type = random.choice(['email', 'EMAIL', 'Email Marketing', 'E-mail', 'EM'])
        elif i % 40 == 0:  # Invalid values
            camp_type = random.choice(['Unknown', 'Mixed', 'Other', 'N/A', ''])
        elif i % 50 == 0:  # Typos
            camp_type = random.choice(['Emal', 'Socail Media', 'PPG', 'Displya'])
        elif i % 60 == 0:  # Multiple types combined
            camp_type = 'Email + Social + PPC'
        else:
            camp_type = random.choice(campaign_types)
        
        record['channel_type'] = camp_type
        
        # Start Date
        if i % 30 == 0:
            start_date = None
        elif i % 40 == 0:  # String format variations
            start_dt = fake.date_between(start_date='-1y', end_date='+6m')
            formats = ['%Y-%m-%d', '%m/%d/%Y', '%d-%m-%Y', '%Y%m%d']
            start_date = start_dt.strftime(random.choice(formats))
        elif i % 50 == 0:  # Invalid dates
            start_date = random.choice(['0000-00-00', '2025-13-45', 'TBD', 'ASAP'])
        elif i % 60 == 0:  # Very old campaigns
            start_date = fake.date_between(start_date='-10y', end_date='-5y')
        elif i % 70 == 0:  # Far future campaigns
            start_date = fake.date_between(start_date='+2y', end_date='+5y')
        else:
            start_date = fake.date_between(start_date='-6m', end_date='+3m')
        
        record['launch_date'] = start_date
        
        # End Date (should be after start date)
        if i % 25 == 0:
            end_date = None
        elif i % 35 == 0:  # Business logic violation: end before start
            if isinstance(start_date, date):
                end_date = start_date - timedelta(days=random.randint(1, 30))
            else:
                end_date = fake.date_between(start_date='-2y', end_date='-1y')
        elif i % 45 == 0:  # Same as start date (single day campaign)
            end_date = start_date
        elif i % 55 == 0:  # Never ending campaign
            end_date = fake.date_between(start_date='+10y', end_date='+20y')
        elif i % 65 == 0:  # String format
            end_dt = fake.date_between(start_date='-3m', end_date='+6m')
            end_date = end_dt.strftime('%m/%d/%Y')
        else:
            if isinstance(start_date, date):
                # Realistic campaign duration (1 day to 3 months)
                duration = random.choices(
                    [random.randint(1, 7), random.randint(8, 30), random.randint(31, 90)],
                    weights=[30, 50, 20],
                    k=1
                )[0]
                end_date = start_date + timedelta(days=duration)
            else:
                end_date = fake.date_between(start_date='-2m', end_date='+6m')
        
        record['completion_date'] = end_date
        
        # Budget
        if i % 20 == 0:
            budget = None
        elif i % 30 == 0:  # String values
            budget = random.choice(['Unlimited', 'TBD', 'Variable', 'N/A', ''])
        elif i % 40 == 0:  # Negative budget
            budget = round(random.uniform(-10000, -100), 2)
        elif i % 50 == 0:  # Zero budget
            budget = 0
        elif i % 60 == 0:  # Extreme budgets
            budget = random.choice([999999999.99, 0.01, -99999])
        else:
            # Realistic budget based on campaign type
            if camp_type in ['TV', 'Radio']:
                budget = round(random.uniform(50000, 500000), 2)
            elif camp_type in ['PPC', 'Display']:
                budget = round(random.uniform(1000, 50000), 2)
            elif camp_type in ['Email', 'SMS']:
                budget = round(random.uniform(100, 10000), 2)
            else:
                budget = round(random.uniform(500, 25000), 2)
        
        record['allocated_budget'] = budget
        
        # Spent Amount (should be <= budget)
        if i % 25 == 0:
            spent = None
        elif i % 35 == 0:  # String values
            spent = random.choice(['In Progress', 'Calculating', 'N/A'])
        elif i % 45 == 0:  # Negative spent
            spent = round(random.uniform(-5000, -10), 2)
        elif i % 55 == 0:  # Spent > budget (business logic violation)
            if isinstance(budget, (int, float)) and budget > 0:
                spent = round(budget * random.uniform(1.1, 2.0), 2)
            else:
                spent = round(random.uniform(10000, 50000), 2)
        elif i % 65 == 0:  # Zero spent but campaign completed
            spent = 0
            record['campaign_status'] = 'Completed'  # Will be overwritten later
        else:
            if isinstance(budget, (int, float)) and budget > 0:
                # Realistic spending based on campaign status
                if isinstance(start_date, date) and isinstance(end_date, date):
                    today = date.today()
                    if end_date < today:  # Completed
                        spent = round(budget * random.uniform(0.7, 1.0), 2)
                    elif start_date > today:  # Not started
                        spent = 0
                    else:  # Active
                        progress = (today - start_date).days / max((end_date - start_date).days, 1)
                        spent = round(budget * progress * random.uniform(0.8, 1.2), 2)
                else:
                    spent = round(budget * random.uniform(0.3, 0.9), 2)
            else:
                spent = round(random.uniform(100, 10000), 2)
        
        record['current_spend'] = spent
        
        # Impressions
        if i % 30 == 0:
            impressions = None
        elif i % 40 == 0:  # String values
            impressions = random.choice(['Many', 'High', 'Low', 'N/A'])
        elif i % 50 == 0:  # Negative impressions
            impressions = random.randint(-10000, -1)
        elif i % 60 == 0:  # Zero impressions but has clicks (violation)
            impressions = 0
        elif i % 70 == 0:  # Extreme values
            impressions = random.choice([999999999, 0.5, -99999])
        else:
            # Realistic impressions based on budget and type
            if isinstance(spent, (int, float)) and spent > 0:
                if camp_type in ['Email', 'SMS']:
                    impressions = int(spent * random.uniform(10, 50))  # Lower CPM
                elif camp_type in ['Display', 'Social Media']:
                    impressions = int(spent * random.uniform(100, 500))  # Higher reach
                else:
                    impressions = int(spent * random.uniform(50, 200))
            else:
                impressions = random.randint(1000, 100000)
        
        record['total_impressions'] = impressions
        
        # Clicks (should be <= impressions)
        if i % 25 == 0:
            clicks = None
        elif i % 35 == 0:  # String values
            clicks = random.choice(['Good CTR', 'Low', 'N/A'])
        elif i % 45 == 0:  # Negative clicks
            clicks = random.randint(-1000, -1)
        elif i % 55 == 0:  # Clicks > impressions (business logic violation)
            if isinstance(impressions, int) and impressions > 0:
                clicks = int(impressions * random.uniform(1.1, 2.0))
            else:
                clicks = random.randint(10000, 50000)
        elif i % 65 == 0:  # Decimal clicks
            clicks = random.choice([100.5, 250.75, 1000.25])
        else:
            if isinstance(impressions, int) and impressions > 0:
                # Realistic CTR (0.5% - 5%)
                ctr = random.uniform(0.005, 0.05)
                clicks = int(impressions * ctr)
            else:
                clicks = random.randint(10, 5000)
        
        record['total_clicks'] = clicks
        
        # Conversions (should be <= clicks)
        if i % 30 == 0:
            conversions = None
        elif i % 40 == 0:  # String values
            conversions = random.choice(['Good', 'Poor', 'TBD', 'N/A'])
        elif i % 50 == 0:  # Negative conversions
            conversions = random.randint(-100, -1)
        elif i % 60 == 0:  # Conversions > clicks (business logic violation)
            if isinstance(clicks, (int, float)) and clicks > 0:
                conversions = int(clicks * random.uniform(1.1, 2.0))
            else:
                conversions = random.randint(1000, 5000)
        elif i % 70 == 0:  # 100% conversion rate (suspicious)
            conversions = clicks
        else:
            if isinstance(clicks, (int, float)) and clicks > 0:
                # Realistic conversion rate (1% - 10%)
                conv_rate = random.uniform(0.01, 0.10)
                conversions = int(clicks * conv_rate)
            else:
                conversions = random.randint(0, 500)
        
        record['conversion_count'] = conversions
        
        # Target Audience
        if i % 25 == 0:
            audience = None
        elif i % 35 == 0:  # Empty or minimal
            audience = random.choice(['', 'Everyone', 'All', 'N/A'])
        elif i % 45 == 0:  # Very long description
            audience = fake.text(max_nb_chars=1000)
        elif i % 55 == 0:  # Invalid/unclear
            audience = random.choice(['TBD', 'See brief', 'Multiple segments', '???'])
        elif i % 65 == 0:  # Special characters
            audience = 'Target: 18-65+, $$$, A/B test'
        else:
            # Generate realistic audience description
            segments = random.sample(audience_segments, random.randint(1, 3))
            audience = '; '.join(segments)
        
        record['target_segment'] = audience
        
        # Campaign Status
        if i % 20 == 0:
            status = None
        elif i % 30 == 0:  # Inconsistent values
            status = random.choice(['active', 'ACTIVE', 'Running', 'Live', '1'])
        elif i % 40 == 0:  # Invalid values
            status = random.choice(['Pending', 'Draft', 'Archived', 'Deleted'])
        elif i % 50 == 0:  # Typos
            status = random.choice(['Activ', 'Pasued', 'Complted'])
        else:
            # Determine status based on dates
            if isinstance(start_date, date) and isinstance(end_date, date):
                today = date.today()
                if start_date > today:
                    status = random.choice(['Scheduled', 'Pending', 'Draft'])
                elif end_date < today:
                    status = 'Completed'
                else:
                    status = random.choice(['Active', 'Active', 'Paused'])  # More likely active
            else:
                status = random.choice(['Active', 'Paused', 'Completed'])
        
        record['campaign_status'] = status
        
        # Created At timestamp
        if i % 30 == 0:
            created = None
        elif i % 40 == 0:  # String format
            created = fake.date_time_between(start_date='-1y', end_date='now').strftime('%Y-%m-%d %H:%M:%S')
        elif i % 50 == 0:  # Created after start date (business logic violation)
            if isinstance(start_date, date):
                created = fake.date_time_between(
                    start_date=start_date + timedelta(days=1),
                    end_date=start_date + timedelta(days=30)
                )
            else:
                created = fake.date_time_between(start_date='+1d', end_date='+30d')
        else:
            if isinstance(start_date, date):
                # Created before start date
                created = fake.date_time_between(
                    start_date=start_date - timedelta(days=30),
                    end_date=start_date - timedelta(days=1)
                )
            else:
                created = fake.date_time_between(start_date='-1y', end_date='now')
        
        record['created_timestamp'] = created
        
        # Additional realistic columns that might exist
        
        # ROI (Return on Investment)
        if random.random() > 0.5:  # 50% have ROI
            if all(isinstance(x, (int, float)) for x in [spent, conversions]) and spent > 0:
                # Assume average order value
                avg_order_value = random.uniform(50, 200)
                revenue = conversions * avg_order_value
                roi = round(((revenue - spent) / spent) * 100, 2)
                
                if i % 75 == 0:  # Extreme ROI
                    roi = random.choice([-100, 1000, 99999])
                elif i % 85 == 0:  # String value
                    roi = random.choice(['Positive', 'Negative', 'Break-even'])
            else:
                roi = None
            record['roi_percentage'] = roi
        
        # CTR (Click-through rate)
        if random.random() > 0.6:  # 40% have CTR
            if all(isinstance(x, (int, float)) for x in [clicks, impressions]) and impressions > 0:
                ctr = round((clicks / impressions) * 100, 2)
                if i % 80 == 0:  # Invalid CTR > 100%
                    ctr = random.uniform(101, 200)
            elif i % 70 == 0:
                ctr = random.choice(['High', 'Low', 'N/A'])
            else:
                ctr = None
            record['ctr_rate'] = ctr
        
        # Conversion Rate
        if random.random() > 0.6:  # 40% have conversion rate
            if all(isinstance(x, (int, float)) for x in [conversions, clicks]) and clicks > 0:
                conv_rate = round((conversions / clicks) * 100, 2)
                if i % 85 == 0:  # Invalid rate > 100%
                    conv_rate = random.uniform(101, 150)
            elif i % 75 == 0:
                conv_rate = random.choice(['Good', 'Poor', 'Average'])
            else:
                conv_rate = None
            record['conversion_rate'] = conv_rate
        
        # Cost per click (CPC)
        if random.random() > 0.7:  # 30% have CPC
            if all(isinstance(x, (int, float)) for x in [spent, clicks]) and clicks > 0:
                cpc = round(spent / clicks, 2)
                if i % 90 == 0:  # Extreme CPC
                    cpc = random.choice([0, 1000, -10])
            else:
                cpc = None
            record['avg_cpc'] = cpc
        
        # Campaign manager/owner
        if random.random() > 0.6:  # 40% have owner
            if i % 65 == 0:
                owner = None
            elif i % 75 == 0:
                owner = random.choice(['Team', 'Agency', 'N/A', ''])
            else:
                owner = fake.name()
            record['campaign_manager'] = owner
        
        # A/B test variant
        if random.random() > 0.7:  # 30% are A/B tests
            if i % 70 == 0:
                variant = None
            elif i % 80 == 0:
                variant = random.choice(['Control', 'Test', 'Winner'])
            else:
                variant = random.choice(['A', 'B', 'C', 'Control', 'Variant 1', 'Variant 2'])
            record['test_variant'] = variant
        
        # Platform (for digital campaigns)
        if camp_type in ['Social Media', 'PPC', 'Display', 'Video']:
            if i % 75 == 0:
                platform = None
            elif i % 85 == 0:
                platform = random.choice(['All', 'Multiple', 'N/A'])
            else:
                platform = random.choice([
                    'Google Ads', 'Facebook', 'Instagram', 'LinkedIn', 'Twitter',
                    'YouTube', 'TikTok', 'Pinterest', 'Snapchat', 'Amazon'
                ])
            record['ad_platform'] = platform
        
        data.append(record)
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Add EXACT duplicate rows (every attribute is same)
    num_exact_duplicates = int(num_rows * 0.02)  # 2% exact duplicates
    for _ in range(num_exact_duplicates):
        if len(df) > 0:
            # Pick a random row to duplicate
            row_to_duplicate = df.sample(1)
            df = pd.concat([df, row_to_duplicate], ignore_index=True)
    
    # Add some completely empty rows
    for _ in range(int(num_rows * 0.005)):  # 0.5% empty rows
        empty_row = pd.Series([None] * len(df.columns), index=df.columns)
        df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)
    
    # Add some rows with all string 'NULL' or 'N/A' values
    for _ in range(int(num_rows * 0.01)):  # 1% NULL string rows
        null_values = ['NULL', 'N/A', 'null', 'NA', '', ' ']
        null_row = pd.Series([random.choice(null_values) for _ in range(len(df.columns))], index=df.columns)
        df = pd.concat([df, pd.DataFrame([null_row])], ignore_index=True)
    
    # Shuffle the dataframe to mix duplicates throughout
    df = df.sample(frac=1).reset_index(drop=True)
    
    return df

def add_more_messiness(df):
    """
    Add additional data quality issues to make the dataset more challenging
    """
    # Add trailing/leading spaces to some string columns
    string_cols = df.select_dtypes(include=['object']).columns
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05  # 5% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: '  ' + str(x) + '  ' if pd.notna(x) else x
        )
    
    # Add case inconsistencies
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03  # 3% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x).upper() if pd.notna(x) and random.random() > 0.5 else str(x).lower() if pd.notna(x) else x
        )
    
    # Add special characters to some values
    for col in string_cols[:3]:  # Only first 3 string columns
        mask = np.random.random(len(df)) < 0.02  # 2% of values
        special_chars = ['@', '#', '!', '*', '&', '%']
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )
    
    return df

# Generate the dataset
if __name__ == "__main__":
    # Set the number of rows you want
    NUM_ROWS = 500  # Change this to your desired number
    
    print(f"Generating {NUM_ROWS} rows of messy marketing campaigns data...")
    df = generate_messy_marketing_campaigns_data(NUM_ROWS)
    
    # Add more messiness
    df = add_more_messiness(df)
    
    # Display basic info
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumn names (realistic but challenging for mapping):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")
    
    print(f"\nFirst 10 rows:")
    print(df.head(10))
    
    # Show data quality issues summary
    print("\n" + "="*50)
    print("DATA QUALITY ISSUES SUMMARY:")
    print("="*50)
    print(f"Total null values: {df.isnull().sum().sum()}")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    print(f"Total rows: {len(df)}")
    
    # Show business logic violations
    print("\n" + "="*50)
    print("BUSINESS LOGIC VIOLATIONS EXAMPLES:")
    print("="*50)
    
    # Check for end date before start date
    if 'launch_date' in df.columns and 'completion_date' in df.columns:
        start_dt = pd.to_datetime(df['launch_date'], errors='coerce')
        end_dt = pd.to_datetime(df['completion_date'], errors='coerce')
        invalid_dates = df[end_dt < start_dt]
        if len(invalid_dates) > 0:
            print(f"✗ End date before start date: {len(invalid_dates)} campaigns")
    
    # Check for spent > budget
    if 'allocated_budget' in df.columns and 'current_spend' in df.columns:
        budget_numeric = pd.to_numeric(df['allocated_budget'], errors='coerce')
        spent_numeric = pd.to_numeric(df['current_spend'], errors='coerce')
        overspent = df[spent_numeric > budget_numeric]
        if len(overspent) > 0:
            print(f"✗ Spent amount > budget: {len(overspent)} campaigns")
            overspend_total = (spent_numeric - budget_numeric)[overspent.index].sum()
            print(f"  Total overspend: ${overspend_total:,.2f}")
    
    # Check for clicks > impressions
    if 'total_impressions' in df.columns and 'total_clicks' in df.columns:
        impressions_numeric = pd.to_numeric(df['total_impressions'], errors='coerce')
        clicks_numeric = pd.to_numeric(df['total_clicks'], errors='coerce')
        invalid_clicks = df[clicks_numeric > impressions_numeric]
        if len(invalid_clicks) > 0:
            print(f"✗ Clicks > impressions: {len(invalid_clicks)} campaigns")
    
    # Check for conversions > clicks
    if 'total_clicks' in df.columns and 'conversion_count' in df.columns:
        clicks_numeric = pd.to_numeric(df['total_clicks'], errors='coerce')
        conversions_numeric = pd.to_numeric(df['conversion_count'], errors='coerce')
        invalid_conversions = df[conversions_numeric > clicks_numeric]
        if len(invalid_conversions) > 0:
            print(f"✗ Conversions > clicks: {len(invalid_conversions)} campaigns")
    
    # Check for negative values
    for col in ['allocated_budget', 'current_spend', 'total_impressions', 'total_clicks', 'conversion_count']:
        if col in df.columns:
            col_numeric = pd.to_numeric(df[col], errors='coerce')
            negative_values = df[col_numeric < 0]
            if len(negative_values) > 0:
                print(f"✗ Negative {col}: {len(negative_values)} campaigns")
    
    # Check for CTR > 100%
    if 'ctr_rate' in df.columns:
        ctr_numeric = pd.to_numeric(df['ctr_rate'], errors='coerce')
        invalid_ctr = df[ctr_numeric > 100]
        if len(invalid_ctr) > 0:
            print(f"✗ CTR > 100%: {len(invalid_ctr)} campaigns")
    
    # Check for suspicious 100% conversion rates
    if 'total_clicks' in df.columns and 'conversion_count' in df.columns:
        clicks_numeric = pd.to_numeric(df['total_clicks'], errors='coerce')
        conversions_numeric = pd.to_numeric(df['conversion_count'], errors='coerce')
        perfect_conversion = df[
            (clicks_numeric > 10) &  # More than 10 clicks
            (conversions_numeric == clicks_numeric)  # 100% conversion
        ]
        if len(perfect_conversion) > 0:
            print(f"✗ Suspicious 100% conversion rate (>10 clicks): {len(perfect_conversion)} campaigns")
    
    # Show campaign performance statistics
    print("\n" + "="*50)
    print("CAMPAIGN PERFORMANCE STATISTICS:")
    print("="*50)
    
    # Budget analysis
    if 'allocated_budget' in df.columns and 'current_spend' in df.columns:
        budget_numeric = pd.to_numeric(df['allocated_budget'], errors='coerce')
        spent_numeric = pd.to_numeric(df['current_spend'], errors='coerce')
        
        valid_budget = budget_numeric[budget_numeric > 0]
        valid_spent = spent_numeric[spent_numeric >= 0]
        
        if len(valid_budget) > 0:
            print(f"Budget metrics:")
            print(f"  Total budget allocated: ${valid_budget.sum():,.2f}")
            print(f"  Average campaign budget: ${valid_budget.mean():,.2f}")
            print(f"  Total spent: ${valid_spent.sum():,.2f}")
            
            # Budget utilization
            both_valid = df[(budget_numeric > 0) & (spent_numeric >= 0)]
            if len(both_valid) > 0:
                utilization = (spent_numeric[both_valid.index] / budget_numeric[both_valid.index] * 100).mean()
                print(f"  Average budget utilization: {utilization:.1f}%")
    
    # CTR and Conversion metrics
    if all(col in df.columns for col in ['total_impressions', 'total_clicks', 'conversion_count']):
        impressions_numeric = pd.to_numeric(df['total_impressions'], errors='coerce')
        clicks_numeric = pd.to_numeric(df['total_clicks'], errors='coerce')
        conversions_numeric = pd.to_numeric(df['conversion_count'], errors='coerce')
        
        # Calculate average CTR
        valid_ctr_data = df[(impressions_numeric > 0) & (clicks_numeric >= 0)]
        if len(valid_ctr_data) > 0:
            avg_ctr = (clicks_numeric[valid_ctr_data.index] / impressions_numeric[valid_ctr_data.index] * 100).mean()
            print(f"\nPerformance metrics:")
            print(f"  Average CTR: {avg_ctr:.2f}%")
        
        # Calculate average conversion rate
        valid_conv_data = df[(clicks_numeric > 0) & (conversions_numeric >= 0)]
        if len(valid_conv_data) > 0:
            avg_conv_rate = (conversions_numeric[valid_conv_data.index] / clicks_numeric[valid_conv_data.index] * 100).mean()
            print(f"  Average conversion rate: {avg_conv_rate:.2f}%")
        
        # Total metrics
        print(f"  Total impressions: {impressions_numeric[impressions_numeric > 0].sum():,.0f}")
        print(f"  Total clicks: {clicks_numeric[clicks_numeric > 0].sum():,.0f}")
        print(f"  Total conversions: {conversions_numeric[conversions_numeric > 0].sum():,.0f}")
    
    # Campaign type distribution
    if 'channel_type' in df.columns:
        type_counts = df['channel_type'].value_counts()
        print(f"\nCampaign type distribution:")
        for camp_type, count in type_counts.head(5).items():
            if camp_type and camp_type not in ['Unknown', 'Mixed', 'Other', 'N/A', '']:
                print(f"  {camp_type}: {count} ({count/len(df)*100:.1f}%)")
    
    # Campaign status distribution
    if 'campaign_status' in df.columns:
        status_counts = df['campaign_status'].value_counts()
        print(f"\nCampaign status distribution:")
        for status, count in status_counts.head(5).items():
            if status:
                print(f"  {status}: {count} ({count/len(df)*100:.1f}%)")
    
    # ROI analysis
    if 'roi_percentage' in df.columns:
        roi_numeric = pd.to_numeric(df['roi_percentage'], errors='coerce')
        valid_roi = roi_numeric[roi_numeric.notna()]
        
        if len(valid_roi) > 0:
            positive_roi = (valid_roi > 0).sum()
            negative_roi = (valid_roi < 0).sum()
            
            print(f"\nROI analysis:")
            print(f"  Campaigns with positive ROI: {positive_roi} ({positive_roi/len(valid_roi)*100:.1f}%)")
            print(f"  Campaigns with negative ROI: {negative_roi} ({negative_roi/len(valid_roi)*100:.1f}%)")
            print(f"  Average ROI: {valid_roi.mean():.1f}%")
    
    # Show main columns analysis
    print("\n" + "="*50)
    print("MAIN COLUMNS ANALYSIS:")
    print("="*50)
    
    main_columns = [
        'campaign_ref', 'campaign_title', 'channel_type', 'launch_date',
        'completion_date', 'allocated_budget', 'current_spend', 'total_impressions',
        'total_clicks', 'conversion_count', 'campaign_status'
    ]
    
    for col in main_columns:
        if col in df.columns:
            null_count = df[col].isnull().sum()
            unique_count = df[col].nunique()
            print(f"\n{col}:")
            print(f"  - Null values: {null_count} ({null_count/len(df)*100:.1f}%)")
            print(f"  - Unique values: {unique_count}")
    
    # Save to CSV
    output_file = 'messy_marketing_campaigns_data.xlsx'
    df.to_excel(output_file, index=False)
    print(f"\n✅ Dataset saved to '{output_file}'")
    
    # Save campaign performance summary
    # campaign_summary = []
    
    # if all(col in df.columns for col in ['channel_type', 'allocated_budget', 'current_spend', 'conversion_count']):
    #     for channel in df['channel_type'].dropna().unique()[:10]:
    #         channel_campaigns = df[df['channel_type'] == channel]
            
    #         budget_sum = pd.to_numeric(channel_campaigns['allocated_budget'], errors='coerce').sum()
    #         spent_sum = pd.to_numeric(channel_campaigns['current_spend'], errors='coerce').sum()
    #         conversions_sum = pd.to_numeric(channel_campaigns['conversion_count'], errors='coerce').sum()
            
    #         campaign_summary.append({
    #             'channel': channel,
    #             'campaign_count': len(channel_campaigns),
    #             'total_budget': round(budget_sum, 2),
    #             'total_spent': round(spent_sum, 2),
    #             'total_conversions': conversions_sum,
    #             'avg_budget': round(budget_sum/len(channel_campaigns), 2) if len(channel_campaigns) > 0 else 0
    #         })
    
    # if campaign_summary:
    #     summary_df = pd.DataFrame(campaign_summary)
    #     summary_df.to_csv('campaigns_channel_summary.csv', index=False)
    #     print(f"✅ Campaign channel summary saved to 'campaigns_channel_summary.csv'")

Generating 500 rows of messy marketing campaigns data...

Dataset shape: (517, 20)

Column names (realistic but challenging for mapping):
  1. campaign_ref
  2. campaign_title
  3. channel_type
  4. launch_date
  5. completion_date
  6. allocated_budget
  7. current_spend
  8. total_impressions
  9. total_clicks
  10. conversion_count
  11. target_segment
  12. campaign_status
  13. created_timestamp
  14. ctr_rate
  15. avg_cpc
  16. campaign_manager
  17. test_variant
  18. roi_percentage
  19. conversion_rate
  20. ad_platform

First 10 rows:
  campaign_ref                   campaign_title channel_type launch_date  \
0     10390  !               Summer Sale 2025          EMAIL        None   
1        10018          Customer Retention 2024          SEO  2025-09-10   
2        10121     Lead Generation 2025 - Final           TV  2025-09-10   
3        10196             Holiday Special 2025   Influencer  2025-09-10   
4        10040     VIP Exclusive 2024 - Phase 1         None    2024

/tmp/ipykernel_7262/2361159400.py:448: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)



✅ Dataset saved to 'messy_marketing_campaigns_data.xlsx'


### Customer Sessions Table Generator

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import string
import uuid

# Initialize Faker
fake = Faker(["en_US", "en_GB"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)


def generate_messy_customer_sessions_data(num_rows=5000, customer_id_format="CUST"):
    """
    Generate a messy, realistic customer sessions dataset with realistic but challenging column names
    and various data quality issues for testing data mapping and cleaning.

    Args:
        num_rows: Number of session records to generate
        customer_id_format: Format of customer IDs ('CUST', 'CUSTOMER', or 'NUMBER')
    """

    data = []

    # Track IDs for creating duplicates and relationships
    used_session_ids = []

    # Generate pool of customer IDs
    num_customers = max(num_rows // 10, 200)  # Average 10 sessions per customer

    customer_ids = []
    for i in range(num_customers):
        if customer_id_format == "CUST":
            cust_id = f"CUST_{str(i + 1000).zfill(5)}"
        elif customer_id_format == "CUSTOMER":
            cust_id = f"CUSTOMER-{str(i + 1000).zfill(5)}"
        else:  # NUMBER
            cust_id = str(10000 + i)
        customer_ids.append(cust_id)

    # Choose one consistent session ID format
    session_id_format_choice = random.choice(["SESS", "SESSION", "UUID"])

    # Referrer sources (matching marketing channels)
    referrer_sources = [
        "Google Ads",
        "Facebook Ads",
        "Instagram",
        "Organic Search",
        "Direct",
        "Email Campaign",
        "Affiliate",
        "Social Media",
        "Referral",
        "Display Ads",
        "YouTube",
        "TikTok",
        "LinkedIn",
        "Twitter",
        "Pinterest",
        "Reddit",
    ]

    # Track customer behavior patterns
    customer_behavior = {}

    for i in range(num_rows):
        record = {}

        # Session ID with CONSISTENT format
        if i % 50 == 0 and used_session_ids:  # 2% duplicates
            session_id = random.choice(used_session_ids)
        else:
            if session_id_format_choice == "SESS":
                session_id = f"SESS{str(i + 1).zfill(5)}"
            elif session_id_format_choice == "SESSION":
                session_id = f"SESSION-{str(i + 1).zfill(5)}"
            else:  # UUID
                session_id = str(uuid.uuid4())[:8].upper()

            used_session_ids.append(session_id)

        record["session_ref"] = session_id if i % 100 != 0 else None  # 1% null IDs

        # Customer ID (null for guest users)
        is_guest = random.random() < 0.35  # 35% guest sessions

        if is_guest:
            cust_id = None
        elif i % 60 == 0:  # Non-existent customer IDs (FK violation)
            if customer_id_format == "CUST":
                cust_id = f"CUST_{str(99999).zfill(5)}"
            elif customer_id_format == "CUSTOMER":
                cust_id = f"CUSTOMER-{str(99999).zfill(5)}"
            else:
                cust_id = "99999"
        elif i % 40 == 0:  # Invalid format
            cust_id = random.choice(["INVALID", "GUEST", "ANONYMOUS", "", "N/A"])
        else:
            cust_id = random.choice(customer_ids)

            # Track returning customers
            if cust_id not in customer_behavior:
                customer_behavior[cust_id] = {"sessions": 0, "converted": False}
            customer_behavior[cust_id]["sessions"] += 1

        record["user_id"] = cust_id

        # Session Start
        if i % 25 == 0:
            session_start = None
        elif i % 35 == 0:  # String format variations
            start_dt = fake.date_time_between(start_date="-30d", end_date="now")
            formats = ["%Y-%m-%d %H:%M:%S", "%m/%d/%Y %H:%M", "%d-%m-%Y %H:%M"]
            session_start = start_dt.strftime(random.choice(formats))
        elif i % 45 == 0:  # Unix timestamp
            session_start = int(
                fake.date_time_between(start_date="-30d", end_date="now").timestamp()
            )
        elif i % 55 == 0:  # Future session (business logic violation)
            session_start = fake.date_time_between(start_date="+1d", end_date="+7d")
        elif i % 65 == 0:  # Very old session
            session_start = fake.date_time_between(start_date="-2y", end_date="-1y")
        else:
            # Realistic distribution: more recent sessions
            days_ago = random.choices(
                [0, random.randint(1, 7), random.randint(8, 30)],
                weights=[30, 50, 20],
                k=1,
            )[0]
            session_start = datetime.now() - timedelta(
                days=days_ago,
                hours=random.randint(0, 23),
                minutes=random.randint(0, 59),
            )

        record["start_timestamp"] = session_start

        # Session End (should be after start)
        if i % 30 == 0:
            session_end = None
        elif i % 40 == 0:  # Business logic violation: end before start
            if isinstance(session_start, datetime):
                session_end = session_start - timedelta(minutes=random.randint(1, 60))
            else:
                session_end = fake.date_time_between(start_date="-32d", end_date="-31d")
        elif i % 50 == 0:  # Still active (no end time)
            session_end = None
        elif i % 60 == 0:  # Very long session (>24 hours - suspicious)
            if isinstance(session_start, datetime):
                session_end = session_start + timedelta(hours=random.randint(25, 72))
            else:
                session_end = fake.date_time_between(start_date="-28d", end_date="-27d")
        else:
            if isinstance(session_start, datetime):
                # Realistic session duration
                duration_minutes = random.choices(
                    [
                        random.randint(1, 5),  # Bounce: 1-5 min
                        random.randint(6, 15),  # Quick browse: 6-15 min
                        random.randint(16, 30),  # Normal: 16-30 min
                        random.randint(31, 60),  # Engaged: 31-60 min
                        random.randint(61, 120),
                    ],  # Very engaged: 1-2 hours
                    weights=[35, 25, 20, 15, 5],
                    k=1,
                )[0]
                session_end = session_start + timedelta(minutes=duration_minutes)
            else:
                session_end = fake.date_time_between(start_date="-29d", end_date="now")

        record["end_timestamp"] = session_end

        # Calculate session duration for behavioral metrics
        session_duration = 0
        if isinstance(session_start, datetime) and isinstance(session_end, datetime):
            session_duration = (
                session_end - session_start
            ).total_seconds() / 60  # minutes

        # Device Type
        if i % 25 == 0:
            device = None
        elif i % 35 == 0:  # Inconsistent values
            device = random.choice(
                ["mobile", "MOBILE", "Mobile Phone", "Smartphone", "Cell"]
            )
        elif i % 45 == 0:  # Invalid values
            device = random.choice(["Unknown", "Bot", "Crawler", "N/A", ""])
        elif i % 55 == 0:  # Typos
            device = random.choice(["Deskop", "Moblie", "Tablte"])
        else:
            # Realistic device distribution
            device = random.choices(
                ["Mobile", "Desktop", "Tablet", "Smart TV", "Game Console"],
                weights=[45, 40, 10, 3, 2],
                k=1,
            )[0]

        record["device_category"] = device

        # Referrer Source
        if i % 20 == 0:
            referrer = None
        elif i % 30 == 0:  # Inconsistent values
            referrer = random.choice(
                ["google", "GOOGLE", "Google.com", "Search", "Organic"]
            )
        elif i % 40 == 0:  # Invalid values
            referrer = random.choice(["Unknown", "N/A", "(not set)", "(none)", ""])
        elif i % 50 == 0:  # Typos
            referrer = random.choice(
                ["Gogle Ads", "Facebok", "Instagran", "Emai Campaign"]
            )
        elif i % 60 == 0:  # Suspicious referrer
            referrer = random.choice(
                ["spam.com", "bot-traffic", "test.localhost", "xxx.com"]
            )
        else:
            # Match with device type for realism
            if device == "Mobile":
                # Mobile users more from social/apps
                referrer = random.choices(
                    referrer_sources,
                    weights=[10, 15, 15, 5, 10, 5, 5, 15, 5, 5, 5, 2, 1, 1, 1, 0],
                    k=1,
                )[0]
            else:
                # Desktop more from search/direct
                referrer = random.choices(
                    referrer_sources,
                    weights=[15, 10, 5, 20, 20, 10, 5, 5, 5, 2, 1, 1, 1, 0, 0, 0],
                    k=1,
                )[0]

        record["traffic_source"] = referrer

        # Pages Viewed
        if i % 25 == 0:
            pages = None
        elif i % 35 == 0:  # String values
            pages = random.choice(["Many", "Few", "Single", "N/A"])
        elif i % 45 == 0:  # Negative pages
            pages = random.randint(-10, -1)
        elif i % 55 == 0:  # Zero pages but has products viewed (violation)
            pages = 0
        elif i % 65 == 0:  # Extreme values
            pages = random.choice([999, 10000, 0.5])
        else:
            # Realistic based on session duration
            if session_duration > 0:
                if session_duration < 5:  # Bounce
                    pages = random.randint(1, 2)
                elif session_duration < 15:
                    pages = random.randint(2, 5)
                elif session_duration < 30:
                    pages = random.randint(4, 10)
                else:
                    pages = random.randint(8, 30)
            else:
                pages = random.randint(1, 15)

        record["page_views"] = pages

        # Products Viewed (should be <= pages viewed)
        if i % 30 == 0:
            products = None
        elif i % 40 == 0:  # String values
            products = random.choice(["Multiple", "None", "Several", "N/A"])
        elif i % 50 == 0:  # Products > pages (business logic violation)
            if isinstance(pages, int) and pages > 0:
                products = pages + random.randint(1, 10)
            else:
                products = random.randint(20, 50)
        elif i % 60 == 0:  # Negative products
            products = random.randint(-5, -1)
        else:
            if isinstance(pages, int) and pages > 0:
                # Realistic: not every page is a product page
                max_products = max(1, int(pages * 0.6))
                products = random.randint(0, max_products)
            else:
                products = random.randint(0, 10)

        record["products_browsed"] = products

        # Conversion Flag
        if i % 35 == 0:
            conversion = None
        elif i % 45 == 0:  # Various boolean representations
            conversion = random.choice(
                ["Y", "N", "Yes", "No", "1", "0", "true", "false"]
            )
        elif i % 55 == 0:  # Invalid values
            conversion = random.choice(["Maybe", "Pending", "Unknown", ""])
        else:
            # Realistic conversion based on behavior
            if isinstance(products, int) and products > 0:
                # More products viewed = higher conversion chance
                if products >= 5:
                    conversion = random.choices([True, False], weights=[30, 70], k=1)[0]
                elif products >= 2:
                    conversion = random.choices([True, False], weights=[15, 85], k=1)[0]
                else:
                    conversion = random.choices([True, False], weights=[5, 95], k=1)[0]
            else:
                conversion = random.choices([True, False], weights=[2, 98], k=1)[0]

            # Track customer conversion
            if cust_id and cust_id in customer_behavior:
                if conversion:
                    customer_behavior[cust_id]["converted"] = True

        record["purchase_made"] = conversion

        # Cart Abandonment Flag
        if i % 40 == 0:
            abandonment = None
        elif i % 50 == 0:  # Various boolean representations
            abandonment = random.choice(["Y", "N", "Yes", "No", "1", "0"])
        elif i % 60 == 0:  # Business logic violation: both conversion and abandonment
            if conversion in [True, "Y", "Yes", "1", "true"]:
                abandonment = True  # Can't abandon if converted
            else:
                abandonment = False
        else:
            # Realistic: can't abandon if converted
            if conversion in [True, "Y", "Yes", "1", "true"]:
                abandonment = False
            else:
                # Higher chance of abandonment if viewed products
                if isinstance(products, int) and products > 0:
                    abandonment = random.choices([True, False], weights=[40, 60], k=1)[
                        0
                    ]
                else:
                    abandonment = False

        record["cart_abandoned"] = abandonment

        # Additional realistic columns that might exist

        # Bounce flag (single page visit)
        if random.random() > 0.6:  # 40% have bounce flag
            if isinstance(pages, int):
                bounce = pages == 1
                if i % 70 == 0:  # Inconsistency
                    bounce = not bounce
            else:
                bounce = None
            record["bounce_session"] = bounce

        # Session duration in seconds
        if random.random() > 0.5:  # 50% have duration
            if isinstance(session_start, datetime) and isinstance(
                session_end, datetime
            ):
                duration = int((session_end - session_start).total_seconds())
                if i % 75 == 0:  # Negative duration
                    duration = -abs(duration)
            elif i % 65 == 0:
                duration = random.choice(["Long", "Short", "N/A"])
            else:
                duration = None
            record["session_duration_sec"] = duration

        # Browser/User Agent
        if random.random() > 0.7:  # 30% have browser info
            if i % 60 == 0:
                browser = None
            elif i % 70 == 0:
                browser = random.choice(["", "N/A", "Unknown"])
            else:
                browser = random.choice(
                    [
                        "Chrome",
                        "Safari",
                        "Firefox",
                        "Edge",
                        "Opera",
                        "Chrome Mobile",
                        "Safari Mobile",
                        "Samsung Browser",
                    ]
                )
            record["browser_name"] = browser

        # Country/Location
        if random.random() > 0.6:  # 40% have location
            if i % 65 == 0:
                country = None
            elif i % 75 == 0:
                country = random.choice(["", "N/A", "Unknown"])
            else:
                country = random.choices(
                    ["US", "UK", "CA", "AU", "DE", "FR", "JP", "IN", "BR", "MX"],
                    weights=[40, 10, 8, 5, 5, 5, 5, 10, 7, 5],
                    k=1,
                )[0]
            record["geo_country"] = country

        # Landing page
        if random.random() > 0.5:  # 50% have landing page
            if i % 70 == 0:
                landing = None
            elif i % 80 == 0:
                landing = random.choice(["", "N/A", "404"])
            else:
                landing = random.choice(
                    [
                        "/home",
                        "/products",
                        "/sale",
                        "/category/electronics",
                        "/product/item-123",
                        "/search?q=laptop",
                        "/cart",
                        "/checkout",
                    ]
                )
            record["entry_page"] = landing

        # Exit page
        if random.random() > 0.5:  # 50% have exit page
            if i % 75 == 0:
                exit_page = None
            elif i % 85 == 0:
                exit_page = random.choice(["", "N/A", "timeout"])
            else:
                if conversion in [True, "Y", "Yes", "1", "true"]:
                    exit_page = "/order-confirmation"
                elif abandonment in [True, "Y", "Yes", "1"]:
                    exit_page = random.choice(["/cart", "/checkout", "/shipping"])
                else:
                    exit_page = random.choice(
                        [
                            "/home",
                            "/products",
                            "/product/item-456",
                            "/about",
                            "/contact",
                        ]
                    )
            record["exit_page"] = exit_page

        # Items added to cart
        if random.random() > 0.6:  # 40% have cart items
            if abandonment in [True, "Y", "Yes", "1"] or conversion in [
                True,
                "Y",
                "Yes",
                "1",
                "true",
            ]:
                if i % 80 == 0:
                    cart_items = None
                elif i % 90 == 0:  # String value
                    cart_items = random.choice(["Multiple", "Few", "Many"])
                else:
                    cart_items = random.randint(1, 10)
            else:
                cart_items = 0
            record["cart_items_count"] = cart_items

        # Revenue (if converted)
        if random.random() > 0.7:  # 30% have revenue
            if conversion in [True, "Y", "Yes", "1", "true"]:
                if i % 85 == 0:
                    revenue = random.choice(["Pending", "Calculate", "N/A"])
                elif i % 95 == 0:  # Negative revenue
                    revenue = round(random.uniform(-500, -10), 2)
                else:
                    revenue = round(random.uniform(20, 1000), 2)
            else:
                revenue = 0
            record["session_revenue"] = revenue

        # New vs Returning
        if random.random() > 0.5:  # 50% have this flag
            if cust_id and cust_id in customer_behavior:
                if customer_behavior[cust_id]["sessions"] > 1:
                    visitor_type = "Returning"
                else:
                    visitor_type = "New"
            else:
                visitor_type = random.choice(["New", "Returning"])

            if i % 90 == 0:
                visitor_type = random.choice(["Unknown", "N/A", ""])

            record["visitor_type"] = visitor_type

        data.append(record)

    # Create DataFrame
    df = pd.DataFrame(data)

    # Add EXACT duplicate rows (every attribute is same)
    num_exact_duplicates = int(num_rows * 0.02)  # 2% exact duplicates
    for _ in range(num_exact_duplicates):
        if len(df) > 0:
            # Pick a random row to duplicate
            row_to_duplicate = df.sample(1)
            df = pd.concat([df, row_to_duplicate], ignore_index=True)

    # Add some completely empty rows
    for _ in range(int(num_rows * 0.005)):  # 0.5% empty rows
        empty_row = pd.Series([None] * len(df.columns), index=df.columns)
        df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)

    # Add some rows with all string 'NULL' or 'N/A' values
    for _ in range(int(num_rows * 0.01)):  # 1% NULL string rows
        null_values = ["NULL", "N/A", "null", "NA", "", " "]
        null_row = pd.Series(
            [random.choice(null_values) for _ in range(len(df.columns))],
            index=df.columns,
        )
        df = pd.concat([df, pd.DataFrame([null_row])], ignore_index=True)

    # Shuffle the dataframe to mix duplicates throughout
    df = df.sample(frac=1).reset_index(drop=True)

    return df


def add_more_messiness(df):
    """
    Add additional data quality issues to make the dataset more challenging
    """
    # Add trailing/leading spaces to some string columns
    string_cols = df.select_dtypes(include=["object"]).columns
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05  # 5% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    # Add case inconsistencies
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03  # 3% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    # Add special characters to some values
    for col in string_cols[:3]:  # Only first 3 string columns
        mask = np.random.random(len(df)) < 0.02  # 2% of values
        special_chars = ["@", "#", "!", "*", "&", "%"]
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: str(x) + random.choice(special_chars) if pd.notna(x) else x
        )

    return df


# Generate the dataset
if __name__ == "__main__":
    # Set the number of rows you want
    NUM_ROWS = 5000  # Change this to your desired number

    # ID format should match your customers dataset
    CUSTOMER_ID_FORMAT = "CUST"  # Options: 'CUST', 'CUSTOMER', or 'NUMBER'

    print(f"Generating {NUM_ROWS} rows of messy customer sessions data...")
    df = generate_messy_customer_sessions_data(NUM_ROWS, CUSTOMER_ID_FORMAT)

    # Add more messiness
    df = add_more_messiness(df)

    # Display basic info
    print(f"\nDataset shape: {df.shape}")
    print(f"\nColumn names (realistic but challenging for mapping):")
    for i, col in enumerate(df.columns, 1):
        print(f"  {i}. {col}")

    print(f"\nFirst 10 rows:")
    print(df.head(10))

    # Show data quality issues summary
    print("\n" + "=" * 50)
    print("DATA QUALITY ISSUES SUMMARY:")
    print("=" * 50)
    print(f"Total null values: {df.isnull().sum().sum()}")
    print(f"Exact duplicate rows: {df.duplicated().sum()}")
    print(f"Total rows: {len(df)}")

    # Show business logic violations
    print("\n" + "=" * 50)
    print("BUSINESS LOGIC VIOLATIONS EXAMPLES:")
    print("=" * 50)

    # Check for session end before start
    if "start_timestamp" in df.columns and "end_timestamp" in df.columns:
        start_dt = pd.to_datetime(df["start_timestamp"], errors="coerce")
        end_dt = pd.to_datetime(df["end_timestamp"], errors="coerce")
        invalid_sessions = df[end_dt < start_dt]
        if len(invalid_sessions) > 0:
            print(f"✗ Session end before start: {len(invalid_sessions)} sessions")

    # Check for products > pages
    if "page_views" in df.columns and "products_browsed" in df.columns:
        pages_numeric = pd.to_numeric(df["page_views"], errors="coerce")
        products_numeric = pd.to_numeric(df["products_browsed"], errors="coerce")
        invalid_browsing = df[products_numeric > pages_numeric]
        if len(invalid_browsing) > 0:
            print(f"✗ Products viewed > pages viewed: {len(invalid_browsing)} sessions")

    # Check for both conversion and abandonment
    if "purchase_made" in df.columns and "cart_abandoned" in df.columns:
        both_flags = df[
            (df["purchase_made"].isin([True, "Y", "Yes", "1", "true"]))
            & (df["cart_abandoned"].isin([True, "Y", "Yes", "1"]))
        ]
        if len(both_flags) > 0:
            print(
                f"✗ Both converted AND abandoned (impossible): {len(both_flags)} sessions"
            )

    # Check for zero pages but products viewed
    if "page_views" in df.columns and "products_browsed" in df.columns:
        pages_numeric = pd.to_numeric(df["page_views"], errors="coerce")
        products_numeric = pd.to_numeric(df["products_browsed"], errors="coerce")
        zero_pages_with_products = df[(pages_numeric == 0) & (products_numeric > 0)]
        if len(zero_pages_with_products) > 0:
            print(
                f"✗ Zero pages but products viewed: {len(zero_pages_with_products)} sessions"
            )

    # Check for negative values
    for col in ["page_views", "products_browsed"]:
        if col in df.columns:
            col_numeric = pd.to_numeric(df[col], errors="coerce")
            negative_values = df[col_numeric < 0]
            if len(negative_values) > 0:
                print(f"✗ Negative {col}: {len(negative_values)} sessions")

    # Check for extremely long sessions
    if "session_duration_sec" in df.columns:
        duration_numeric = pd.to_numeric(df["session_duration_sec"], errors="coerce")
        long_sessions = df[duration_numeric > 86400]  # > 24 hours
        if len(long_sessions) > 0:
            print(f"✗ Sessions > 24 hours: {len(long_sessions)} sessions")

    # Check for invalid foreign keys
    if "user_id" in df.columns:
        invalid_customers = df[
            df["user_id"].isin(["INVALID", "GUEST", "ANONYMOUS", "N/A"])
        ]
        if len(invalid_customers) > 0:
            print(f"✗ Invalid customer references: {len(invalid_customers)} rows")

    # Show session statistics
    print("\n" + "=" * 50)
    print("SESSION STATISTICS:")
    print("=" * 50)

    # Guest vs Registered
    if "user_id" in df.columns:
        guest_sessions = df[df["user_id"].isna()]
        registered_sessions = df[df["user_id"].notna()]
        print(f"Session types:")
        print(
            f"  Guest sessions: {len(guest_sessions)} ({len(guest_sessions)/len(df)*100:.1f}%)"
        )
        print(
            f"  Registered sessions: {len(registered_sessions)} ({len(registered_sessions)/len(df)*100:.1f}%)"
        )

    # Session duration analysis
    if "session_duration_sec" in df.columns:
        duration_numeric = pd.to_numeric(df["session_duration_sec"], errors="coerce")
        valid_durations = duration_numeric[
            (duration_numeric > 0) & (duration_numeric < 86400)
        ]

        if len(valid_durations) > 0:
            print(f"\nSession duration:")
            print(f"  Average: {valid_durations.mean()/60:.1f} minutes")
            print(f"  Median: {valid_durations.median()/60:.1f} minutes")

            # Bounce rate (< 30 seconds)
            bounce_sessions = (valid_durations < 30).sum()
            print(
                f"  Bounce rate (<30 sec): {bounce_sessions/len(valid_durations)*100:.1f}%"
            )

    # Conversion analysis
    if "purchase_made" in df.columns:
        converted = df["purchase_made"].isin([True, "Y", "Yes", "1", "true"]).sum()
        conversion_rate = converted / len(df) * 100
        print(f"\nConversion metrics:")
        print(f"  Sessions with purchase: {converted}")
        print(f"  Conversion rate: {conversion_rate:.2f}%")

    # Cart abandonment analysis
    if "cart_abandoned" in df.columns:
        abandoned = df["cart_abandoned"].isin([True, "Y", "Yes", "1"]).sum()
        abandonment_rate = abandoned / len(df) * 100
        print(f"  Cart abandonment rate: {abandonment_rate:.2f}%")

    # Device distribution
    if "device_category" in df.columns:
        device_counts = df["device_category"].value_counts()
        print(f"\nDevice distribution:")
        for device, count in device_counts.head(5).items():
            if device and device not in ["Unknown", "Bot", "Crawler", "N/A", ""]:
                print(f"  {device}: {count} ({count/len(df)*100:.1f}%)")

    # Traffic source analysis
    if "traffic_source" in df.columns:
        source_counts = df["traffic_source"].value_counts()
        print(f"\nTop traffic sources:")
        for source, count in source_counts.head(5).items():
            if source and source not in ["Unknown", "N/A", "(not set)", "(none)", ""]:
                print(f"  {source}: {count} ({count/len(df)*100:.1f}%)")

    # Page engagement metrics
    if "page_views" in df.columns and "products_browsed" in df.columns:
        pages_numeric = pd.to_numeric(df["page_views"], errors="coerce")
        products_numeric = pd.to_numeric(df["products_browsed"], errors="coerce")

        valid_pages = pages_numeric[pages_numeric > 0]
        valid_products = products_numeric[products_numeric >= 0]

        if len(valid_pages) > 0:
            print(f"\nEngagement metrics:")
            print(f"  Average pages per session: {valid_pages.mean():.1f}")
            print(f"  Average products viewed: {valid_products.mean():.1f}")

    # Conversion by device
    if "device_category" in df.columns and "purchase_made" in df.columns:
        print(f"\nConversion by device:")
        for device in ["Mobile", "Desktop", "Tablet"]:
            device_sessions = df[df["device_category"] == device]
            if len(device_sessions) > 0:
                device_conversions = (
                    device_sessions["purchase_made"]
                    .isin([True, "Y", "Yes", "1", "true"])
                    .sum()
                )
                device_rate = device_conversions / len(device_sessions) * 100
                print(f"  {device}: {device_rate:.2f}%")

    # Show main columns analysis
    print("\n" + "=" * 50)
    print("MAIN COLUMNS ANALYSIS:")
    print("=" * 50)

    main_columns = [
        "session_ref",
        "user_id",
        "start_timestamp",
        "end_timestamp",
        "device_category",
        "traffic_source",
        "page_views",
        "products_browsed",
        "purchase_made",
        "cart_abandoned",
    ]

    for col in main_columns:
        if col in df.columns:
            null_count = df[col].isnull().sum()
            unique_count = df[col].nunique()
            print(f"\n{col}:")
            print(f"  - Null values: {null_count} ({null_count/len(df)*100:.1f}%)")
            print(f"  - Unique values: {unique_count}")

    # Save to CSV
    output_file = "messy_customer_sessions_data.xlsx"
    df.to_excel(output_file, index=False)
    print(f"\n✅ Dataset saved to '{output_file}'")

    # Save session summary by traffic source
    session_summary = []

    if "traffic_source" in df.columns and "purchase_made" in df.columns:
        for source in df["traffic_source"].dropna().unique()[:15]:
            source_sessions = df[df["traffic_source"] == source]

            conversions = (
                source_sessions["purchase_made"]
                .isin([True, "Y", "Yes", "1", "true"])
                .sum()
            )
            abandonments = (
                source_sessions["cart_abandoned"].isin([True, "Y", "Yes", "1"]).sum()
                if "cart_abandoned" in df.columns
                else 0
            )
            avg_pages = (
                pd.to_numeric(source_sessions["page_views"], errors="coerce").mean()
                if "page_views" in df.columns
                else 0
            )

            session_summary.append(
                {
                    "traffic_source": source,
                    "session_count": len(source_sessions),
                    "conversions": conversions,
                    "conversion_rate": (
                        round(conversions / len(source_sessions) * 100, 2)
                        if len(source_sessions) > 0
                        else 0
                    ),
                    "abandonments": abandonments,
                    "avg_page_views": round(avg_pages, 1),
                }
            )

    if session_summary:
        summary_df = pd.DataFrame(session_summary)
        summary_df.to_csv("sessions_traffic_summary.csv", index=False)
        print(f"✅ Session traffic summary saved to 'sessions_traffic_summary.csv'")

Generating 5000 rows of messy customer sessions data...


/tmp/ipykernel_7262/2875802591.py:512: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)
/tmp/ipykernel_7262/2875802591.py:512: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)
/tmp/ipykernel_7262/2875802591.py:512: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or 


Dataset shape: (5175, 19)

Column names (realistic but challenging for mapping):
  1. session_ref
  2. user_id
  3. start_timestamp
  4. end_timestamp
  5. device_category
  6. traffic_source
  7. page_views
  8. products_browsed
  9. purchase_made
  10. cart_abandoned
  11. bounce_session
  12. entry_page
  13. exit_page
  14. visitor_type
  15. session_duration_sec
  16. cart_items_count
  17. browser_name
  18. geo_country
  19. session_revenue

First 10 rows:
    session_ref     user_id             start_timestamp  \
0      ec570e42  CUST_01003  2025-09-11 01:44:32.733694   
1      92564A19  CUST_01173  2025-09-08 20:59:32.693717   
2      c0dcdd86  CUST_01055  2025-09-05 02:58:32.691686   
3      F55AF0DF  CUST_01067  2025-09-04 13:29:32.790205   
4      DE6E7AA9  CUST_01267  2025-09-06 21:10:32.704861   
5      9F3211D4  cust_01494  2025-09-09 06:06:32.689138   
6      7B464609        None  2025-08-28 06:53:32.721625   
7      A967C412  cust_01247  2025-09-09 08:30:32.651176   


### Supplier Table Generator

In [1]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import string

# Initialize Faker with multiple locales for diversity
fake = Faker(["en_US", "en_GB", "de_DE", "fr_FR", "es_ES", "zh_CN", "ja_JP"])
Faker.seed(42)
np.random.seed(42)
random.seed(42)


def generate_messy_supplier_data(num_rows=1000):
    """
    Generate a messy, realistic supplier dataset with various data quality issues
    for testing data mapping, cleaning, and validation processes.
    """

    data = []

    # Track some data for creating duplicates and relationships
    used_ids = []
    duplicate_names = []
    common_domains = [
        "gmail.com",
        "yahoo.com",
        "hotmail.com",
        "outlook.com",
        "company.com",
        "supplier.com",
        "manufacturing.com",
    ]

    # Choose consistent ID format for the dataset (with some violations)
    id_format_choice = random.choice(["SUP", "SUPPLIER", "VENDOR", "NUMBER"])

    for i in range(num_rows):
        record = {}

        # Supplier ID with various formats and issues
        if i % 45 == 0 and used_ids:  # ~2% duplicates
            supplier_id = random.choice(used_ids)
        elif i % 100 == 0:  # 1% null IDs (violates PRIMARY KEY constraint)
            supplier_id = None
        elif i % 60 == 0:  # Format violations
            # Mix different formats
            formats = [
                f"S{str(i + 1000).zfill(6)}",
                f"VND-{str(i + 1000)}",
                f"PARTNER_{i + 1000}",
                str(i + 100000),
                f"{fake.company_suffix().upper()[:3]}-{i + 1000}",
            ]
            supplier_id = random.choice(formats)
        elif i % 75 == 0:  # Case inconsistencies
            if id_format_choice == "SUP":
                supplier_id = random.choice(
                    [
                        f"sup_{str(i + 1000).zfill(5)}",
                        f"Sup_{str(i + 1000).zfill(5)}",
                        f"SUP_{str(i + 1000).zfill(5)}",
                    ]
                )
            else:
                supplier_id = f"SUPPLIER-{str(i + 1000).zfill(5)}"
        else:
            # Normal ID generation based on chosen format
            if id_format_choice == "SUP":
                supplier_id = f"SUP_{str(i + 1000).zfill(5)}"
            elif id_format_choice == "SUPPLIER":
                supplier_id = f"SUPPLIER-{str(i + 1000).zfill(5)}"
            elif id_format_choice == "VENDOR":
                supplier_id = f"VND{str(i + 1000).zfill(6)}"
            else:  # NUMBER
                supplier_id = str(10000 + i)

        used_ids.append(supplier_id)
        record["supplier_id"] = supplier_id

        # Supplier Name with various issues
        if i % 200 == 0:  # 0.5% null names (violates NOT NULL constraint)
            name = None
        elif i % 30 == 0:  # Duplicates with slight variations
            if duplicate_names:
                base_name = random.choice(duplicate_names)
                variations = [
                    base_name,
                    base_name.upper(),
                    base_name.lower(),
                    base_name + " Inc",
                    base_name + " LLC",
                    base_name + " Co.",
                    base_name + " Corporation",
                    base_name.replace(" ", "-"),
                    base_name + " (USA)",
                ]
                name = random.choice(variations)
            else:
                name = fake.company()
                duplicate_names.append(name)
        elif i % 40 == 0:  # Names with special characters
            name = fake.company() + random.choice(
                [" & Co.", " @ Supply", " #1", " *Premium*"]
            )
        elif i % 50 == 0:  # Unicode/International characters
            intl_suffixes = [" 中国供应商", " ГмбХ", " société", " 株式会社", " شركة"]
            name = fake.company() + random.choice(intl_suffixes)
        elif i % 60 == 0:  # Abbreviations and acronyms
            name = (
                "".join([word[0].upper() for word in fake.company().split()[:3]])
                + " Corp"
            )
        elif i % 70 == 0:  # Very long names
            name = (
                fake.company()
                + " "
                + fake.catch_phrase()
                + " International Trading Company Limited"
            )
        elif i % 80 == 0:  # Names with typos
            company = fake.company()
            typos = [
                company.replace("e", "3"),
                company.replace("a", "@"),
                company.replace("o", "0"),
                company.replace("i", "1"),
                company.replace("s", "$"),
            ]
            name = random.choice(typos)
        elif i % 90 == 0:  # Placeholder/test data
            name = random.choice(
                [
                    "TEST",
                    "Test Supplier",
                    "DO NOT USE",
                    "DELETE ME",
                    "temp",
                    "PLACEHOLDER",
                    "Example Co",
                    "Sample Vendor",
                ]
            )
        else:
            name = fake.company()

        record["supplier_name"] = name

        # Contact Email with various formats and issues
        if i % 15 == 0:  # Nulls
            email = None
        elif i % 25 == 0:  # Invalid formats
            invalid_emails = [
                "not-an-email",
                "@" + random.choice(common_domains),
                fake.user_name(),
                fake.user_name() + "@",
                fake.user_name() + "@@" + random.choice(common_domains),
                "email@",
                "N/A",
                "NA",
                "TBD",
                "UNKNOWN",
                "",
                " ",
                "NULL",
            ]
            email = random.choice(invalid_emails)
        elif i % 35 == 0:  # Case issues
            email = fake.company_email().upper()
        elif i % 45 == 0:  # Personal emails for business (potential issue)
            personal_domains = ["gmail.com", "yahoo.com", "hotmail.com", "aol.com"]
            email = (
                fake.user_name()
                + str(random.randint(1, 999))
                + "@"
                + random.choice(personal_domains)
            )
        elif i % 55 == 0:  # Multiple emails concatenated
            email = fake.company_email() + ";" + fake.company_email()
        elif i % 65 == 0:  # Old/outdated email formats
            email = fake.user_name() + "@compuserve.com"
        elif i % 75 == 0:  # Email with spaces or special chars
            email = fake.user_name() + " @" + random.choice(common_domains)
        elif i % 85 == 0:  # Duplicate emails (business logic issue)
            if i > 100:
                # Use an email from a previous supplier
                email = data[random.randint(0, min(i - 1, len(data) - 1))].get(
                    "contact_email", fake.company_email()
                )
            else:
                email = fake.company_email()
        else:
            email = fake.company_email()

        record["contact_email"] = email

        # Phone Number with format variations
        if i % 20 == 0:  # Nulls
            phone = None
        elif i % 30 == 0:  # Invalid formats
            invalid_phones = [
                "0000000000",
                "9999999999",
                "1111111111",
                "123",
                "CALL US",
                "N/A",
                "NA",
                "TBD",
                "PHONE",
                "NULL",
                "",
                "123-456-789",  # Too short
                "12345678901234567890",  # Too long
                "555-555-5555",  # Classic fake number
            ]
            phone = random.choice(invalid_phones)
        elif i % 40 == 0:  # International formats
            intl_formats = [
                "+" + str(random.randint(1, 99)) + " " + fake.phone_number(),
                "00" + str(random.randint(1, 99)) + " " + fake.phone_number(),
                fake.phone_number().replace("-", "."),
                fake.phone_number().replace("-", " "),
                "("
                + str(random.randint(100, 999))
                + ") "
                + str(random.randint(100, 999))
                + "-"
                + str(random.randint(1000, 9999)),
            ]
            phone = random.choice(intl_formats)
        elif i % 50 == 0:  # Extensions
            phone = fake.phone_number() + " ext " + str(random.randint(100, 9999))
        elif i % 60 == 0:  # Multiple numbers
            phone = fake.phone_number() + " / " + fake.phone_number()
        elif i % 70 == 0:  # Letters in phone
            phone = "1-800-" + "".join(random.choices(string.ascii_uppercase, k=7))
        elif i % 80 == 0:  # Stripped format
            phone = "".join(filter(str.isdigit, fake.phone_number()))
        else:
            phone = fake.phone_number()

        record["phone_number"] = phone

        # Country with various formats and issues
        if i % 25 == 0:  # Nulls
            country = None
        elif i % 35 == 0:  # Country codes instead of names
            country = fake.country_code()
        elif i % 45 == 0:  # ISO3 codes
            iso3_codes = ["USA", "GBR", "DEU", "FRA", "CHN", "JPN", "IND", "BRA"]
            country = random.choice(iso3_codes)
        elif i % 55 == 0:  # Mixed formats
            country_variations = [
                "United States",
                "United States of America",
                "USA",
                "US",
                "U.S.",
                "U.S.A.",
                "America",
                "États-Unis",
                "Estados Unidos",
            ]
            country = random.choice(country_variations)
        elif i % 65 == 0:  # Invalid/Old country names
            old_countries = [
                "USSR",
                "Yugoslavia",
                "Czechoslovakia",
                "East Germany",
                "Burma",
                "Ceylon",
                "Zaire",
            ]
            country = random.choice(old_countries)
        elif i % 75 == 0:  # Typos
            country_name = fake.country()
            typos = [
                country_name.lower(),
                country_name.upper(),
                country_name.replace("a", "@"),
                country_name[:-1],  # Missing last char
                country_name + "a",  # Extra char
            ]
            country = random.choice(typos)
        elif i % 85 == 0:  # Cities instead of countries
            country = fake.city()
        elif i % 95 == 0:  # Placeholder values
            country = random.choice(
                ["N/A", "NA", "Unknown", "TBD", "NULL", "", "International"]
            )
        else:
            country = fake.country()

        record["country"] = country

        # Created At timestamp with various formats
        if i % 30 == 0:  # Nulls
            created = None
        elif i % 40 == 0:  # String timestamps
            created = fake.date_time_between(start_date="-5y", end_date="now").strftime(
                "%Y-%m-%d %H:%M:%S"
            )
        elif i % 50 == 0:  # Unix timestamp
            created = int(
                fake.date_time_between(start_date="-5y", end_date="now").timestamp()
            )
        elif i % 60 == 0:  # ISO format
            created = fake.date_time_between(
                start_date="-5y", end_date="now"
            ).isoformat()
        elif i % 70 == 0:  # Future dates (business logic violation)
            created = fake.date_time_between(start_date="+1d", end_date="+1y")
        elif i % 80 == 0:  # Very old dates (potential data issue)
            created = fake.date_time_between(start_date="-50y", end_date="-20y")
        elif i % 90 == 0:  # Invalid date strings
            invalid_dates = [
                "0000-00-00 00:00:00",
                "1900-01-01 00:00:00",
                "9999-99-99 99:99:99",
                "Not Available",
                "TBD",
                "ASAP",
            ]
            created = random.choice(invalid_dates)
        elif i % 100 == 0:  # Date only (no time)
            created = fake.date_between(start_date="-5y", end_date="today")
        else:
            created = fake.date_time_between(start_date="-5y", end_date="now")

        record["created_at"] = created

        # Add some additional realistic columns that might exist in real data

        # Supplier Status (not in original schema but commonly found)
        if random.random() > 0.3:  # 70% have this field
            if i % 25 == 0:
                status = None
            elif i % 35 == 0:
                # Inconsistent values
                status = random.choice(
                    [
                        "active",
                        "ACTIVE",
                        "1",
                        "A",
                        "inactive",
                        "INACTIVE",
                        "0",
                        "I",
                        "approved",
                        "APPROVED",
                        "pending",
                        "PENDING",
                        "blocked",
                        "BLOCKED",
                    ]
                )
            elif i % 45 == 0:
                # Typos
                status = random.choice(["Actve", "Inactiv", "Pendng", "Aporved"])
            else:
                status = random.choice(
                    [
                        "Active",
                        "Inactive",
                        "Pending",
                        "Approved",
                        "Blocked",
                        "Under Review",
                    ]
                )
            record["status"] = status

        # Supplier Category (commonly found additional field)
        if random.random() > 0.4:  # 60% have this field
            if i % 30 == 0:
                category = None
            elif i % 40 == 0:
                # Coded values
                category = random.choice(
                    ["CAT1", "CAT2", "CAT3", "A", "B", "C", "1", "2", "3"]
                )
            elif i % 50 == 0:
                # Mixed terminology
                category = random.choice(
                    [
                        "Manufacturer",
                        "manufacturer",
                        "MANUFACTURER",
                        "Mfg",
                        "MFG",
                        "Distributor",
                        "distributor",
                        "Dist",
                        "DIST",
                        "Wholesaler",
                        "wholesale",
                        "Retailer",
                        "retail",
                    ]
                )
            else:
                category = random.choice(
                    [
                        "Manufacturer",
                        "Distributor",
                        "Wholesaler",
                        "Retailer",
                        "Service Provider",
                        "Logistics",
                        "Raw Materials",
                    ]
                )
            record["supplier_category"] = category

        # Tax ID / Business Number (common additional field)
        if random.random() > 0.5:  # 50% have this field
            if i % 25 == 0:
                tax_id = None
            elif i % 35 == 0:
                tax_id = random.choice(
                    ["N/A", "NA", "PENDING", "NOT PROVIDED", "000000000"]
                )
            elif i % 45 == 0:
                # Invalid format
                tax_id = "".join(
                    random.choices(
                        string.ascii_uppercase + string.digits, k=random.randint(3, 20)
                    )
                )
            else:
                # Realistic tax ID formats
                tax_formats = [
                    f"{random.randint(10, 99)}-{random.randint(1000000, 9999999)}",  # US EIN
                    f"GB{random.randint(100000000, 999999999)}",  # UK VAT
                    f"DE{random.randint(100000000, 999999999)}",  # German VAT
                    f"FR{random.randint(10, 99)}{random.randint(100000000, 999999999)}",  # French VAT
                ]
                tax_id = random.choice(tax_formats)
            record["tax_id"] = tax_id

        data.append(record)

    # Create DataFrame
    df = pd.DataFrame(data)

    # Add EXACT duplicate rows
    num_exact_duplicates = int(num_rows * 0.02)  # 2% exact duplicates
    for _ in range(num_exact_duplicates):
        if len(df) > 0:
            row_to_duplicate = df.sample(1)
            df = pd.concat([df, row_to_duplicate], ignore_index=True)

    # Add some completely empty rows
    for _ in range(int(num_rows * 0.005)):  # 0.5% empty rows
        empty_row = pd.Series([None] * len(df.columns), index=df.columns)
        df = pd.concat([df, pd.DataFrame([empty_row])], ignore_index=True)

    # Add rows with all NULL string values
    for _ in range(int(num_rows * 0.01)):  # 1% NULL string rows
        null_values = ["NULL", "N/A", "null", "NA", "", " ", "None", "nil"]
        null_row = pd.Series(
            [random.choice(null_values) for _ in range(len(df.columns))],
            index=df.columns,
        )
        df = pd.concat([df, pd.DataFrame([null_row])], ignore_index=True)

    # Shuffle the dataframe
    df = df.sample(frac=1).reset_index(drop=True)

    return df


def add_supplier_messiness(df):
    """
    Add additional data quality issues specific to supplier data
    """
    # Add trailing/leading spaces
    string_cols = df.select_dtypes(include=["object"]).columns
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.05  # 5% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: "  " + str(x) + "  " if pd.notna(x) else x
        )

    # Add case inconsistencies
    for col in string_cols:
        mask = np.random.random(len(df)) < 0.03  # 3% of values
        df.loc[mask, col] = df.loc[mask, col].apply(
            lambda x: (
                str(x).upper()
                if pd.notna(x) and random.random() > 0.5
                else str(x).lower() if pd.notna(x) else x
            )
        )

    # Add encoding issues
    for col in ["supplier_name", "contact_email"]:
        if col in df.columns:
            mask = np.random.random(len(df)) < 0.01  # 1% of values
            df.loc[mask, col] = df.loc[mask, col].apply(
                lambda x: (
                    str(x)
                    .encode("utf-8", errors="ignore")
                    .decode("latin-1", errors="ignore")
                    if pd.notna(x) and random.random() > 0.5
                    else x
                )
            )

    # Add HTML entities (common in web-scraped data)
    for col in ["supplier_name"]:
        if col in df.columns:
            mask = np.random.random(len(df)) < 0.02  # 2% of values
            df.loc[mask, col] = df.loc[mask, col].apply(
                lambda x: (
                    str(x)
                    .replace("&", "&amp;")
                    .replace("<", "&lt;")
                    .replace(">", "&gt;")
                    if pd.notna(x)
                    else x
                )
            )

    return df


def analyze_supplier_data_quality(df):
    """
    Comprehensive analysis of data quality issues in supplier dataset
    """
    print("\n" + "=" * 60)
    print("SUPPLIER DATA QUALITY ANALYSIS REPORT")
    print("=" * 60)

    # Basic statistics
    print(f"\nDataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

    # Schema columns
    schema_columns = [
        "supplier_id",
        "supplier_name",
        "contact_email",
        "phone_number",
        "country",
        "created_at",
    ]

    print("\n" + "=" * 60)
    print("COLUMN ANALYSIS (SCHEMA COLUMNS):")
    print("=" * 60)

    for col in schema_columns:
        if col in df.columns:
            print(f"\n📊 {col}:")
            print(f"   - Data Type: {df[col].dtype}")
            print(
                f"   - Null Count: {df[col].isnull().sum()} ({df[col].isnull().sum()/len(df)*100:.1f}%)"
            )
            print(f"   - Empty Strings: {(df[col] == '').sum()}")
            print(f"   - Unique Values: {df[col].nunique()}")
            print(f"   - Duplicates: {len(df[col]) - df[col].nunique()}")

            if df[col].dtype == "object":
                # Check for common placeholder values
                placeholders = [
                    "NULL",
                    "N/A",
                    "NA",
                    "None",
                    "TBD",
                    "UNKNOWN",
                    "null",
                    "n/a",
                ]
                placeholder_count = df[col].isin(placeholders).sum()
                if placeholder_count > 0:
                    print(f"   - Placeholder Values: {placeholder_count}")

                # Sample values
                non_null_values = df[col].dropna()
                if len(non_null_values) > 0:
                    print(f"   - Sample Values: {non_null_values.head(3).tolist()}")

    # Constraint violations
    print("\n" + "=" * 60)
    print("DATABASE CONSTRAINT VIOLATIONS:")
    print("=" * 60)

    # Primary Key violations
    if "supplier_id" in df.columns:
        null_ids = df["supplier_id"].isnull().sum()
        duplicate_ids = df["supplier_id"].duplicated().sum()
        if null_ids > 0:
            print(f"❌ PRIMARY KEY Violation: {null_ids} NULL supplier_ids")
        if duplicate_ids > 0:
            print(f"❌ PRIMARY KEY Violation: {duplicate_ids} duplicate supplier_ids")
        if null_ids == 0 and duplicate_ids == 0:
            print(f"✅ PRIMARY KEY Constraint: No violations found")

    # NOT NULL violations
    if "supplier_name" in df.columns:
        null_names = df["supplier_name"].isnull().sum()
        empty_names = (df["supplier_name"] == "").sum()
        if null_names > 0:
            print(f"❌ NOT NULL Violation: {null_names} NULL supplier_names")
        if empty_names > 0:
            print(f"⚠️  Warning: {empty_names} empty supplier_names")
        if null_names == 0:
            print(f"✅ NOT NULL Constraint: No violations for supplier_name")

    # Business Logic Issues
    print("\n" + "=" * 60)
    print("BUSINESS LOGIC ISSUES:")
    print("=" * 60)

    # Email validation
    if "contact_email" in df.columns:
        email_series = df["contact_email"].dropna()
        invalid_emails = []
        for email in email_series:
            if isinstance(email, str) and "@" not in email:
                invalid_emails.append(email)
        if len(invalid_emails) > 0:
            print(f"❌ Invalid Email Formats: {len(invalid_emails)} emails without '@'")
            print(f"   Examples: {invalid_emails[:3]}")

    # Phone number validation
    if "phone_number" in df.columns:
        phone_series = df["phone_number"].dropna()
        suspicious_phones = phone_series[
            phone_series.isin(["0000000000", "9999999999", "1111111111", "123"])
        ]
        if len(suspicious_phones) > 0:
            print(
                f"⚠️  Suspicious Phone Numbers: {len(suspicious_phones)} placeholder/test numbers"
            )

    # Future created_at dates
    if "created_at" in df.columns:
        current_time = datetime.now()
        future_dates = 0
        for val in df["created_at"]:
            if pd.notna(val):
                try:
                    if isinstance(val, (datetime, pd.Timestamp)):
                        if val > current_time:
                            future_dates += 1
                    elif isinstance(val, str):
                        parsed_date = pd.to_datetime(val, errors="coerce")
                        if pd.notna(parsed_date) and parsed_date > current_time:
                            future_dates += 1
                except:
                    pass
        if future_dates > 0:
            print(
                f"❌ Future Created Dates: {future_dates} records created in the future"
            )

    # Duplicate Analysis
    print("\n" + "=" * 60)
    print("DUPLICATE ANALYSIS:")
    print("=" * 60)

    exact_duplicates = df.duplicated().sum()
    print(f"Exact Duplicate Rows: {exact_duplicates}")

    if "supplier_name" in df.columns:
        name_duplicates = df["supplier_name"].duplicated().sum()
        print(f"Duplicate Supplier Names: {name_duplicates}")

    if "contact_email" in df.columns:
        email_duplicates = df["contact_email"].dropna().duplicated().sum()
        print(f"Duplicate Emails: {email_duplicates}")

    # Format Consistency Issues
    print("\n" + "=" * 60)
    print("FORMAT CONSISTENCY ISSUES:")
    print("=" * 60)

    if "supplier_id" in df.columns:
        id_formats = (
            df["supplier_id"]
            .dropna()
            .apply(
                lambda x: (
                    "ALPHA"
                    if str(x).replace("-", "").replace("_", "").isalpha()
                    else "NUMERIC" if str(x).isdigit() else "MIXED"
                )
            )
            .value_counts()
        )
        if len(id_formats) > 1:
            print(f"⚠️  Mixed ID Formats Detected:")
            for format_type, count in id_formats.items():
                print(f"   - {format_type}: {count} ({count/len(df)*100:.1f}%)")

    if "country" in df.columns:
        country_formats = []
        for country in df["country"].dropna().unique()[:20]:  # Check first 20 unique
            if len(str(country)) == 2:
                country_formats.append("ISO2")
            elif len(str(country)) == 3 and str(country).isupper():
                country_formats.append("ISO3")
            else:
                country_formats.append("NAME")
        format_distribution = pd.Series(country_formats).value_counts()
        if len(format_distribution) > 1:
            print(f"⚠️  Mixed Country Formats: {format_distribution.to_dict()}")

    # Data Entry Issues
    print("\n" + "=" * 60)
    print("DATA ENTRY QUALITY INDICATORS:")
    print("=" * 60)

    # Check for test data
    test_indicators = ["TEST", "TEMP", "DELETE", "PLACEHOLDER", "SAMPLE", "DO NOT USE"]
    test_data_count = 0
    for col in df.select_dtypes(include=["object"]).columns:
        for indicator in test_indicators:
            test_data_count += (
                df[col].astype(str).str.upper().str.contains(indicator, na=False).sum()
            )
    if test_data_count > 0:
        print(
            f"⚠️  Potential Test Data: {test_data_count} occurrences of test indicators"
        )

    # Check for whitespace issues
    whitespace_issues = 0
    for col in df.select_dtypes(include=["object"]).columns:
        whitespace_issues += df[col].astype(str).str.startswith(" ", na=False).sum()
        whitespace_issues += df[col].astype(str).str.endswith(" ", na=False).sum()
    if whitespace_issues > 0:
        print(f"⚠️  Leading/Trailing Whitespace: {whitespace_issues} instances")

    return df


# Main execution
if __name__ == "__main__":
    # Configuration
    NUM_ROWS = 1000  # Change this to your desired number of rows

    print(f"🔧 Generating {NUM_ROWS} rows of messy supplier data...")
    print("=" * 60)

    # Generate the dataset
    df = generate_messy_supplier_data(NUM_ROWS)

    # Add additional messiness
    df = add_supplier_messiness(df)

    # Perform comprehensive analysis
    df = analyze_supplier_data_quality(df)

    # Show sample of the data
    print("\n" + "=" * 60)
    print("SAMPLE DATA (First 10 Rows):")
    print("=" * 60)
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", None)
    print(df.head(10))

    # Save to multiple formats
    print("\n" + "=" * 60)
    print("SAVING DATA:")
    print("=" * 60)

    # Save to Excel
    excel_file = "messy_supplier_data.xlsx"
    df.to_excel(excel_file, index=False)
    print(f"✅ Saved to Excel: {excel_file}")

    # Save to CSV
    # csv_file = "messy_supplier_data.csv"
    # df.to_csv(csv_file, index=False)
    # print(f"✅ Saved to CSV: {csv_file}")

    # Save to JSON (for testing different parsing scenarios)
    # json_file = "messy_supplier_data.json"
    # df.to_json(json_file, orient="records", date_format="iso")
    # print(f"✅ Saved to JSON: {json_file}")

    print("\n" + "=" * 60)
    print("✨ Data generation complete!")
    print("=" * 60)

🔧 Generating 1000 rows of messy supplier data...

SUPPLIER DATA QUALITY ANALYSIS REPORT

Dataset Shape: 1035 rows × 9 columns
Memory Usage: 0.51 MB

COLUMN ANALYSIS (SCHEMA COLUMNS):

📊 supplier_id:
   - Data Type: object
   - Null Count: 14 (1.4%)
   - Empty Strings: 3
   - Unique Values: 979
   - Duplicates: 56
   - Placeholder Values: 7
   - Sample Values: ['SUP_01369', 'SUP_01078', 'SUP_01299']

📊 supplier_name:
   - Data Type: object
   - Null Count: 10 (1.0%)
   - Empty Strings: 1
   - Unique Values: 953
   - Duplicates: 82
   - Placeholder Values: 5
   - Sample Values: ['Pearce-Williams', 'Allison Ltd', '黄石金承网络有限公司']

📊 contact_email:
   - Data Type: object
   - Null Count: 73 (7.1%)
   - Empty Strings: 1
   - Unique Values: 927
   - Duplicates: 108
   - Placeholder Values: 21
   - Sample Values: ['rikafujiwara@kobayashi.com', 'esato@ito.jp', 'wigginsjesse@newman.com']

📊 phone_number:
   - Data Type: object
   - Null Count: 56 (5.4%)
   - Empty Strings: 2
   - Unique Values: 95